# Biohub E31 Primary Consensus Exploratory

Acknowledges pilkwang CC0 model/support packs. E31 primary-only consensus exploratory; not Private Score improvement proof.


In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree

BIOHUB_PRESET = 'harmonic_mutual_support_association_fusion_v1'
BIOHUB_SCORE_AXIS = 'weighted harmonic forward/reverse association consensus on the fixed-90 dual-seed baseline'

# Parameters for constrained lineage reconstruction.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.96875"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "1.5"
os.environ["BIOHUB_ILP_DIVISION_WEIGHT"] = "1.0"  # REVIEW: reverted after testing -- 0.3/1.0/2.0/3.0 all scored 0.915 on the real leaderboard; local proxy dropped at 0.3 (messier graph, not more true divisions), so staying at the neutral, proven value
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.8"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
# REVIEW: reverted, was "4.66"  -- tested at 7.0, local proxy improved (first non-zero division_jaccard) but real score dropped to 0.914. Likely explained by validator/test-set division-content mismatch, not a validator bug -- back to the proven value.
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "8.0"  # REVIEW: widened to match the 0.917 notebook directly. The earlier 7.0 test (pre-divergence-check) added noise that pure geometry couldn't filter out; divergence + mutual-NN are active now and should do that filtering instead.
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "11.0"  # REVIEW: matches 0.917, was "8.5"
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "10.0"  # REVIEW: matches 0.917, was "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"  # unchanged -- never bound at 0.920 (cap_skipped=0), leaving it as the safety net for this test
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"  # unchanged, same reasoning
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "0"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "2"  # REVIEW: best.pt is epoch 2, not checkpoint_last.pt's epoch 500
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = '1'  # REVIEW: was 0; funnel telemetry showed safe-div accepting 81-100% of its own candidates -- turning on the one gate built specifically to catch that
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"

# Exact candidate-arm settings formerly supplied by the diagnostic harness.
os.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"] = "0.30"  # REVIEW: was 0.20 (0.915 reference value, never tuned). Harmonic fusion is the one mechanism with proven benefit (+0.002 in the reference notebook) -- testing whether more weight on the reverse-time evidence extracts more of it, rather than exploring another unproven lever.
os.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"] = "harmonic_probability"
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
os.environ["BIOHUB_DIAGNOSTIC_ARM"] = "harmonic_association_production"

print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

In [ ]:
# Configuration guard: assert the single intended model-level change.
import json as _guard_json
import math as _guard_math
import os as _guard_os

_EXPECTED_NUMERIC = {
    "BIOHUB_DET_THRESHOLD": 0.96875,
    "BIOHUB_ILP_APPEARANCE_WEIGHT": 0.0,
    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": 1.5,
    "BIOHUB_GAP_CLOSE_UM": 5.8,
    "BIOHUB_OUTPUT_MIN_TRACK_LEN": 6.0,
    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": 0.30,
}

_EXPECTED_TEXT = {
    "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",
    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",
}

_drift = {}
for _key, _want in _EXPECTED_NUMERIC.items():
    _raw = _guard_os.environ.get(_key)
    if _raw is None:
        _drift[_key] = "missing"
        continue
    _got = float(_raw)
    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):
        _drift[_key] = {"expected": _want, "actual": _got}

for _key, _want in _EXPECTED_TEXT.items():
    _got = _guard_os.environ.get(_key)
    if _got != _want:
        _drift[_key] = {"expected": _want, "actual": _got}

if _drift:
    raise RuntimeError(
        "Configuration drift detected: " + _guard_json.dumps(_drift, sort_keys=True)
    )

print("Configuration guard: PASS")
print("Baseline: fixed-90 dual-seed clean pipeline (public LB 0.913)")
print("Single model-level change: harmonic mutual-support association fusion")
print("Reverse-time association weight: 0.200")

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
# Hard-bound production input: competition test images only.
TEST_DIR = COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Hard-bound full production coverage. Internal GPU sharding remains exhaustive.
SLICE = ""

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "8.0"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "11.0"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "10.0"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# --- new config: add next to the existing SAFE_DIV_* block ---
SAFE_DIV_DIVERGE_UM = float(os.environ.get("BIOHUB_SAFE_DIV_DIVERGE_UM", "2.25"))
SAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE", "1") != "0"
SAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN", "1") != "0"

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/best.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)

# Verify every materialized support-repo Python source before the next cell
# performs the sealed dynamic inference patch. Also hash the actual primary and
# DeepCenter checkpoint bytes that the production path will use.
import hashlib as _integrity_hashlib

_support_expected_sha256 = {
    "scripts/augmentations.py": "13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e",
    "scripts/dataspec.py": "e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f",
    "scripts/evaluate.py": "614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c",
    "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",
    "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",
    "src/biohub_tracking/__init__.py": "26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571",
    "src/biohub_tracking/division_metrics.py": "d1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be",
    "src/biohub_tracking/img_proc.py": "00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f",
    "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd",
    "src/biohub_tracking/metrics.py": "31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83",
    "src/biohub_tracking/models/__init__.py": "ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d",
    "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",
    "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac"
}
_support_expected_manifest_sha256 = "978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029"
_primary_expected_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
_deepcenter_expected_sha256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"  # best.pt (epoch 2), not checkpoint_last.pt (epoch 500)


def _integrity_sha256_file(path: Path) -> str:
    digest = _integrity_hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


_support_materialized_paths = {
    path.relative_to(REPO_DIR).as_posix(): path
    for path in REPO_DIR.rglob("*.py")
}
_support_actual_names = set(_support_materialized_paths)
_support_expected_names = set(_support_expected_sha256)
if _support_actual_names != _support_expected_names:
    raise RuntimeError({
        "support_repo_python_files_missing": sorted(
            _support_expected_names - _support_actual_names
        ),
        "support_repo_python_files_extra": sorted(
            _support_actual_names - _support_expected_names
        ),
    })
_support_actual_sha256 = {
    relative: _integrity_sha256_file(_support_materialized_paths[relative])
    for relative in sorted(_support_materialized_paths)
}
if _support_actual_sha256 != _support_expected_sha256:
    raise RuntimeError({
        "support_repo_python_checksum_mismatch": {
            relative: {
                "expected": _support_expected_sha256[relative],
                "actual": _support_actual_sha256[relative],
            }
            for relative in sorted(_support_expected_sha256)
            if _support_actual_sha256[relative]
            != _support_expected_sha256[relative]
        }
    })
_support_manifest_bytes = "".join(
    f"{_support_actual_sha256[relative]}  {relative}\n"
    for relative in sorted(_support_actual_sha256)
).encode("utf-8")
_support_actual_manifest_sha256 = _integrity_hashlib.sha256(
    _support_manifest_bytes
).hexdigest()
if _support_actual_manifest_sha256 != _support_expected_manifest_sha256:
    raise RuntimeError(
        "Support repo manifest checksum mismatch: "
        f"expected {_support_expected_manifest_sha256}, "
        f"got {_support_actual_manifest_sha256}"
    )

_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE
_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)
if _primary_actual_sha256 != _primary_expected_sha256:
    raise RuntimeError(
        "Materialized primary model checksum mismatch: "
        f"expected {_primary_expected_sha256}, got {_primary_actual_sha256}"
    )

_deepcenter_candidate_strings = [
    os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", "").strip(),
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/"
    "full_frame_center/best.pt",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/"
    "weights/full_frame_center/best.pt",
]
_deepcenter_candidates = []
for _candidate_string in _deepcenter_candidate_strings:
    if not _candidate_string:
        continue
    _candidate_path = Path(_candidate_string)
    if _candidate_path not in _deepcenter_candidates:
        _deepcenter_candidates.append(_candidate_path)
_deepcenter_materialized_path = next(
    (path for path in _deepcenter_candidates if path.is_file()),
    None,
)
if _deepcenter_materialized_path is None:
    raise FileNotFoundError({
        "missing_deepcenter_checkpoint": [str(path) for path in _deepcenter_candidates]
    })
_deepcenter_actual_sha256 = _integrity_sha256_file(
    _deepcenter_materialized_path
)
if _deepcenter_actual_sha256 != _deepcenter_expected_sha256:
    raise RuntimeError(
        "DeepCenter checkpoint checksum mismatch: "
        f"expected {_deepcenter_expected_sha256}, "
        f"got {_deepcenter_actual_sha256}"
    )
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = str(
    _deepcenter_materialized_path
)

print("Support repo Python manifest SHA256:", _support_actual_manifest_sha256)
print("Primary materialized SHA256:", _primary_actual_sha256)
print("DeepCenter materialized SHA256:", _deepcenter_actual_sha256)


# Resolve the independent-seed pack by checksum, then materialize only its weights.
import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.475"
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"

_runtime_integrity_receipt = {
    "status": "complete_label_free_runtime_integrity",
    "verified_before_dynamic_source_patch": True,
    "support_repo_python_file_count": len(_support_actual_sha256),
    "support_repo_python_sha256": _support_actual_sha256,
    "support_repo_python_manifest_sha256": _support_actual_manifest_sha256,
    "checkpoint_sha256": {
        "primary": _primary_actual_sha256,
        "secondary": _secondary_actual_sha256,
        "deepcenter": _deepcenter_actual_sha256,
    },
    "materialized_paths": {
        "primary": str(_primary_materialized_path),
        "secondary": str(SECONDARY_WEIGHTS_PATH),
        "deepcenter": str(_deepcenter_materialized_path),
    },
    "ground_truth_accessed": False,
}
_runtime_integrity_receipt_path = (
    WORKING_DIR / "bidirectional_production_runtime_integrity.json"
)
_runtime_integrity_receipt_path.write_text(
    json.dumps(_runtime_integrity_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Runtime integrity receipt:", _runtime_integrity_receipt_path)

In [ ]:
# Fail fast instead of silently running volumetric inference on CPU.
import torch as _torch

if not _torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this notebook. Enable a Kaggle GPU accelerator and commit again."
    )
print("CUDA device:", _torch.cuda.get_device_name(0))

# Apply eight-view planar detection TTA before graph prediction.
_ps = REPO_DIR / "scripts" / "predict_unet_transformer.py"
_s = _ps.read_text()
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
if _old in _s:
    _ps.write_text(_s.replace(_old, _new))
    print("TTA patch applied (400ep spatial D4-style)")
else:
    print("TTA WARNING: block not found - using default 4-way")

# Evaluate and calibrate two temporal models on one candidate graph.
_s = _ps.read_text()
_ensemble_replacements = [
    ('    downsample: tuple[int, ...] = (1, 4, 4),\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'),
    ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        del imgs', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    del secondary_imgs_t, secondary_det_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    del secondary_imgs_at, secondary_det_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    det_logits[f] = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n\n            del secondary_det_logits\n\n        del imgs'),
    ('            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            raw = edge_logits_pair[0]', '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]'),
    ('        del unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n'),
    ('    model, window_size, downsample = load_model(weights_path, device)\n    print(', '    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print('),
    ('                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n            )', '                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )'),
]
for _patch_index, (_ensemble_old, _ensemble_new) in enumerate(
    _ensemble_replacements, start=1
):
    _ensemble_count = _s.count(_ensemble_old)
    if _ensemble_count != 1:
        raise RuntimeError(
            f'Calibrated dual-seed patch {_patch_index} expected one match, '
            f'found {_ensemble_count}'
        )
    _s = _s.replace(_ensemble_old, _ensemble_new, 1)
compile(_s, str(_ps), 'exec')
_ps.write_text(_s)
print('Calibrated dual-seed runtime patch applied')


# Frozen label-free frame retention guard.
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
for _guard_old_log in WORKING_DIR.glob("retention_guard_*.jsonl"):
    _guard_old_log.unlink()

_s = _ps.read_text()
_guard_old = """                    det_logits[f] = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )"""
_guard_new = """                    blended_det = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )
                    primary_candidates = len(_detect_cells_pooled(
                        primary_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    blended_candidates = len(_detect_cells_pooled(
                        blended_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    minimum_retention = float(os.environ.get(
                        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION",
                        "0.90",
                    ))
                    candidate_retention = (
                        blended_candidates / primary_candidates
                        if primary_candidates
                        else 1.0
                    )
                    use_primary_detection = bool(
                        primary_candidates > 0
                        and candidate_retention < minimum_retention
                    )
                    det_logits[f] = (
                        primary_det if use_primary_detection else blended_det
                    )
                    if int(frame_indices[f]) not in seen_frames:
                        shard = os.environ.get(
                            "BIOHUB_GPU_SHARD", "single"
                        ).replace("/", "_")
                        guard_log = (
                            Path("/kaggle/working")
                            / f"retention_guard_{shard}.jsonl"
                        )
                        guard_record = {
                            "dataset": ds_path.stem,
                            "frame": int(frame_indices[f]),
                            "primary_candidates": int(primary_candidates),
                            "blended_candidates": int(blended_candidates),
                            "retention": float(candidate_retention),
                            "minimum_retention": float(minimum_retention),
                            "use_primary": bool(use_primary_detection),
                        }
                        with guard_log.open("a") as guard_handle:
                            guard_handle.write(
                                json.dumps(guard_record, sort_keys=True)
                                + "\\n"
                            )
                        if use_primary_detection:
                            print(
                                "BIOHUB_RETENTION_GUARD "
                                + json.dumps(guard_record, sort_keys=True),
                                flush=True,
                            )"""
_guard_matches = _s.count(_guard_old)
if _guard_matches != 1:
    raise RuntimeError(
        f"Retention guard expected one blend block, found {_guard_matches}"
    )
_s = _s.replace(_guard_old, _guard_new, 1)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Frozen frame retention guard applied at "
    + os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"]
)

# Fixed public reverse-time primary-association experiment. Detection, the
# secondary low-margin mix, ILP, and graph post-processing are unchanged.
import math as _bidirectional_math

_bidirectional_weight_guard = float(
    os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")
)
if not _bidirectional_math.isclose(
    _bidirectional_weight_guard, 0.30, rel_tol=0.0, abs_tol=1e-12
):
    raise ValueError({
        "expected_bidirectional_weight": 0.30,
        "actual_bidirectional_weight": _bidirectional_weight_guard,
    })

_s = _ps.read_text()
_bi_old = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n'
_bi_new = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            _bidirectional_weight = float(\n                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")\n            )\n            if _bidirectional_weight > 0.0:\n                reverse_logits_native = model.predict_edges(\n                    unet_feat_tgt, unet_feat_src,\n                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,\n                    p_pos_tgt, p_pos_src,\n                    p_mask_tgt, p_mask_src,\n                )  # (1, n_tgt, n_src)\n                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\n\n                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                forward_scale = edge_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)\n                reverse_scale = reverse_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)\n                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)\n                reverse_aligned = (\n                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio\n                    + forward_center\n                )\n                # Biohub 145: require mutual forward/reverse support in probability space.\n                # The harmonic mean penalizes a candidate when either temporal direction\n                # assigns it very low probability, while calibration preserves the forward\n                # logit scale used by the unchanged downstream candidate threshold and ILP.\n                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)\n                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)\n                harmonic_prob = 1.0 / (\n                    (1.0 - _bidirectional_weight) / forward_prob\n                    + _bidirectional_weight / reverse_prob\n                )\n                harmonic_prob = harmonic_prob / harmonic_prob.sum(\n                    dim=1, keepdim=True\n                ).clamp_min(1e-8)\n                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))\n                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)\n                harmonic_scale = harmonic_logits.std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)\n                edge_logits_pair = (\n                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio\n                    + forward_center\n                ).to(reverse_aligned.dtype)\n                del (\n                    reverse_logits_native,\n                    reverse_logits_pair,\n                    reverse_aligned,\n                    forward_prob,\n                    reverse_prob,\n                    harmonic_prob,\n                    harmonic_logits,\n                )\n            if secondary_model is not None:\n'
_bi_count = _s.count(_bi_old)
if _bi_count != 1:
    raise RuntimeError(
        f"Bidirectional edge patch expected one transformed block, found {_bi_count}"
    )
_s = _s.replace(_bi_old, _bi_new, 1)

_coordinate_manifest_old = '    coords = coords.astype(np.int16)\n    return coords, all_edges'
_coordinate_manifest_new = '    coords = coords.astype(np.int16)\n\n    # Label-free, pre-ILP detector-coordinate manifest. This executes inside\n    # predict_video, before build_graph and the ILP call in predict().\n    _coordinate_manifest_arm = os.environ.get(\n        "BIOHUB_DIAGNOSTIC_ARM", ""\n    ).strip()\n    if _coordinate_manifest_arm:\n        import hashlib as _coordinate_hashlib\n\n        _coordinate_shard = os.environ.get(\n            "BIOHUB_GPU_SHARD", "single"\n        ).replace("/", "_")\n        _coordinate_array = np.ascontiguousarray(\n            coords.astype("<i2", copy=False)\n        )\n        _coordinate_frame_counts = [\n            [int(_coordinate_t), int((_coordinate_array[:, 0] == _coordinate_t).sum())]\n            for _coordinate_t in np.unique(_coordinate_array[:, 0])\n        ]\n        _coordinate_record = {\n            "columns": ["t", "z", "y", "x"],\n            "coordinate_sha256": _coordinate_hashlib.sha256(\n                _coordinate_array.tobytes(order="C")\n            ).hexdigest(),\n            "dataset": ds_path.stem,\n            "dtype": "<i2",\n            "frame_counts": _coordinate_frame_counts,\n            "rows": int(len(_coordinate_array)),\n            "stage": "post_detection_pre_graph_pre_ilp",\n        }\n        _coordinate_manifest_path = (\n            Path("/kaggle/working")\n            / f"detector_coordinates_{_coordinate_manifest_arm}_"\n            f"{_coordinate_shard}.jsonl"\n        )\n        with _coordinate_manifest_path.open("a") as _coordinate_handle:\n            _coordinate_handle.write(\n                json.dumps(_coordinate_record, sort_keys=True) + "\\n"\n            )\n\n    return coords, all_edges'
_coordinate_manifest_count = _s.count(_coordinate_manifest_old)
if _coordinate_manifest_count != 1:
    raise RuntimeError(
        "Coordinate-manifest patch expected one pre-return block, found "
        f"{_coordinate_manifest_count}"
    )
_s = _s.replace(
    _coordinate_manifest_old, _coordinate_manifest_new, 1
)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Bidirectional harmonic-probability association fusion applied | weight=",
    _bidirectional_weight_guard,
)
print("Pre-ILP detector-coordinate manifest hook applied")



In [ ]:
E31_FILES = {'src/biohub/__init__.py': '"""Biohub cell-tracking working package (torch-free local tooling)."""\n', 'src/biohub/io.py': '"""Light-weight readers for the competition data (no torch, no dask).\n\nImage volume : ``<name>.zarr`` (zarr v3), array ``0`` of shape (T, Z, Y, X),\n               uint16, one blosc2/zstd chunk per timepoint at ``0/c/{t}/0/0/0``.\nGround truth : ``<name>.geff`` (tracksdata graph). Nodes carry t, z, y, x in\n               *voxel* units; edges are (source_id, target_id) with t+1 targets.\nSubmission   : CSV with columns id, dataset, row_type, node_id, t, z, y, x,\n               source_id, target_id (node rows use -1 for source/target,\n               edge rows use -1 for node_id/t/z/y/x).\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport blosc2\nimport numpy as np\n\n# (Z, Y, X) micrometres per voxel; same as the official DEFAULT_SCALE.\nDEFAULT_SCALE: tuple[float, float, float] = (1.625, 0.40625, 0.40625)\n\nSUBMISSION_COLUMNS = ("dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id")\n\n\n@dataclass(frozen=True)\nclass ZarrVolume:\n    """Metadata handle for one video. Frames are decoded on demand."""\n\n    path: Path\n    shape: tuple[int, int, int, int]  # (T, Z, Y, X)\n    dtype: np.dtype\n    scale: tuple[float, float, float]\n\n    @property\n    def n_t(self) -> int:\n        return self.shape[0]\n\n    def frame(self, t: int) -> np.ndarray:\n        """Decode timepoint ``t`` -> (Z, Y, X) array (new, writable copy)."""\n        if not 0 <= t < self.n_t:\n            raise IndexError(f"t={t} out of range [0, {self.n_t})")\n        chunk = self.path / "0" / "c" / str(t) / "0" / "0" / "0"\n        raw = blosc2.decompress(chunk.read_bytes())\n        arr = np.frombuffer(raw, dtype=self.dtype)\n        expected = int(np.prod(self.shape[1:]))\n        if arr.size != expected:\n            raise ValueError(f"{chunk}: decoded {arr.size} values, expected {expected}")\n        return arr.reshape(self.shape[1:]).copy()\n\n\ndef read_scale(zarr_path: Path) -> tuple[float, float, float]:\n    """(Z, Y, X) voxel scale from the OME-NGFF multiscales attrs, else DEFAULT_SCALE."""\n    root_meta = zarr_path / "zarr.json"\n    if not root_meta.exists():\n        return DEFAULT_SCALE\n    attrs = json.loads(root_meta.read_text()).get("attributes", {})\n    # OME-NGFF 0.5 nests under "ome"; older layouts put multiscales at the top.\n    attrs = attrs.get("ome", attrs)\n    if "multiscales" not in attrs:\n        return DEFAULT_SCALE\n    transform = attrs["multiscales"][0]["datasets"][0]["coordinateTransformations"][0]\n    if transform.get("type") != "scale":\n        raise ValueError(f"{root_meta}: first transform is not \'scale\': {transform}")\n    z, y, x = (float(v) for v in transform["scale"][-3:])\n    return (z, y, x)\n\n\ndef open_volume(zarr_path: Path | str) -> ZarrVolume:\n    """Open ``<name>.zarr`` (metadata only; no image bytes are read)."""\n    zarr_path = Path(zarr_path)\n    meta_path = zarr_path / "0" / "zarr.json"\n    if not meta_path.exists():\n        raise FileNotFoundError(meta_path)\n    meta = json.loads(meta_path.read_text())\n    shape = tuple(int(s) for s in meta["shape"])\n    if len(shape) != 4:\n        raise ValueError(f"{meta_path}: expected 4-D (T,Z,Y,X) shape, got {shape}")\n    return ZarrVolume(\n        path=zarr_path,\n        shape=shape,  # type: ignore[arg-type]\n        dtype=np.dtype(meta["data_type"]),\n        scale=read_scale(zarr_path),\n    )\n\n\ndef list_videos(split_dir: Path | str, require_geff: bool = False) -> list[str]:\n    """Video stems under ``split_dir`` (sorted). With ``require_geff`` keep only GT-paired ones."""\n    split_dir = Path(split_dir)\n    stems = sorted(p.name[:-5] for p in split_dir.glob("*.zarr"))\n    if require_geff:\n        stems = [s for s in stems if (split_dir / f"{s}.geff").exists()]\n    return stems\n\n\ndef load_geff_graph(geff_path: Path | str):\n    """Load a ``.geff`` as a tracksdata graph (import deferred: tracksdata is slow to import)."""\n    import tracksdata as td\n\n    result = td.graph.IndexedRXGraph.from_geff(Path(geff_path))\n    return result[0] if isinstance(result, tuple) else result\n\n\ndef estimated_number_of_nodes(geff_path: Path | str) -> float:\n    """``estimated_number_of_nodes`` from the GEFF metadata extras (NaN if absent)."""\n    from geff import GeffMetadata\n\n    try:\n        meta = GeffMetadata.read(Path(geff_path))\n    except Exception:\n        return float("nan")\n    val = (meta.extra or {}).get("estimated_number_of_nodes")\n    return float(val) if val is not None else float("nan")\n', 'src/biohub/appearance_cost.py': '"""Appearance cost between cell embeddings (mean over two seed sources)."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n_DIM = 32\n_MAX_SIDE = 2048\n_MAX_DENSE = 1_000_000\n_TOL = 1e-12\n\n\ndef _check_structure(label, arr):\n    if not isinstance(arr, np.ndarray):\n        raise ValueError(f"{label} must be a numpy ndarray")\n    if arr.dtype != np.float32:\n        raise ValueError(f"{label} must have dtype float32")\n    if arr.ndim != 2:\n        raise ValueError(f"{label} must be 2-D (n_rows, {_DIM})")\n    if arr.shape[1] != _DIM:\n        raise ValueError(f"{label} feature dimension must be {_DIM}")\n\n\ndef _validate_pair(side, prim, sec, other_prim, other_sec):\n    for label, arr in ((f"primary_{side}", prim),\n                       (f"secondary_{side}", sec),\n                       (f"primary_{side}_counterpart", other_prim),\n                       (f"secondary_{side}_counterpart", other_sec)):\n        _check_structure(label, arr)\n    if prim.shape != sec.shape:\n        raise ValueError(f"primary/secondary {side} shapes differ: "\n                         f"{prim.shape} vs {sec.shape}")\n    n_src, n_tgt = prim.shape[0], other_prim.shape[0]\n    if max(n_src, n_tgt) > _MAX_SIDE:\n        raise ValueError(f"side length exceeds {_MAX_SIDE}")\n    if n_src * n_tgt > _MAX_DENSE:\n        raise ValueError(f"dense product {n_src * n_tgt} exceeds {_MAX_DENSE}")\n\n\ndef _normalize(label, arr):\n    if not np.all(np.isfinite(arr)):\n        raise ValueError(f"{label} contains non-finite values")\n    norms = np.linalg.norm(arr, axis=1)\n    if arr.shape[0] and np.any(norms == 0.0):\n        raise ValueError(f"{label} contains a zero-norm row")\n    return arr / norms[:, None]\n\n\ndef _cosine(source, target, label):\n    src = _normalize(f"{label} source", np.array(source, dtype=np.float64))\n    tgt = _normalize(f"{label} target", np.array(target, dtype=np.float64))\n    cos = src @ tgt.T\n    if not np.all(np.isfinite(cos)):\n        raise ValueError(f"{label} cosine matrix contains non-finite values")\n    if cos.size and (np.min(cos) < -1.0 - _TOL or np.max(cos) > 1.0 + _TOL):\n        raise ValueError(f"{label} cosine outside [-1, 1]")\n    return np.clip(cos, -1.0, 1.0)\n\n\ndef appearance_penalty(primary_source, primary_target,\n                       secondary_source, secondary_target):\n    """Return mean of per-seed (1 - cosine) costs as float64 (n_s, n_t)."""\n    _validate_pair("source", primary_source, secondary_source,\n                   primary_target, secondary_target)\n    _validate_pair("target", primary_target, secondary_target,\n                   primary_source, secondary_source)\n    cos_p = _cosine(primary_source, primary_target, "primary")\n    cos_s = _cosine(secondary_source, secondary_target, "secondary")\n    return ((1.0 - cos_p) + (1.0 - cos_s)) / 2.0\n\n\ndef frame_appearance_penalty(\n    frame: dict,\n    t: int,\n    source_ids: list[int],\n    target_ids: list[int],\n) -> np.ndarray:\n    """Validate one appearance feature frame and return its penalty matrix.\n\n    The frame must be an exact dict with the eight canonical keys. Source and\n    target ID lists passed here must exactly equal the frame\'s stored lists\n    (same order). The returned array is the existing\n    ``appearance_penalty`` output for the four feature blocks in primary\n    source, primary target, secondary source, secondary target order.\n    """\n    expected_keys = (\n        "t_source",\n        "t_target",\n        "source_ids",\n        "target_ids",\n        "primary_source_features",\n        "primary_target_features",\n        "secondary_source_features",\n        "secondary_target_features",\n    )\n\n    if type(frame) is not dict or set(frame) != set(expected_keys):\n        raise ValueError("frame must be a dict with exactly the eight canonical keys")\n\n    def _check_builtin_int(value: object, name: str) -> int:\n        if type(value) is not int:\n            raise ValueError(f"{name} must be a builtin int")\n        return value\n\n    t_value = _check_builtin_int(t, "t")\n    t_source = _check_builtin_int(frame["t_source"], "t_source")\n    t_target = _check_builtin_int(frame["t_target"], "t_target")\n\n    if not 0 <= t_value <= 98:\n        raise ValueError("t must satisfy 0 <= t <= 98")\n    if t_source != t_value or t_target != t_value + 1:\n        raise ValueError("frame t_source/t_target must equal t/t+1")\n\n    def _check_id_list(value: object, name: str) -> list[int]:\n        if type(value) is not list:\n            raise ValueError(f"{name} must be a list of unique builtin ints")\n        seen = set()\n        for item in value:\n            if type(item) is not int:\n                raise ValueError(f"{name} must be a list of unique builtin ints")\n            if item in seen:\n                raise ValueError(f"{name} contains duplicate ids")\n            seen.add(item)\n        return value\n\n    frame_sources = _check_id_list(frame["source_ids"], "frame source_ids")\n    frame_targets = _check_id_list(frame["target_ids"], "frame target_ids")\n    call_sources = _check_id_list(source_ids, "source_ids")\n    call_targets = _check_id_list(target_ids, "target_ids")\n\n    if frame_sources != call_sources:\n        raise ValueError("source_ids must exactly equal frame source_ids including order")\n    if frame_targets != call_targets:\n        raise ValueError("target_ids must exactly equal frame target_ids including order")\n\n    if set(call_sources) & set(call_targets):\n        raise ValueError("source_ids and target_ids must not overlap")\n\n    penalties = appearance_penalty(\n        frame["primary_source_features"],\n        frame["primary_target_features"],\n        frame["secondary_source_features"],\n        frame["secondary_target_features"],\n    )\n\n    if penalties.shape != (len(call_sources), len(call_targets)):\n        raise ValueError("appearance_penalty shape mismatch for frame id lists")\n\n    return penalties\n', 'src/biohub/consensus_edges.py': '"""Reciprocal-consensus edge helper for Biohub cell tracking."""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n_MAX_NS = 2048\n_MAX_NT = 2048\n_MAX_PACKET = 1_000_000\n\n\ndef _validate(matrix: np.ndarray, name: str) -> tuple[int, int]:\n    if type(matrix) is not np.ndarray:\n        raise ValueError(f"{name} must be an exact np.ndarray")\n    if matrix.dtype != np.float32:\n        raise ValueError(f"{name} must have dtype float32")\n    if matrix.ndim != 2:\n        raise ValueError(f"{name} must be 2-dimensional")\n    if not np.isfinite(matrix).all():\n        raise ValueError(f"{name} must be finite")\n    ns, nt = matrix.shape\n    if ns > _MAX_NS or nt > _MAX_NT:\n        raise ValueError(f"{name} exceeds axis cap {_MAX_NS}/{_MAX_NT}")\n    if ns * nt > _MAX_PACKET:\n        raise ValueError(f"{name} exceeds packet size {_MAX_PACKET}")\n    return ns, nt\n\n\ndef _unique_max_along_axis(logits: np.ndarray, axis: int) -> np.ndarray:\n    """Boolean mask that is True only where logits has a unique maximum."""\n    best = logits.max(axis=axis, keepdims=True)\n    hits = logits == best\n    counts = hits.sum(axis=axis, keepdims=True)\n    return hits & (counts == 1)\n\n\ndef reciprocal_consensus_pairs(\n    primary_forward: np.ndarray,\n    primary_reverse: np.ndarray,\n    secondary_forward: np.ndarray,\n) -> list[tuple[int, int]]:\n    """Return (source, target) pairs agreed by all three detectors.\n\n    A pair (i, j) is kept when i is the unique argmax of both forward matrices\n    in column j and j is the unique argmax of the reverse matrix in row i.\n    Logits are compared within a model/direction only. Ties are excluded.\n    """\n    shapes = (\n        _validate(primary_forward, "primary_forward"),\n        _validate(primary_reverse, "primary_reverse"),\n        _validate(secondary_forward, "secondary_forward"),\n    )\n    ns, nt = shapes[0]\n    if any(shape != shapes[0] for shape in shapes[1:]):\n        raise ValueError("all matrices must share identical (ns, nt) shapes")\n    if ns == 0 or nt == 0:\n        return []\n    src_ok = _unique_max_along_axis(primary_forward, axis=0) & _unique_max_along_axis(secondary_forward, axis=0)\n    tgt_ok = _unique_max_along_axis(primary_reverse, axis=1)\n    mask = src_ok & tgt_ok\n    rows, cols = np.nonzero(mask)\n    order = np.lexsort((cols, rows))\n    return [(int(i), int(j)) for i, j in zip(rows[order], cols[order], strict=True)]\n\n\ndef positive_reciprocal_consensus_pairs(\n    primary_forward: np.ndarray,\n    primary_reverse: np.ndarray,\n    secondary_forward: np.ndarray,\n) -> list[tuple[int, int]]:\n    """Return reciprocal pairs whose three selected logits are strictly positive."""\n    pairs = reciprocal_consensus_pairs(\n        primary_forward, primary_reverse, secondary_forward\n    )\n    return [\n        (i, j)\n        for i, j in pairs\n        if primary_forward[i, j] > 0\n        and primary_reverse[i, j] > 0\n        and secondary_forward[i, j] > 0\n    ]\n\n\ndef primary_reciprocal_consensus_pairs(\n    primary_forward: np.ndarray,\n    primary_reverse: np.ndarray,\n) -> list[tuple[int, int]]:\n    """Return strict reciprocal unique-argmax pairs from the primary model only."""\n    forward_shape = _validate(primary_forward, "primary_forward")\n    reverse_shape = _validate(primary_reverse, "primary_reverse")\n    if forward_shape != reverse_shape:\n        raise ValueError("primary matrices must share identical shapes")\n    ns, nt = forward_shape\n    if ns == 0 or nt == 0:\n        return []\n    src_ok = _unique_max_along_axis(primary_forward, axis=0)\n    tgt_ok = _unique_max_along_axis(primary_reverse, axis=1)\n    rows, cols = np.nonzero(src_ok & tgt_ok)\n    order = np.lexsort((cols, rows))\n    return [(int(i), int(j)) for i, j in zip(rows[order], cols[order], strict=True)]\n\n\ndef map_consensus_to_raw(pairs, source_detector_indices, target_detector_indices, graph_to_detector, raw_node_ids):\n    lists = (pairs, source_detector_indices, target_detector_indices, raw_node_ids)\n    if any(type(x) is not list for x in lists):\n        raise ValueError("list inputs must be exact lists")\n    if type(graph_to_detector) is not dict:\n        raise ValueError("graph_to_detector must be an exact dict")\n\n    def is_int(v):\n        return type(v) is int and v >= 0\n\n    for g, d in graph_to_detector.items():\n        if not (is_int(g) and is_int(d)):\n            raise ValueError("mapping keys/values must be nonnegative ints")\n    graphs = list(graph_to_detector)\n    dets = list(graph_to_detector.values())\n    if len(set(graphs)) != len(graphs) or len(set(dets)) != len(dets):\n        raise ValueError("mapping must be one-to-one")\n\n    for name, arr in (("source", source_detector_indices), ("target", target_detector_indices), ("raw", raw_node_ids)):\n        if any(not is_int(v) for v in arr):\n            raise ValueError(f"{name} entries must be nonnegative ints")\n        if len(set(arr)) != len(arr):\n            raise ValueError(f"{name} entries must be unique")\n\n    if set(source_detector_indices) & set(target_detector_indices):\n        raise ValueError("source/target detector sets must be disjoint")\n    mapped = set(dets)\n    for v in source_detector_indices + target_detector_indices:\n        if v not in mapped:\n            raise ValueError("detector index missing from mapping values")\n    keyset = set(graphs)\n    for v in raw_node_ids:\n        if v not in keyset:\n            raise ValueError("raw id missing from mapping keys")\n\n    ns, nt = len(source_detector_indices), len(target_detector_indices)\n    for p in pairs:\n        if type(p) is not tuple or len(p) != 2 or not all(is_int(x) for x in p):\n            raise ValueError("pairs must be tuples of two nonnegative ints")\n    if pairs != sorted(pairs):\n        raise ValueError("pairs must be sorted")\n    if len(set(pairs)) != len(pairs):\n        raise ValueError("pairs must be unique")\n    for i, j in pairs:\n        if not (0 <= i < ns and 0 <= j < nt):\n            raise ValueError("pair indices out of range")\n    if len({i for i, _ in pairs}) != len(pairs):\n        raise ValueError("repeated source row")\n    if len({j for _, j in pairs}) != len(pairs):\n        raise ValueError("repeated target column")\n\n    inverse = {d: g for g, d in graph_to_detector.items()}\n    rawset = set(raw_node_ids)\n    out = []\n    for i, j in pairs:\n        sg = inverse[source_detector_indices[i]]\n        tg = inverse[target_detector_indices[j]]\n        if sg in rawset and tg in rawset:\n            out.append((sg, tg))\n    return sorted(out)\n', 'src/biohub/primary_consensus_observer.py': '"""Primary-consensus association observer for the Biohub tracking reconstruction.\n\nA passive, stateful hook receiver. It records the primary-only reciprocal-\nconsensus edge decision surface for every time transition without influencing\nthe run. Only primary logits are consulted; secondary/mixed/features are\naccepted for protocol compatibility but never copied or stored.\n"""\n\nfrom __future__ import annotations\n\nimport functools\nfrom typing import Any\n\nimport numpy as np\n\nfrom biohub.consensus_edges import (\n    map_consensus_to_raw,\n    primary_reciprocal_consensus_pairs,\n)\n\nCONFIG = {\n    "window_size": 2,\n    "downsample": (1, 4, 4),\n    "scale": (1.625, 0.40625, 0.40625),\n    "activation": "softmax",\n    "threshold": 0.48,\n    "max_parents": None,\n    "max_children": None,\n}\n\n_PAIR = "pair"\n_PRIMARY = "primary"\n_REVERSE = "reverse"\n_SECONDARY = "secondary"\n_MIXED = "mixed"\n_NEXT = {\n    _PAIR: _PRIMARY,\n    _PRIMARY: _REVERSE,\n    _REVERSE: _SECONDARY,\n    _SECONDARY: _MIXED,\n    _MIXED: _PAIR,\n}\n_NODE_KEYS = ("node_id", "t", "z", "y", "x")\n\n\ndef _require(condition: bool, message: str) -> None:\n    if not condition:\n        raise ValueError(message)\n\n\ndef _is_int(value: Any) -> bool:\n    return isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))\n\n\ndef _fail_closed(where: str):\n    """Decorator that marks the observer permanently failed on any exception."""\n\n    def decorate(func):\n        @functools.wraps(func)\n        def wrapper(self, *args, **kwargs):\n            if getattr(self, "_failed", False):\n                raise ValueError("observer is permanently failed")\n            try:\n                return func(self, *args, **kwargs)\n            except ValueError:\n                self._failed = True\n                raise\n            except Exception as exc:\n                self._failed = True\n                raise ValueError(f"{where} failed: {type(exc).__name__}: {exc}") from exc\n\n        return wrapper\n\n    return decorate\n\n\nclass _Video:\n    __slots__ = (\n        "name",\n        "frames",\n        "next_t",\n        "ranges",\n        "local_pairs",\n        "state",\n        "forward",\n        "final_coords",\n        "graph_to_detector",\n        "selected_ids",\n    )\n\n    def __init__(self, name: str, frames: int):\n        self.name = name\n        self.frames = frames\n        self.next_t = 0\n        self.ranges: list[tuple[int, int, int, int, int]] = []\n        self.local_pairs: dict[int, list[tuple[int, int]]] = {}\n        self.state: str = _PAIR\n        self.forward: np.ndarray | None = None\n        self.final_coords: np.ndarray | None = None\n        self.graph_to_detector: dict[int, int] | None = None\n        self.selected_ids: set[int] | None = None\n\n\nclass PrimaryConsensusObserver:\n    """Collect primary-only consensus decisions across videos; never mutate graphs."""\n\n    def __init__(self):\n        self._videos: dict[str, _Video] = {}\n        self._active: _Video | None = None\n        self._failed: bool = False\n\n    # ------------------------------------------------------------- video setup\n\n    @_fail_closed("start_video")\n    def start_video(\n        self,\n        name: str,\n        frames: int,\n        window_size: int,\n        downsample: tuple[int, ...],\n        scale: tuple[float, ...],\n        activation: str,\n        *,\n        threshold: float = 0.48,\n        max_parents: int | None = None,\n        max_children: int | None = None,\n    ) -> str:\n        _require(isinstance(name, str) and bool(name), "name must be a nonempty str")\n        _require(_is_int(frames) and int(frames) > 0, "frames must be a positive integer")\n        _require(name not in self._videos, f"duplicate video name {name!r}")\n        _require(self._active is None, "previous video is still incomplete")\n        _require(window_size == CONFIG["window_size"], "window_size must be 2")\n        _require(tuple(downsample) == CONFIG["downsample"], "downsample must be (1, 4, 4)")\n        _require(tuple(scale) == CONFIG["scale"], "scale must be (1.625, 0.40625, 0.40625)")\n        _require(activation == CONFIG["activation"], "activation must be \'softmax\'")\n        _require(threshold == CONFIG["threshold"], "threshold must be 0.48")\n        _require(max_parents is None, "max_parents must be None")\n        _require(max_children is None, "max_children must be None")\n        for v in self._videos.values():\n            _require(\n                v.selected_ids is not None,\n                "previous video has not completed selected_graph",\n            )\n        self._active = _Video(name, int(frames))\n        return self._active.state\n\n    # ------------------------------------------------------------ transitions\n\n    @_fail_closed("start_pair")\n    def start_pair(\n        self,\n        t: int,\n        s_src: int,\n        e_src: int,\n        s_tgt: int,\n        e_tgt: int,\n        coords: np.ndarray,\n    ) -> str:\n        video = self._active_video("start_pair")\n        _require(video.state == _PAIR, "start_pair requires pair-ready state")\n        _require(\n            _is_int(t) and int(t) == video.next_t,\n            f"transition sequence error: expected {video.next_t}",\n        )\n        bounds = [s_src, e_src, s_tgt, e_tgt]\n        _require(all(_is_int(v) for v in bounds), "range bounds must be ints")\n        s_src, e_src, s_tgt, e_tgt = (int(v) for v in bounds)\n        _require(0 <= s_src <= e_src == s_tgt <= e_tgt, "invalid or inconsistent range bounds")\n        _require(_is_int(t) and int(t) < video.frames - 1, "t must be less than frames-1")\n        n = self._check_coords(coords)\n        _require(e_src <= n and e_tgt <= n, "range bounds exceed coords length")\n        _require(\n            bool((coords[s_src:e_src, 0] == int(t)).all()),\n            "source block times must equal t",\n        )\n        _require(\n            bool((coords[s_tgt:e_tgt, 0] == int(t) + 1).all()),\n            "target block times must equal t+1",\n        )\n        if video.next_t == 0:\n            _require(s_src == 0, "first transition must start at source 0")\n        else:\n            prev = video.ranges[-1]\n            _require(\n                (s_src, e_src) == (prev[3], prev[4]),\n                "source range must equal the previous target range",\n            )\n        video.ranges.append((t, s_src, e_src, s_tgt, e_tgt))\n        video.local_pairs[video.next_t] = []\n        if s_src == e_src or s_tgt == e_tgt:\n            # Empty transition: skip ALL tensor hooks, return pair-ready immediately.\n            video.next_t += 1\n            video.state = _PAIR\n            return video.state\n        video.state = _PRIMARY\n        return video.state\n\n    @staticmethod\n    def _check_coords(coords: np.ndarray) -> int:\n        _require(type(coords) is np.ndarray, "coords must be an exact ndarray")\n        _require(coords.ndim == 2 and coords.shape[1] == 4, "coords must have shape (n, 4)")\n        _require(\n            np.issubdtype(coords.dtype, np.floating) or np.issubdtype(coords.dtype, np.integer),\n            "grid coords must use a numeric dtype",\n        )\n        _require(not np.issubdtype(coords.dtype, np.bool_), "grid coords must not be bool")\n        _require(not np.issubdtype(coords.dtype, np.complexfloating), "grid coords must not be complex")\n        _require(coords.dtype != object, "grid coords must not be object dtype")\n        _require(bool(np.isfinite(coords).all()), "grid coords must be finite")\n        _require(bool((coords >= 0).all()), "grid coords must be nonnegative")\n        return int(coords.shape[0])\n\n    # ------------------------------------------------------------ logits hooks\n\n    @_fail_closed("primary")\n    def primary(self, logits: Any, source: Any, target: Any) -> str:\n        video = self._hook_state(_PRIMARY, "primary")\n        matrix = self._matrix(logits, "primary")\n        r = video.ranges[-1]\n        _require(matrix.shape == (r[2] - r[1], r[4] - r[3]), "primary logits shape must match current ranges")\n        video.forward = matrix\n        video.state = _REVERSE\n        return video.state\n\n    @_fail_closed("reverse")\n    def reverse(self, transposed_logits: Any) -> str:\n        video = self._hook_state(_REVERSE, "reverse")\n        matrix = self._matrix(transposed_logits, "reverse")\n        fwd = video.forward\n        _require(fwd is not None, "reverse called without forward")\n        _require(\n            matrix.shape == fwd.shape,\n            "reverse logits shape must match primary logits shape",\n        )\n        video.local_pairs[video.next_t] = primary_reciprocal_consensus_pairs(fwd, matrix)\n        video.forward = None\n        video.state = _SECONDARY\n        return video.state\n\n    @_fail_closed("secondary")\n    def secondary(self, logits: Any, source: Any, target: Any) -> str:\n        video = self._hook_state(_SECONDARY, "secondary")\n        video.state = _MIXED\n        return video.state\n\n    @_fail_closed("mixed")\n    def mixed(self, raw: Any, probabilities: Any) -> str:\n        video = self._hook_state(_MIXED, "mixed")\n        video.next_t += 1\n        video.state = _PAIR\n        return video.state\n\n    def _active_video(self, where: str) -> _Video:\n        _require(self._active is not None, f"{where} needs an active video")\n        return self._active\n\n    def _hook_state(self, expected: str, where: str) -> _Video:\n        video = self._active_video(where)\n        _require(\n            video.state == expected,\n            f"{where} requires state \'{expected}\', found \'{video.state}\'",\n        )\n        return video\n\n    @staticmethod\n    def _matrix(logits: Any, name: str) -> np.ndarray:\n        array = logits.detach().cpu().numpy()\n        _require(type(array) is np.ndarray, f"{name} logits must yield ndarray")\n        _require(array.dtype == np.float32, f"{name} logits must be float32")\n        _require(\n            array.ndim == 3 and array.shape[0] == 1,\n            f"{name} logits must have shape (1, ns, nt)",\n        )\n        _require(np.isfinite(array).all(), f"{name} logits must be finite")\n        return np.array(array[0], copy=True)\n\n    # ------------------------------------------------------------- graph hooks\n\n    @_fail_closed("end_video")\n    def end_video(self, coords: np.ndarray, edges: Any) -> str:\n        video = self._active_video("end_video")\n        _require(video.state == _PAIR, "end_video requires no pending logits hooks")\n        _require(\n            video.next_t == video.frames - 1,\n            "end_video requires every declared transition",\n        )\n        n = self._check_coords(coords)\n        if video.frames > 1:\n            expected = video.ranges[-1][4]\n        else:\n            _require(bool((coords[:, 0] == 0).all()), "single-frame coords times must equal zero")\n            expected = n\n        _require(n == expected, "final coords length must equal the last target range end")\n        for t_val, s_src, e_src, s_tgt, e_tgt in video.ranges:\n            _require(\n                bool((coords[s_src:e_src, 0] == t_val).all()),\n                "source block times must equal the transition index",\n            )\n            _require(\n                bool((coords[s_tgt:e_tgt, 0] == t_val + 1).all()),\n                "target block times must equal the following frame",\n            )\n        video.final_coords = np.array(coords, copy=True)\n        self._videos[video.name] = video\n        self._active = None\n        return video.name\n\n    @_fail_closed("pre_graph")\n    def pre_graph(\n        self,\n        coords: np.ndarray,\n        edges: Any,\n        node_ids: list[int],\n        graph: Any,\n    ) -> str:\n        video = self._pending("pre_graph")\n        n = self._check_coords(coords)\n        _require(\n            n == video.final_coords.shape[0],\n            "pre_graph coords length must match end_video coords",\n        )\n        _require(\n            np.array_equal(coords, video.final_coords),\n            "pre_graph coords must equal the end_video coordinates",\n        )\n        ids = self._as_node_ids(node_ids, n)\n        rows = list(graph.node_attrs(attr_keys=list(_NODE_KEYS)).iter_rows(named=True))\n        _require(len(rows) == n, "graph node count must match coords count")\n        by_id = {row["node_id"]: row for row in rows}\n        _require(len(by_id) == n, "duplicate node ids in graph")\n        _require(set(by_id) == set(ids), "graph node ids must equal node_ids set")\n        for index, gid in enumerate(ids):\n            row = by_id[gid]\n            got = np.asarray([row["t"], row["z"], row["y"], row["x"]])\n            _require(got.shape == (4,), "graph coordinate arity mismatch")\n            _require(\n                bool((got == video.final_coords[index]).all()),\n                "graph coordinates must equal the voxel coordinates",\n            )\n        video.graph_to_detector = {gid: idx for idx, gid in enumerate(ids)}\n        return video.name\n\n    @staticmethod\n    def _as_node_ids(node_ids: Any, expected_len: int) -> list[int]:\n        _require(\n            not isinstance(node_ids, dict),\n            "node_ids is a positional list of graph ids, not a dict",\n        )\n        try:\n            items = list(node_ids)\n        except TypeError as exc:\n            raise ValueError("node_ids must be iterable") from exc\n        _require(\n            len(items) == expected_len,\n            "node_ids length must match the coords count",\n        )\n        out: list[int] = []\n        for value in items:\n            _require(\n                _is_int(value) and int(value) >= 0,\n                "node_ids entries must be nonnegative ints",\n            )\n            out.append(int(value))\n        _require(len(set(out)) == len(out), "node_ids must be unique")\n        return out\n\n    @_fail_closed("selected_graph")\n    def selected_graph(self, graph: Any) -> str:\n        video = self._pending("selected_graph")\n        _require(\n            video.graph_to_detector is not None,\n            "selected_graph requires pre_graph",\n        )\n        _require(\n            video.selected_ids is None,\n            "selected_graph already completed for this video",\n        )\n        rows = list(graph.node_attrs(attr_keys=list(_NODE_KEYS)).iter_rows(named=True))\n        # Empty selected graph is allowed.\n        selected = self._as_node_ids([row["node_id"] for row in rows], len(rows))\n        _require(\n            set(selected).issubset(video.graph_to_detector),\n            "selected nodes must be a subset of the pre-graph nodes",\n        )\n        for row in rows:\n            pos = video.graph_to_detector[row["node_id"]]\n            got = np.asarray([row["t"], row["z"], row["y"], row["x"]])\n            _require(\n                bool((got == video.final_coords[pos]).all()),\n                "selected node coordinates must be unchanged",\n            )\n        video.selected_ids = set(selected)\n        return video.name\n\n    def _pending(self, where: str) -> _Video:\n        _require(\n            self._active is None,\n            f"{where} requires end_video to close the active video",\n        )\n        if where == "pre_graph":\n            waiting = [v for v in self._videos.values() if v.final_coords is not None and v.graph_to_detector is None]\n        else:\n            waiting = [v for v in self._videos.values() if v.graph_to_detector is not None and v.selected_ids is None]\n        _require(not any(w.selected_ids is not None for w in waiting), "already completed or no video awaiting graph")\n        _require(bool(waiting), "already completed or no video awaiting graph")\n        return waiting[-1]\n\n    # ----------------------------------------------------------------- lookup\n\n    @_fail_closed("frames_for")\n    def frames_for(self, dataset: str, raw_nodes: dict[int, dict[str, Any]]) -> dict[int, list[tuple[int, int]]]:\n        _require(dataset in self._videos, f"unknown dataset {dataset!r}")\n        video = self._videos[dataset]\n        _require(\n            video.selected_ids is not None,\n            f"{dataset} has not completed selected_graph",\n        )\n        _require(type(raw_nodes) is dict, "raw_nodes must be an exact dict")\n        mapping = video.graph_to_detector\n        times: dict[int, int] = {}\n        for node_id, record in raw_nodes.items():\n            _require(\n                _is_int(node_id) and int(node_id) >= 0,\n                "raw node keys must be nonnegative ints",\n            )\n            node_id = int(node_id)\n            _require(isinstance(record, dict), "raw node records must be dicts")\n            _require(\n                set(record) >= set(_NODE_KEYS),\n                "raw node records need at least node_id, t, z, y, x",\n            )\n            _require(\n                _is_int(record["node_id"]) and _is_int(record["t"]),\n                "raw node_id and t must be ints",\n            )\n            _require(\n                record["node_id"] == node_id,\n                "raw node key must match its node_id attribute",\n            )\n            _require(\n                node_id in mapping,\n                "raw node id is outside the recorded graph mapping",\n            )\n            _require(\n                node_id in video.selected_ids,\n                "raw node id is not in the selected graph",\n            )\n            want = video.final_coords[mapping[node_id]]\n            got = np.asarray([record["t"], record["z"], record["y"], record["x"]])\n            _require(got.shape == (4,), "raw coordinate arity mismatch")\n            _require(\n                bool((got == want).all()),\n                "raw coordinates must match the recorded voxel coords",\n            )\n            times[node_id] = int(record["t"])\n        present = set(times.values())\n        result: dict[int, list[tuple[int, int]]] = {}\n        for t_idx in range(video.frames - 1):\n            if t_idx not in present or t_idx + 1 not in present:\n                continue\n            _, s_src, e_src, s_tgt, e_tgt = video.ranges[t_idx]\n            raw_ids = [gid for gid, stamp in times.items() if stamp in (t_idx, t_idx + 1)]\n            result[t_idx] = map_consensus_to_raw(\n                video.local_pairs[t_idx],\n                list(range(s_src, e_src)),\n                list(range(s_tgt, e_tgt)),\n                mapping,\n                raw_ids,\n            )\n        return result\n', 'src/biohub/association_instrumentation.py': '"""Static-only, reversible observer insertion into the pinned E23 reconstruction."""\n\nimport ast\nimport hashlib\n\nSOURCE_SHA256 = "8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de"\n\n# Anchors are never replaced. Each addition is removable byte-for-byte.\nINSERTIONS = (\n    ("    edges: list[tuple[int, int, float, float]],\\n", "    association_observer=None,\\n"),\n    ("    secondary_low_margin_max: float = 0.2,\\n", "    association_observer=None,\\n"),\n    ("    evaluate: bool = False,\\n", "    association_observer=None,\\n"),\n    ("    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.start_video(\\n"\n     "            ds_path.stem, T, W, downsample, ds.scale, cfg.edge_activation,\\n"\n     "            threshold=cfg.threshold, max_parents=cfg.max_parents_per_node,\\n"\n     "            max_children=cfg.max_children_per_node)\\n"),\n    ("            s_tgt, e_tgt = coord_offset[t_tgt]\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.start_pair(t_src, s_src, e_src, s_tgt, e_tgt, coords_so_far)\\n"),\n    ("            )  # (1, n_src, n_tgt)\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.primary(edge_logits_pair, unet_feat_src, unet_feat_tgt)\\n"),\n    ("                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.reverse(reverse_logits_pair)\\n"),\n    ("                    p_mask_src, p_mask_tgt,\\n                )\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.secondary(\\n"\n     "                        secondary_logits_pair, secondary_feat_src, secondary_feat_tgt)\\n"),\n    ("                probs = torch.sigmoid(raw).cpu().numpy()\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.mixed(raw, probs)\\n"),\n    ("    return coords, all_edges\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.end_video(coords, all_edges)\\n"),\n    ("    return graph\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.pre_graph(coords, edges, node_ids, graph)\\n"),\n    ("                secondary_low_margin_max=secondary_low_margin_max,\\n",\n     "                association_observer=association_observer,\\n"),\n    ("        graph = build_graph(coords, edges)\\n",\n     ""),\n    ("        save_graph(graph, output_dir / f\\"{name}.geff\\")\\n",\n     "        if association_observer is not None:\\n"\n     "            association_observer.selected_graph(graph)\\n"),\n)\n\n\ndef instrument_source(source: str) -> tuple[str, dict]:\n    """Return text and a receipt. Never imports or executes the supplied source.\n\n    The build_graph call needs a keyword, so insert it before the closing paren\n    instead of replacing the call. All original executable AST nodes survive.\n    """\n    digest = hashlib.sha256(source.encode()).hexdigest()\n    if digest != SOURCE_SHA256:\n        raise ValueError("E23 reconstructed source SHA mismatch")\n    patched = source\n    additions = []\n    for index, (anchor, body) in enumerate(INSERTIONS):\n        if patched.count(anchor) != 1:\n            raise ValueError(f"observer anchor {index} is not unique")\n        if not body:  # build_graph keyword insertion, not a replacement function\n            original = "build_graph(coords, edges)"\n            addition = ", association_observer=association_observer"\n            patched = patched.replace(original, original[:-1] + addition + ")", 1)\n            additions.append((original[:-1] + addition + ")", original))\n            continue\n        # End-of-function and post-solver hooks must precede return/save.\n        before = anchor.lstrip().startswith(("return ", "save_graph("))\n        replacement = body + anchor if before else anchor + body\n        patched = patched.replace(anchor, replacement, 1)\n        additions.append((replacement, anchor))\n    restored = patched\n    for inserted, original in reversed(additions):\n        if restored.count(inserted) != 1:\n            raise ValueError("observer removal is ambiguous")\n        restored = restored.replace(inserted, original, 1)\n    if restored != source or ast.dump(ast.parse(restored)) != ast.dump(ast.parse(source)):\n        raise ValueError("original E23 program changed")\n    ast.parse(patched)\n    receipt = {"status": "STATIC_OBSERVER_INSERTION_ONLY", "original_sha256": digest,\n               "instrumented_sha256": hashlib.sha256(patched.encode()).hexdigest(),\n               "restored_bytes_equal": True, "restored_ast_equal": True,\n               "insertions": len(additions), "source_executed": False,\n               "inference_parity_verified": False, "submission_authorized": False}\n    return patched, receipt\n', 'src/biohub/output_bounds.py': '"""Bounds-correcting serializer for Biohub output node rows.\n\nThis module repairs missing *upper* coordinate bounds only. Graph/topology,\nrow order, node/time/row IDs, sentinels and model settings are preserved.\nRounding is Python\'s built-in ``round`` (ties-to-even). No dataset IDs, node\nIDs or image dimensions are hardcoded in runtime logic.\n"""\n\nimport json\nfrom fractions import Fraction\nfrom numbers import Integral, Real\nfrom pathlib import Path\n\nCSV_NODE_FIELDS = (\n    "id",\n    "dataset",\n    "row_type",\n    "node_id",\n    "t",\n    "z",\n    "y",\n    "x",\n    "source_id",\n    "target_id",\n)\n\nCSV_SAMPLE_FIELDS = (\n    "dataset",\n    "row_id",\n    "node_id",\n    "t",\n    "axis",\n    "original_float",\n    "rounded_int",\n    "clipped_int",\n    "signed_delta",\n    "absolute_delta",\n)\n\nREPORT_FIELDS = (\n    "dataset",\n    "shape",\n    "node_count",\n    "corrected_nodes",\n    "axis_counts",\n    "max_absolute_correction",\n    "samples",\n    "sample_limit",\n    "samples_truncated",\n)\n\nAXIS_ORDER = ("T", "Z", "Y", "X")\nCOORD_AXES = ("z", "y", "x")\nDIM_INDEX = {"z": 1, "y": 2, "x": 3}\n\nINT64_MIN = -(2**63)\nINT64_MAX = 2**63 - 1\n\nSAMPLE_LIMIT = 20\n\n\nclass OutputBoundsError(ValueError):\n    """Raised when metadata, a node row or a report fails validation."""\n\n\ndef _is_python_int(value: object) -> bool:\n    """True only for a genuine Python ``int`` (bool excluded)."""\n    return type(value) is int\n\n\ndef _validate_rank4_shape_input(shape: object) -> tuple[int, int, int, int]:\n    """Validate shape input before any coercion.\n\n    Rejects None, scalars, strings, generators and other non-sequence types\n    directly with OutputBoundsError. Only list/tuple of length 4 with positive\n    Python ints is accepted.\n    """\n    if isinstance(shape, (str, bytes)):\n        raise OutputBoundsError("array shape must be a rank-4 sequence")\n    if not isinstance(shape, (list, tuple)):\n        raise OutputBoundsError("array shape must be a rank-4 sequence")\n    if len(shape) != 4:\n        raise OutputBoundsError("array shape must be a rank-4 sequence")\n    dims: list[int] = []\n    for axis, value in zip(AXIS_ORDER, shape, strict=True):\n        if not _is_python_int(value):\n            raise OutputBoundsError(f"{axis} dimension is not a Python int")\n        if value <= 0:\n            raise OutputBoundsError(f"{axis} dimension must be positive")\n        dims.append(value)\n    return dims[0], dims[1], dims[2], dims[3]\n\n\ndef _positive_rank4_shape(shape: object) -> tuple[int, int, int, int]:\n    """Validate a rank-4 sequence of positive Python ints.\n\n    Accepts only list or tuple; used for already-extracted shape values from\n    JSON metadata where the container type is known to be safe.\n    """\n    if not isinstance(shape, (list, tuple)) or len(shape) != 4:\n        raise OutputBoundsError("array shape must be a rank-4 sequence")\n    dims: list[int] = []\n    for axis, value in zip(AXIS_ORDER, shape, strict=True):\n        if not _is_python_int(value):\n            raise OutputBoundsError(f"{axis} dimension is not a Python int")\n        if value <= 0:\n            raise OutputBoundsError(f"{axis} dimension must be positive")\n        dims.append(value)\n    return dims[0], dims[1], dims[2], dims[3]\n\n\ndef _multiscales_entry_paths(entry: object) -> list[str]:\n    """Return the raw ``datasets[*]["path"]`` list for a multiscales entry."""\n    if not isinstance(entry, dict):\n        raise OutputBoundsError("multiscales entries must be JSON objects")\n    datasets = entry.get("datasets")\n    if not isinstance(datasets, list) or not datasets:\n        raise OutputBoundsError(\n            "multiscales entries require a non-empty datasets list"\n        )\n    paths: list[str] = []\n    for item in datasets:\n        if not isinstance(item, dict) or "path" not in item:\n            raise OutputBoundsError(\n                "multiscales dataset entries require a path"\n            )\n        path = item["path"]\n        if not isinstance(path, str) or not path:\n            raise OutputBoundsError(\n                "multiscales dataset path must be a non-empty string"\n            )\n        paths.append(path)\n    return paths\n\n\ndef _entry_axes_are_tzyx(entry: dict) -> bool:\n    """True when the entry declares exactly T, Z, Y, X axes in that order."""\n    axes = entry.get("axes")\n    if not isinstance(axes, list) or len(axes) != 4:\n        return False\n    names: list[str] = []\n    for axis in axes:\n        if not isinstance(axis, dict):\n            return False\n        name = axis.get("name")\n        if not isinstance(name, str) or not name:\n            return False\n        names.append(name.upper())\n    return names == list(AXIS_ORDER)\n\n\ndef _validate_multiscales_axes(multiscales: list, dataset: str) -> None:\n    """Every multiscales entry must carry a non-empty axes list of dicts."""\n    for index, entry in enumerate(multiscales):\n        if not isinstance(entry, dict):\n            raise OutputBoundsError(\n                f"multiscales[{index}] must be a JSON object for {dataset}"\n            )\n        axes = entry.get("axes")\n        if not isinstance(axes, list) or not axes:\n            raise OutputBoundsError(\n                f"multiscales[{index}] requires a non-empty axes list "\n                f"for {dataset}"\n            )\n        for axis_index, axis in enumerate(axes):\n            if not isinstance(axis, dict):\n                raise OutputBoundsError(\n                    f"multiscales[{index}].axes[{axis_index}] must be a "\n                    f"JSON object for {dataset}"\n                )\n            name = axis.get("name")\n            if not isinstance(name, str) or not name:\n                raise OutputBoundsError(\n                    f"multiscales[{index}].axes[{axis_index}] requires a "\n                    f"non-empty name for {dataset}"\n                )\n\n\ndef read_output_shape(\n    test_dir: str | Path, dataset: str\n) -> tuple[int, int, int, int]:\n    """Return ``(T, Z, Y, X)`` for ``dataset`` from Zarr format-3 metadata.\n\n    Reads ``<dataset>.zarr/zarr.json`` and ``<dataset>.zarr/0/zarr.json``. No\n    shape fallback and no image-array reads are performed. Exactly one\n    reference to the literal path ``"0"`` is required across *all* multiscales\n    entries, counted before any axis filtering. Every multiscales entry must\n    declare a non-empty axes list; only the selected path-0 entry is required\n    to match the normalized T,Z,Y,X ordering.\n    """\n    if not isinstance(dataset, str) or not dataset.strip():\n        raise OutputBoundsError("dataset must be a non-empty string")\n    root = Path(test_dir) / f"{dataset}.zarr"\n\n    try:\n        group_meta = json.loads((root / "zarr.json").read_text(encoding="utf-8"))\n        array_meta = json.loads(\n            (root / "0" / "zarr.json").read_text(encoding="utf-8")\n        )\n    except OSError as exc:\n        raise OutputBoundsError(\n            f"missing zarr metadata for {dataset}: {exc}"\n        ) from exc\n    except json.JSONDecodeError as exc:\n        raise OutputBoundsError(\n            f"malformed zarr metadata for {dataset}: {exc}"\n        ) from exc\n    except UnicodeDecodeError as exc:\n        raise OutputBoundsError(\n            f"malformed zarr metadata encoding for {dataset}: {exc}"\n        ) from exc\n\n    if not isinstance(group_meta, dict) or not isinstance(array_meta, dict):\n        raise OutputBoundsError("zarr metadata must be JSON objects")\n\n    group_version = group_meta.get("zarr_format")\n    array_version = array_meta.get("zarr_format")\n    if not _is_python_int(group_version) or group_version != 3:\n        raise OutputBoundsError("only zarr format 3 metadata is supported")\n    if not _is_python_int(array_version) or array_version != 3:\n        raise OutputBoundsError("only zarr format 3 metadata is supported")\n    if group_meta.get("node_type") != "group":\n        raise OutputBoundsError("root metadata node_type must be \'group\'")\n    if array_meta.get("node_type") != "array":\n        raise OutputBoundsError("child metadata node_type must be \'array\'")\n\n    attributes = group_meta.get("attributes")\n    if not isinstance(attributes, dict):\n        raise OutputBoundsError("root metadata is missing attributes")\n    multiscales = attributes.get("multiscales")\n    if not isinstance(multiscales, list):\n        raise OutputBoundsError(\n            "root metadata is missing multiscales attributes"\n        )\n\n    _validate_multiscales_axes(multiscales, dataset)\n\n    path_zero_total = 0\n    for entry in multiscales:\n        paths = _multiscales_entry_paths(entry)\n        path_zero_total += sum(1 for path in paths if path == "0")\n    if path_zero_total == 0:\n        raise OutputBoundsError(\n            f"no multiscales entry references dataset path \'0\' for {dataset}"\n        )\n    if path_zero_total > 1:\n        raise OutputBoundsError(\n            f"ambiguous duplicate path \'0\' references for {dataset}"\n        )\n\n    matching = [\n        entry\n        for entry in multiscales\n        if any(path == "0" for path in _multiscales_entry_paths(entry))\n    ]\n    if len(matching) != 1:\n        raise OutputBoundsError(\n            "expected exactly one multiscales entry referencing path \'0\' "\n            f"for {dataset}"\n        )\n\n    entry = matching[0]\n    if not _entry_axes_are_tzyx(entry):\n        raise OutputBoundsError(\n            "multiscales entry for path \'0\' must declare T,Z,Y,X axes "\n            f"for {dataset}"\n        )\n\n    return _positive_rank4_shape(array_meta.get("shape"))\n\n\ndef new_output_bounds_report(\n    dataset: str, shape: tuple[int, int, int, int]\n) -> dict:\n    """Build a fresh, JSON-serializable, mutable bounds-correction report."""\n    if not isinstance(dataset, str) or not dataset.strip():\n        raise OutputBoundsError("dataset must be a non-empty string")\n    if type(dataset) is not str:\n        raise OutputBoundsError("dataset must be a builtin str")\n    t_dim, z_dim, y_dim, x_dim = _validate_rank4_shape_input(shape)\n    return {\n        "dataset": dataset,\n        "shape": [t_dim, z_dim, y_dim, x_dim],\n        "node_count": 0,\n        "corrected_nodes": 0,\n        "axis_counts": {"z": 0, "y": 0, "x": 0},\n        "max_absolute_correction": 0,\n        "samples": [],\n        "sample_limit": SAMPLE_LIMIT,\n        "samples_truncated": False,\n    }\n\n\ndef _validate_sample(\n    sample: object,\n    index: int,\n    dataset: str,\n    shape: tuple[int, int, int, int],\n) -> None:\n    """Validate a single sample entry against the schema and dataset."""\n    if not isinstance(sample, dict):\n        raise OutputBoundsError(f"report samples[{index}] must be a dict")\n    if set(sample) != set(CSV_SAMPLE_FIELDS):\n        raise OutputBoundsError(\n            f"report samples[{index}] fields do not match the sample schema"\n        )\n    if type(sample["dataset"]) is not str or sample["dataset"] != dataset:\n        raise OutputBoundsError(f"report samples[{index}] dataset mismatch")\n    if type(sample["axis"]) is not str or sample["axis"] not in COORD_AXES:\n        raise OutputBoundsError(f"report samples[{index}] axis is invalid")\n    for key in (\n        "row_id",\n        "node_id",\n        "t",\n        "rounded_int",\n        "clipped_int",\n        "signed_delta",\n        "absolute_delta",\n    ):\n        if not _is_python_int(sample[key]):\n            raise OutputBoundsError(\n                f"report samples[{index}] {key} must be a Python int"\n            )\n    if sample["row_id"] < 0 or sample["row_id"] > INT64_MAX:\n        raise OutputBoundsError(\n            f"report samples[{index}] row_id exceeds int64 range"\n        )\n    if sample["node_id"] < 0 or sample["node_id"] > INT64_MAX:\n        raise OutputBoundsError(\n            f"report samples[{index}] node_id exceeds int64 range"\n        )\n    if sample["t"] < 0 or sample["t"] > INT64_MAX:\n        raise OutputBoundsError(\n            f"report samples[{index}] t exceeds int64 range"\n        )\n    if sample["t"] >= shape[0]:\n        raise OutputBoundsError(\n            f"report samples[{index}] t exceeds T dimension"\n        )\n    original = sample["original_float"]\n    if isinstance(original, bool) or type(original) is not float:\n        raise OutputBoundsError(\n            f"report samples[{index}] original_float must be a Python float"\n        )\n    if original != original or original in (float("inf"), float("-inf")):\n        raise OutputBoundsError(\n            f"report samples[{index}] original_float must be finite"\n        )\n    rounded = sample["rounded_int"]\n    clipped = sample["clipped_int"]\n    signed = sample["signed_delta"]\n    absolute = sample["absolute_delta"]\n    if round(original) != rounded:\n        raise OutputBoundsError(\n            f"report samples[{index}] rounded_int inconsistent with original"\n        )\n    dim_size = shape[DIM_INDEX[sample["axis"]]]\n    expected_clipped = min(max(rounded, 0), dim_size - 1)\n    if clipped != expected_clipped:\n        raise OutputBoundsError(\n            f"report samples[{index}] clipped_int inconsistent with shape"\n        )\n    if signed != clipped - rounded:\n        raise OutputBoundsError(\n            f"report samples[{index}] signed_delta is inconsistent"\n        )\n    if absolute != abs(signed):\n        raise OutputBoundsError(\n            f"report samples[{index}] absolute_delta is inconsistent"\n        )\n    if absolute == 0:\n        raise OutputBoundsError(\n            f"report samples[{index}] records a non-correction"\n        )\n    if rounded == clipped:\n        raise OutputBoundsError(\n            f"report samples[{index}] rounded equals clipped"\n        )\n\n\ndef _validate_report(\n    report: object, dataset: str, shape: tuple[int, int, int, int]\n) -> None:\n    """Fail closed on any malformed report without mutating it."""\n    if not isinstance(report, dict):\n        raise OutputBoundsError("report must be a dict")\n    if set(report) != set(REPORT_FIELDS):\n        raise OutputBoundsError(\n            "report fields do not match the report schema"\n        )\n    if type(report["dataset"]) is not str or report["dataset"] != dataset:\n        raise OutputBoundsError("report dataset identity mismatch")\n    report_shape = report["shape"]\n    if not isinstance(report_shape, list):\n        raise OutputBoundsError("report shape must be a JSON list")\n    if _positive_rank4_shape(report_shape) != tuple(shape):\n        raise OutputBoundsError("report shape mismatch")\n    for key in ("node_count", "corrected_nodes", "max_absolute_correction"):\n        value = report[key]\n        if not _is_python_int(value) or value < 0:\n            raise OutputBoundsError(f"report {key} is invalid")\n    counts = report["axis_counts"]\n    if not isinstance(counts, dict) or set(counts) != set(COORD_AXES):\n        raise OutputBoundsError(\n            "report axis_counts must cover z, y and x"\n        )\n    for axis in COORD_AXES:\n        if not _is_python_int(counts[axis]) or counts[axis] < 0:\n            raise OutputBoundsError(\n                f"report axis_counts[{axis!r}] is invalid"\n            )\n    if (\n        not _is_python_int(report["sample_limit"])\n        or report["sample_limit"] != SAMPLE_LIMIT\n    ):\n        raise OutputBoundsError("report sample_limit is invalid")\n    if not isinstance(report["samples_truncated"], bool):\n        raise OutputBoundsError(\n            "report samples_truncated must be a bool"\n        )\n    samples = report["samples"]\n    if not isinstance(samples, list) or len(samples) > SAMPLE_LIMIT:\n        raise OutputBoundsError(\n            "report samples must be a list bounded by the sample limit"\n        )\n    for index, sample in enumerate(samples):\n        _validate_sample(sample, index, dataset, shape)\n    corrected = report["corrected_nodes"]\n    node_count = report["node_count"]\n    axis_total = sum(counts[axis] for axis in COORD_AXES)\n    if corrected > node_count:\n        raise OutputBoundsError(\n            "report corrected_nodes exceeds node_count"\n        )\n    for axis in COORD_AXES:\n        if counts[axis] > corrected:\n            raise OutputBoundsError(\n                f"report axis_counts[{axis!r}] exceeds corrected_nodes"\n            )\n    if axis_total < corrected:\n        raise OutputBoundsError(\n            "report axis_counts are inconsistent with corrected_nodes"\n        )\n    if axis_total > 3 * corrected:\n        raise OutputBoundsError(\n            "report axis_counts exceed per-call maximum"\n        )\n    if corrected == 0:\n        if axis_total != 0:\n            raise OutputBoundsError(\n                "report axis_counts must be zero when no corrections"\n            )\n        if report["max_absolute_correction"] != 0:\n            raise OutputBoundsError(\n                "report max must be zero when no corrections"\n            )\n        if samples:\n            raise OutputBoundsError(\n                "report samples must be empty when no corrections"\n            )\n        if report["samples_truncated"]:\n            raise OutputBoundsError(\n                "report cannot be truncated without corrections"\n            )\n    visible_axis_counts = {"z": 0, "y": 0, "x": 0}\n    visible_max = 0\n    visible_identities: set[tuple[int, int, int]] = set()\n    for sample in samples:\n        visible_axis_counts[sample["axis"]] += 1\n        visible_max = max(visible_max, sample["absolute_delta"])\n        visible_identities.add(\n            (sample["row_id"], sample["node_id"], sample["t"])\n        )\n    expected_len = min(axis_total, SAMPLE_LIMIT)\n    if len(samples) != expected_len:\n        raise OutputBoundsError(\n            "report samples length inconsistent with axis totals"\n        )\n    expected_truncated = axis_total > SAMPLE_LIMIT\n    if report["samples_truncated"] != expected_truncated:\n        raise OutputBoundsError(\n            "report samples_truncated flag is inconsistent"\n        )\n    for axis in COORD_AXES:\n        if visible_axis_counts[axis] > counts[axis]:\n            raise OutputBoundsError(\n                f"report visible {axis} samples exceed axis_counts"\n            )\n    if report["max_absolute_correction"] < visible_max:\n        raise OutputBoundsError(\n            "report max_absolute_correction is below its samples"\n        )\n    if not report["samples_truncated"]:\n        if report["max_absolute_correction"] != visible_max:\n            raise OutputBoundsError(\n                "untruncated report max must equal visible sample max"\n            )\n    if len(visible_identities) > corrected:\n        raise OutputBoundsError(\n            "report distinct sample identities exceed corrected_nodes"\n        )\n\n\ndef _as_exact_int(value: object, label: str) -> int:\n    """Return an exact Python int without ever routing Reals through float."""\n    if isinstance(value, bool):\n        raise OutputBoundsError(f"{label} must not be a bool")\n    if isinstance(value, Fraction):\n        if value.denominator != 1:\n            raise OutputBoundsError(f"{label} must be exactly integral")\n        number = int(value.numerator)\n    elif isinstance(value, Integral):\n        number = int(value)\n    elif isinstance(value, Real):\n        try:\n            candidate = round(value)\n        except (OverflowError, ValueError) as exc:\n            raise OutputBoundsError(\n                f"{label} is not a finite real number"\n            ) from exc\n        if value != candidate:\n            raise OutputBoundsError(f"{label} must be exactly integral")\n        number = int(candidate)\n    else:\n        raise OutputBoundsError(\n            f"{label} must be a real number, not {type(value).__name__}"\n        )\n    if number < INT64_MIN or number > INT64_MAX:\n        raise OutputBoundsError(f"{label} exceeds signed 64-bit limits")\n    return number\n\n\ndef _coord_to_float(value: object, axis: str) -> float:\n    """Convert a coordinate to a finite float, rejecting bool/str/non-finite."""\n    if isinstance(value, bool):\n        raise OutputBoundsError(f"coordinate {axis} must not be a bool")\n    if isinstance(value, str):\n        raise OutputBoundsError(f"coordinate {axis} must not be a string")\n    if isinstance(value, Fraction):\n        try:\n            number = float(value)\n        except (OverflowError, ValueError) as exc:\n            raise OutputBoundsError(\n                f"coordinate {axis} is not a finite real number"\n            ) from exc\n    elif isinstance(value, Real):\n        try:\n            number = float(value)\n        except (OverflowError, ValueError) as exc:\n            raise OutputBoundsError(\n                f"coordinate {axis} is not a finite real number"\n            ) from exc\n    else:\n        raise OutputBoundsError(\n            f"coordinate {axis} must be a real number"\n        )\n    if number != number or number in (float("inf"), float("-inf")):\n        raise OutputBoundsError(f"coordinate {axis} must be finite")\n    return number\n\n\ndef bounded_output_node(\n    node: dict,\n    dataset: str,\n    row_id: int,\n    shape: tuple[int, int, int, int],\n    report: dict,\n) -> dict:\n    """Return one complete, bounds-corrected node CSV row dict.\n\n    Neither ``node`` nor ``shape`` is mutated. The report is updated only after\n    the whole node validates successfully. A correction is recorded only when\n    the rounded integer differs from the clipped integer.\n    """\n    if not isinstance(node, dict):\n        raise OutputBoundsError("node must be a dict")\n    if not isinstance(dataset, str) or not dataset.strip():\n        raise OutputBoundsError("dataset must be a non-empty string")\n    if type(dataset) is not str:\n        raise OutputBoundsError("dataset must be a builtin str")\n    t_dim, z_dim, y_dim, x_dim = _validate_rank4_shape_input(shape)\n    _validate_report(report, dataset, (t_dim, z_dim, y_dim, x_dim))\n\n    row = _as_exact_int(row_id, "row_id")\n    if row < 0:\n        raise OutputBoundsError("row_id must be non-negative")\n    node_id = _as_exact_int(node.get("node_id"), "node_id")\n    if node_id < 0:\n        raise OutputBoundsError("node_id must be non-negative")\n    t_val = _as_exact_int(node.get("t"), "t")\n    if t_val < 0 or t_val > INT64_MAX:\n        raise OutputBoundsError("t exceeds signed 64-bit limits")\n    if t_val >= t_dim:\n        raise OutputBoundsError(\n            f"t={t_val} outside [0, T-1] for T={t_dim}"\n        )\n\n    dims = {"z": z_dim, "y": y_dim, "x": x_dim}\n    originals: dict[str, float] = {}\n    rounded: dict[str, int] = {}\n    clipped: dict[str, int] = {}\n    corrections: list[str] = []\n    for axis in COORD_AXES:\n        original = _coord_to_float(node.get(axis), axis)\n        near = int(round(original))\n        bound = min(max(near, 0), dims[axis] - 1)\n        originals[axis] = original\n        rounded[axis] = near\n        clipped[axis] = bound\n        if near != bound:\n            corrections.append(axis)\n\n    out = {\n        "id": row,\n        "dataset": dataset,\n        "row_type": "node",\n        "node_id": node_id,\n        "t": t_val,\n        "z": clipped["z"],\n        "y": clipped["y"],\n        "x": clipped["x"],\n        "source_id": -1,\n        "target_id": -1,\n    }\n\n    pending_samples: list[dict] = []\n    max_delta = 0\n    for axis in COORD_AXES:\n        if axis not in corrections:\n            continue\n        signed = clipped[axis] - rounded[axis]\n        absolute = abs(signed)\n        max_delta = max(max_delta, absolute)\n        pending_samples.append(\n            {\n                "dataset": dataset,\n                "row_id": row,\n                "node_id": node_id,\n                "t": t_val,\n                "axis": axis,\n                "original_float": originals[axis],\n                "rounded_int": rounded[axis],\n                "clipped_int": clipped[axis],\n                "signed_delta": signed,\n                "absolute_delta": absolute,\n            }\n        )\n\n    samples = report["samples"]\n    room = SAMPLE_LIMIT - len(samples)\n    if len(pending_samples) > room:\n        report["samples_truncated"] = True\n    if room > 0:\n        samples.extend(pending_samples[:room])\n    report["node_count"] += 1\n    if corrections:\n        report["corrected_nodes"] += 1\n        for axis in corrections:\n            report["axis_counts"][axis] += 1\n        if max_delta > report["max_absolute_correction"]:\n            report["max_absolute_correction"] = max_delta\n    return out\n', 'src/biohub/screen_output_bounds.py': '"""Explicit screen-only CSV bounds adapter and complete correction audit.\n\nNo graph mutation or dataset-specific correction policy. The submitted helper is\nreused unchanged; its sampled report is supplemented by every corrected node.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport json\nfrom copy import deepcopy\nfrom pathlib import Path\n\nfrom biohub.output_bounds import (\n    OutputBoundsError,\n    bounded_output_node,\n    new_output_bounds_report,\n)\n\nBOUNDS_SCHEMA = "biohub.e26_screen.output_bounds.v2"\nBOUNDS_POLICY = "python_round_then_xyz_clamp_to_registered_image_shape"\n_AXES = ("z", "y", "x")\n\n\ndef _canonical(value: object) -> str:\n    return json.dumps(value, sort_keys=True, allow_nan=False, separators=(",", ":"))\n\n\nclass ScreenNodeSerializer:\n    """Callable for the existing sole CSV writer; snapshots never alias state."""\n\n    def __init__(self, shapes: dict[str, tuple[int, int, int, int]]) -> None:\n        self.shapes = {dataset: tuple(shape) for dataset, shape in shapes.items()}\n        self.reports = {dataset: new_output_bounds_report(dataset, shape)\n                        for dataset, shape in self.shapes.items()}\n        self.corrections: list[dict] = []\n\n    def __call__(self, node: dict, dataset: str, row_id: int) -> dict:\n        if dataset not in self.shapes:\n            raise OutputBoundsError("node dataset missing from registered image shapes")\n        report = self.reports[dataset]\n        before = report["corrected_nodes"]\n        row = bounded_output_node(node, dataset, row_id, self.shapes[dataset], report)\n        if report["corrected_nodes"] != before:\n            self.corrections.append({\n                "dataset": dataset, "row_id": row_id,\n                "node": {"node_id": row["node_id"], "t": row["t"],\n                         **{axis: float(node[axis]) for axis in _AXES}},\n                "legacy_csv_delta": {axis: row[axis] - max(0, round(float(node[axis]))) for axis in _AXES},\n            })\n        return row\n\n    def snapshot(self) -> dict:\n        return deepcopy({"schema_version": BOUNDS_SCHEMA, "policy": BOUNDS_POLICY,\n                         "per_dataset": list(self.reports.values()), "corrections": self.corrections})\n\n\ndef verify_screen_output_bounds(\n    payload: dict, csv_path: Path, *, shapes: dict[str, tuple[int, int, int, int]],\n) -> None:\n    """Replay every correction; compare aggregates and each actual CSV node row.\n\n    The independent full CSV validator remains mandatory. This verifies captured\n    corrections, not unrecorded pre-serialization floats or graph provenance.\n    """\n    if (type(payload) is not dict\n            or set(payload) != {"schema_version", "policy", "per_dataset", "corrections"}\n            or payload["schema_version"] != BOUNDS_SCHEMA or payload["policy"] != BOUNDS_POLICY\n            or type(payload["corrections"]) is not list or type(payload["per_dataset"]) is not list):\n        raise OutputBoundsError("invalid screen bounds schema/policy")\n    replay = ScreenNodeSerializer(shapes)\n    expected_rows = {}\n    previous = -1\n    for correction in payload["corrections"]:\n        if (type(correction) is not dict or set(correction) != {"dataset", "row_id", "node", "legacy_csv_delta"}\n                or type(correction["row_id"]) is not int or correction["row_id"] <= previous\n                or type(correction["dataset"]) is not str\n                or type(correction["node"]) is not dict\n                or set(correction["node"]) != {"node_id", "t", *_AXES}):\n            raise OutputBoundsError("invalid or reordered correction audit")\n        # Captured node representation is closed and JSON-type exact.\n        node = correction["node"]\n        if (any(type(node[key]) is not int for key in ("node_id", "t"))\n                or any(type(node[axis]) is not float for axis in _AXES)):\n            raise OutputBoundsError("invalid correction node types")\n        previous = correction["row_id"]\n        before = len(replay.corrections)\n        row = replay(node, correction["dataset"], previous)\n        if len(replay.corrections) != before + 1 or _canonical(replay.corrections[-1]) != _canonical(correction):\n            raise OutputBoundsError("correction audit differs from recomputed bounds/legacy delta")\n        expected_rows[previous] = {key: str(value) for key, value in row.items()}\n\n    counts = dict.fromkeys(shapes, 0)\n    found = set()\n    with csv_path.open(newline="") as handle:\n        for row in csv.DictReader(handle):\n            if row["dataset"] not in counts:\n                raise OutputBoundsError("CSV dataset missing from bounds registration")\n            if row["row_type"] == "node":\n                counts[row["dataset"]] += 1\n            row_id = int(row["id"])\n            if row_id in expected_rows:\n                if row_id in found or row != expected_rows[row_id]:\n                    raise OutputBoundsError("correction audit disagrees with actual CSV row")\n                found.add(row_id)\n    if found != set(expected_rows):\n        raise OutputBoundsError("correction row missing from actual CSV")\n    for dataset, report in replay.reports.items():\n        if report["node_count"] > counts[dataset]:\n            raise OutputBoundsError("more corrected nodes than actual CSV nodes")\n        report["node_count"] = counts[dataset]\n    if _canonical(payload) != _canonical(replay.snapshot()):\n        raise OutputBoundsError("saved bounds report differs from complete audit/CSV counts")\n', 'src/biohub/e31_shards.py': 'from __future__ import annotations\n\nimport csv\nimport hashlib\nimport heapq\nimport json\nimport math\nimport os\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom contextlib import ExitStack\nfrom pathlib import Path\n\nfrom biohub.screen_output_bounds import ScreenNodeSerializer, verify_screen_output_bounds\n\nCSV_COLS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\n\n\ndef _stream_sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        while True:\n            chunk = f.read(65536)\n            if not chunk:\n                break\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _load_child_receipt(child_dir: Path) -> dict:\n    p = child_dir / "e31_submission_receipt.json"\n    if not p.exists():\n        raise FileNotFoundError(f"Missing receipt: {p}")\n    with p.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef _load_child_bounds(child_dir: Path) -> dict:\n    p = child_dir / "e31_output_bounds.json"\n    if not p.exists():\n        raise FileNotFoundError(f"Missing bounds: {p}")\n    with p.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef _validated_child(child_dir: Path, assigned: list[str], expected_hashes: dict) -> tuple[dict, dict]:\n    receipt = _load_child_receipt(child_dir)\n    if receipt["stems"] != assigned:\n        raise ValueError("receipt stems mismatch")\n    if set(receipt["shapes"]) != set(assigned):\n        raise ValueError("receipt shapes mismatch")\n    if receipt["E31_HASHES"] != expected_hashes:\n        raise ValueError("E31 hashes mismatch")\n    csv_path = child_dir / "submission.csv"\n    if receipt["csv_sha256"] != _stream_sha256(csv_path):\n        raise ValueError("csv sha256 mismatch")\n    if receipt.get("hypothesis") != "primary-only reciprocal consensus":\n        raise ValueError("hypothesis mismatch")\n    elapsed = receipt.get("elapsed")\n    if isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)):\n        raise ValueError("elapsed type invalid")\n    if not math.isfinite(elapsed) or elapsed < 0:\n        raise ValueError("elapsed non-finite or negative")\n    bounds = _load_child_bounds(child_dir)\n    verify_screen_output_bounds(bounds, csv_path, shapes=receipt["shapes"])\n    seen: set[str] = set()\n    prev_ds: str | None = None\n    with csv_path.open(newline="") as fh:\n        reader = csv.DictReader(fh)\n        if reader.fieldnames != CSV_COLS:\n            raise ValueError("CSV header mismatch")\n        for index, row in enumerate(reader):\n            if len(row) != len(CSV_COLS):\n                raise ValueError(f"row {index} column count mismatch")\n            if any(k is None or v is None for k, v in row.items()):\n                raise ValueError(f"row {index} has None key/value")\n            if row["id"] != str(index):\n                raise ValueError(f"row {index} id mismatch")\n            ds = row["dataset"]\n            if ds not in assigned:\n                raise ValueError(f"row {index} dataset not in assigned")\n            if prev_ds is not None and ds < prev_ds:\n                raise ValueError(f"row {index} dataset not nondecreasing")\n            prev_ds = ds\n            seen.add(ds)\n    if seen != set(assigned):\n        raise ValueError("seen datasets != assigned")\n    return receipt, bounds\n\n\ndef _merge_csv(working_dir: Path, child_dirs: list[Path], receipts: list[dict], bounds: list[dict]) -> dict:\n    global_shapes = {}\n    for r in receipts:\n        for ds, shape in r["shapes"].items():\n            global_shapes.setdefault(ds, shape)\n    global_shapes = {k: global_shapes[k] for k in sorted(global_shapes)}\n    serializer = ScreenNodeSerializer(global_shapes)\n    correction_map: dict[tuple[int, str], dict] = {}\n    for idx, b in enumerate(bounds):\n        for c in b.get("corrections", []):\n            key = (idx, str(c["row_id"]))\n            if key in correction_map:\n                raise ValueError(f"Duplicate correction key {key}")\n            correction_map[key] = c\n    node_counts: dict[str, int] = {ds: 0 for ds in global_shapes}\n    consumed: set[tuple[int, str]] = set()\n    with ExitStack() as stack:\n        readers = []\n        for idx, cd in enumerate(child_dirs):\n            fh = stack.enter_context(open(cd / "submission.csv", newline="", encoding="utf-8"))\n            readers.append((csv.DictReader(fh), idx))\n        dst = stack.enter_context(open(working_dir / "submission.csv", "x", newline="", encoding="utf-8"))\n        writer = csv.DictWriter(dst, fieldnames=CSV_COLS)\n        writer.writeheader()\n\n        def tagged(reader, idx):\n            for row in reader:\n                yield (idx, row)\n\n        merged = heapq.merge(\n            *(tagged(r, i) for r, i in readers),\n            key=lambda item: item[1]["dataset"],\n        )\n        for new_id, (idx, row) in enumerate(merged):\n            row_out = dict(row)\n            row_out["id"] = str(new_id)\n            ckey = (idx, str(row["id"]))\n            if ckey in correction_map:\n                corr = correction_map[ckey]\n                serialized = serializer(corr["node"], row["dataset"], new_id)\n                expected = {k: str(v) for k, v in serialized.items()}\n                if expected != {k: str(v) for k, v in row_out.items()}:\n                    raise ValueError(f"Correction mismatch at child={idx} row_id={row[\'id\']}")\n                consumed.add(ckey)\n            if row_out.get("row_type") == "node":\n                node_counts[row["dataset"]] += 1\n            writer.writerow(row_out)\n    unconsumed = set(correction_map.keys()) - consumed\n    if unconsumed:\n        raise ValueError(f"Unconsumed corrections: {unconsumed}")\n    for ds in global_shapes:\n        serializer.reports[ds]["node_count"] = node_counts[ds]\n    snapshot = serializer.snapshot()\n    verify_screen_output_bounds(snapshot, working_dir / "submission.csv", shapes=global_shapes)\n    return snapshot\n\n\ndef merge_shards(working_dir: Path, assignments: list[list[str]], expected_hashes: dict) -> dict:\n    if type(assignments) is not list or len(assignments) not in (1, 2):\n        raise ValueError("assignments must be a list of length 1 or 2")\n    all_assigned: list[str] = []\n    child_dirs: list[Path] = []\n    for i, assigned in enumerate(assignments):\n        if type(assigned) is not list or not assigned:\n            raise ValueError(f"assignment[{i}] must be a non-empty list of str")\n        if any(type(a) is not str for a in assigned):\n            raise TypeError(f"assignment[{i}] contains non-str elements")\n        if assigned != sorted(set(assigned)):\n            raise ValueError(f"assignment[{i}] must be sorted and unique")\n        all_assigned.extend(assigned)\n        child_dirs.append(working_dir / f"e31_shard_{i}")\n    if len(set(all_assigned)) != len(all_assigned):\n        raise ValueError("duplicate datasets across assignments")\n\n    validated = [_validated_child(cd, asgn, expected_hashes) for cd, asgn in zip(child_dirs, assignments, strict=False)]\n    receipts, bounds_list = zip(*validated, strict=False)\n\n    ref_keys = ("insertion_receipt", "deepcenter_receipt", "hypothesis")\n    ref = receipts[0]\n    for r in receipts[1:]:\n        for k in ref_keys:\n            if k not in ref or k not in r:\n                raise KeyError(f"missing key \'{k}\' in child receipt")\n            if ref[k] != r[k]:\n                raise ValueError(f"mismatch on \'{k}\' between shards")\n\n    final_files = ["submission.csv", "run_stats.csv", "e31_output_bounds.json", "e31_submission_receipt.json"]\n    for fn in final_files:\n        if (working_dir / fn).exists():\n            raise FileExistsError(fn)\n\n    stats_rows: list[dict] = []\n    headers: list[str] | None = None\n    for cd, asgn in zip(child_dirs, assignments, strict=False):\n        p = cd / "run_stats.csv"\n        with open(p, newline="", encoding="utf-8") as f:\n            reader = csv.DictReader(f)\n            hdr = reader.fieldnames or []\n            if "dataset" not in hdr:\n                raise ValueError("run_stats.csv missing \'dataset\' column")\n            if headers is None:\n                headers = list(hdr)\n            elif list(hdr) != headers:\n                raise ValueError("run_stats.csv header mismatch between shards")\n            rows = list(reader)\n            if any(None in row.keys() or None in row.values() for row in rows):\n                raise ValueError("None key/value in run_stats.csv")\n            ds_vals = [row["dataset"] for row in rows]\n            if len(ds_vals) != len(asgn) or set(ds_vals) != set(asgn):\n                raise ValueError("dataset rows do not match assigned list")\n            if len(set(ds_vals)) != len(ds_vals):\n                raise ValueError("duplicate dataset in run_stats.csv")\n            stats_rows.extend(rows)\n    stats_rows.sort(key=lambda r: r["dataset"])\n\n    snapshot = _merge_csv(working_dir, child_dirs, list(receipts), list(bounds_list))\n\n    for cd, receipt in zip(child_dirs, receipts, strict=False):\n        actual_sha = _stream_sha256(cd / "submission.csv")\n        if actual_sha != receipt["csv_sha256"]:\n            raise ValueError("child CSV changed after validation")\n\n    merged_sha = _stream_sha256(working_dir / "submission.csv")\n    all_stems = sorted(all_assigned)\n    all_shapes = {ds: r["shapes"][ds] for r in receipts for ds in sorted(r["shapes"])}\n    all_shapes = dict(sorted(all_shapes.items()))\n\n    with open(working_dir / "e31_output_bounds.json", "x", encoding="utf-8") as f:\n        json.dump(snapshot, f)\n\n    with open(working_dir / "run_stats.csv", "x", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=headers)\n        writer.writeheader()\n        writer.writerows(stats_rows)\n\n    max_elapsed = max(r["elapsed"] for r in receipts)\n    parent_receipt = {\n        "csv_sha256": merged_sha,\n        "stems": all_stems,\n        "shapes": all_shapes,\n        "E31_HASHES": expected_hashes,\n        "insertion_receipt": ref["insertion_receipt"],\n        "deepcenter_receipt": ref["deepcenter_receipt"],\n        "hypothesis": ref["hypothesis"],\n        "elapsed": max_elapsed,\n        "elapsed_kind": "max_shard_predict_seconds",\n        "shards": list(receipts),\n    }\n    with open(working_dir / "e31_submission_receipt.json", "x", encoding="utf-8") as f:\n        json.dump(parent_receipt, f)\n    return parent_receipt\n\n\nEXACT7 = (\n    "WORKING_DIR",\n    "REPO_DIR",\n    "TEST_DIR",\n    "_ps",\n    "_primary_materialized_path",\n    "_deepcenter_materialized_path",\n    "SECONDARY_WEIGHTS_PATH",\n)\n\nCHILD_CODE = """\\\nimport sys, json, runpy\nfrom pathlib import Path\n_PK = ["WORKING_DIR","REPO_DIR","TEST_DIR","_ps",\n       "_primary_materialized_path","_deepcenter_materialized_path","SECONDARY_WEIGHTS_PATH"]\nwith open(sys.argv[1]) as _f:\n    _job = json.load(_f)\nfor _k in _PK:\n    if _k in _job:\n        _job[_k] = Path(_job[_k])\nrunpy.run_path(sys.argv[2], init_globals=_job)\n"""\n\n\ndef run_workers(\n    job: dict,\n    payloadroot: Path,\n    cuda_tokens: list[str],\n    timeout_seconds: float = 39600.0,\n) -> dict:\n    if (\n        len(cuda_tokens) < 2\n        or not isinstance(cuda_tokens[0], str)\n        or not isinstance(cuda_tokens[1], str)\n        or cuda_tokens[0] == ""\n        or cuda_tokens[1] == ""\n        or cuda_tokens[0] == cuda_tokens[1]\n    ):\n        raise ValueError("cuda_tokens must have >=2 distinct nonempty strings")\n    if not (math.isfinite(timeout_seconds) and timeout_seconds > 0):\n        raise ValueError("timeout_seconds must be finite positive")\n\n    test_dir: Path = job["TEST_DIR"]\n    if list(test_dir.glob("*.geff")):\n        raise ValueError("GT geff forbidden in test_dir")\n    stems = sorted(p.stem for p in test_dir.glob("*.zarr"))\n    if not stems:\n        raise ValueError("no zarr stems found")\n    worker_count = min(2, len(stems))\n    assignments = [stems[i::worker_count] for i in range(worker_count)]\n\n    runtime = payloadroot / "scripts" / "e31_submission_runtime.py"\n    hashes: dict = job["E31_HASHES"]\n    if "scripts/e31_submission_runtime.py" not in hashes:\n        raise ValueError("runtime path missing from E31_HASHES")\n    for rel, expected in hashes.items():\n        if os.path.isabs(rel) or ".." in rel.split("/"):\n            raise ValueError(f"unsafe hash key: {rel}")\n        fp = payloadroot / rel\n        if not fp.exists():\n            raise FileNotFoundError(fp)\n        if _stream_sha256(fp) != expected:\n            raise ValueError(f"hash mismatch: {rel}")\n\n    parent_work: Path = job["WORKING_DIR"]\n    repo_dir: Path = job["REPO_DIR"]\n    child_dirs = []\n    for i in range(worker_count):\n        cd = parent_work / f"e31_shard_{i}"\n        cd.mkdir(exist_ok=False)\n        child_dirs.append(cd)\n\n    procs: list[subprocess.Popen] = []\n    logfiles = []\n    t0 = time.monotonic()\n    with ExitStack() as stack:\n        try:\n            for i in range(worker_count):\n                cjob = {}\n                for k, v in job.items():\n                    cjob[k] = str(v) if k in EXACT7 else v\n                cjob["WORKING_DIR"] = str(child_dirs[i])\n                cjob["E31_ASSIGNED_STEMS"] = assignments[i]\n                cjob["E31_WORKER_TAG"] = str(i)\n                cjob["E31_HASHES"] = hashes\n                jpath = child_dirs[i] / "job.json"\n                with open(jpath, "x", encoding="utf-8") as jf:\n                    json.dump(cjob, jf)\n\n                lf = stack.enter_context(open(child_dirs[i] / "child.log", "x", encoding="utf-8"))\n                logfiles.append(jpath.parent / "child.log")\n                env = os.environ.copy()\n                env["PYTHONHASHSEED"] = "0"\n                env["CUDA_VISIBLE_DEVICES"] = cuda_tokens[i]\n                env["BIOHUB_GPU_SHARD"] = str(i)\n                env["PYTHONPATH"] = os.pathsep.join(\n                    [\n                        str(payloadroot / "src"),\n                        str(repo_dir / "src"),\n                        str(repo_dir / "scripts"),\n                        str(repo_dir),\n                    ]\n                )\n                p = subprocess.Popen(\n                    [sys.executable, "-c", CHILD_CODE, str(jpath), str(runtime)],\n                    cwd=str(repo_dir),\n                    start_new_session=True,\n                    stdout=lf,\n                    stderr=subprocess.STDOUT,\n                    env=env,\n                )\n                procs.append(p)\n\n            deadline = t0 + timeout_seconds\n            while True:\n                if time.monotonic() > deadline:\n                    raise TimeoutError("workers exceeded timeout")\n                all_done = True\n                for i, p in enumerate(procs):\n                    rc = p.poll()\n                    if rc is None:\n                        all_done = False\n                    elif rc != 0:\n                        raise RuntimeError(f"worker {i} exited with returncode {rc}; log: {logfiles[i]}")\n                if all_done:\n                    break\n                time.sleep(1.0)\n\n            for i, p in enumerate(procs):\n                if p.returncode != 0:\n                    raise RuntimeError(f"child e31_shard_{i} exited {p.returncode}: {logfiles[i]}")\n        finally:\n            for p in procs:\n                if p.poll() is None:\n                    try:\n                        os.killpg(p.pid, signal.SIGTERM)\n                    except ProcessLookupError:\n                        pass\n            for p in procs:\n                try:\n                    p.wait(timeout=30)\n                except subprocess.TimeoutExpired:\n                    try:\n                        os.killpg(p.pid, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n                    p.wait(timeout=30)\n\n    for rel, expected in hashes.items():\n        fp = payloadroot / rel\n        if not fp.exists():\n            raise FileNotFoundError(fp)\n        if _stream_sha256(fp) != expected:\n            raise ValueError(f"hash mismatch: {rel}")\n    receipt = merge_shards(parent_work, assignments, job["E31_HASHES"])\n    timing = {\n        "total_wall_seconds": time.monotonic() - t0,\n        "worker_count": worker_count,\n        "assignments": assignments,\n        "cuda_tokens": cuda_tokens[:worker_count],\n    }\n    tpath = parent_work / "e31_parallel_timing.json"\n    with open(tpath, "x", encoding="utf-8") as tf:\n        json.dump(timing, tf)\n    print("VALIDATED_E31_SUBMISSION")\n    return receipt\n', 'scripts/e31_submission_runtime.py': '# e31_submission_runtime.py — notebook cell fragment (not importable module)\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom biohub.association_instrumentation import instrument_source\nfrom biohub.primary_consensus_observer import PrimaryConsensusObserver\n\nstems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))\nassert stems, "no test zarr stems"\nassert len(stems) == len(set(stems))\nassert not list(TEST_DIR.glob("*.geff"))\nif "E31_ASSIGNED_STEMS" in globals():\n    _e31_assigned = globals()["E31_ASSIGNED_STEMS"]\n    if _e31_assigned is None:\n        raise TypeError("E31_ASSIGNED_STEMS must not be None")\n    if type(_e31_assigned) is not list:\n        raise TypeError("E31_ASSIGNED_STEMS must be a list")\n    if not _e31_assigned:\n        raise ValueError("E31_ASSIGNED_STEMS must be nonempty")\n    for _e31_entry in _e31_assigned:\n        if type(_e31_entry) is not str:\n            raise TypeError("each E31_ASSIGNED_STEMS entry must be str")\n    if _e31_assigned != sorted(set(_e31_assigned)):\n        raise ValueError("E31_ASSIGNED_STEMS must equal sorted(set(list))")\n    if not set(_e31_assigned).issubset(set(stems)):\n        raise ValueError("E31_ASSIGNED_STEMS must be subset of discovered stems")\n    stems = list(_e31_assigned)\n_e31_worker_tag = globals().get("E31_WORKER_TAG", "")\nif type(_e31_worker_tag) is not str or _e31_worker_tag not in ("", "0", "1"):\n    raise ValueError("E31_WORKER_TAG must be \'\', \'0\', or \'1\'")\n_e31_predict_name = {"": "e31_predict.py", "0": "e31_predict_0.py", "1": "e31_predict_1.py"}[_e31_worker_tag]\n\nassert float(os.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"]) == 0.30\nassert Path(os.environ["BIOHUB_SECONDARY_WEIGHTS"]) == SECONDARY_WEIGHTS_PATH\nassert SECONDARY_WEIGHTS_PATH.is_file()\nsecondary_edge_weight = float(os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])\nassert secondary_edge_weight == 0.15\nfusion_mode = os.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"]\nassert fusion_mode == "harmonic_probability"\n\npatched, insertion_receipt = instrument_source(_ps.read_text())\npredict_path = REPO_DIR / "scripts" / _e31_predict_name\nwith open(predict_path, "x", encoding="utf-8") as f:\n    f.write(patched)\n\nimport dataspec\n\ndataspec.PREDICTIONS_PATH = WORKING_DIR / "e31_predictions"\nassert not dataspec.PREDICTIONS_PATH.exists()\n\nspec = importlib.util.spec_from_file_location("e31_predict", predict_path)\nmodule = importlib.util.module_from_spec(spec)\nsys.modules["e31_predict"] = module\nspec.loader.exec_module(module)\n\nobserver = PrimaryConsensusObserver()\n\nsplits_path = WORKING_DIR / "e31_splits.json"\nwith open(splits_path, "x", encoding="utf-8") as f:\n    json.dump([{"split": 0, "train": [], "test": stems}], f)\n\nfrom biohub.output_bounds import read_output_shape\n\nshapes = {ds: read_output_shape(TEST_DIR, ds) for ds in stems}\n\nt0 = time.monotonic()\nmodule.predict(\n    data_dir=TEST_DIR,\n    fold=0,\n    splits_file=splits_path,\n    weights_path=_primary_materialized_path,\n    cfg=module.PredictConfig(\n        det_threshold=0.96875,\n        use_ilp=True,\n        ilp_edge_weight=-1.0,\n        ilp_appearance_weight=0.0,\n        ilp_disappearance_weight=1.5,\n        ilp_division_weight=1.0,\n    ),\n    method="unet_transformer",\n    unet_batch_size=4,\n    evaluate=False,\n    association_observer=observer,\n)\nelapsed = time.monotonic() - t0\n\ngeffs = sorted(dataspec.PREDICTIONS_PATH.rglob("*.geff"))\nassert len(geffs) == len(stems)\nassert {g.stem for g in geffs} == set(stems)\n\nfrom biohub.public_postproc.config import build_config\nfrom biohub.public_postproc.deepcenter import (\n    load_deepcenter_veto_detector_e31_target as load_deepcenter_veto_detector_strict,\n)\nfrom biohub.public_postproc.pipeline import run_postproc_core\nfrom biohub.screen_output_bounds import ScreenNodeSerializer, verify_screen_output_bounds\n\nmanifest = _deepcenter_materialized_path.parents[2] / "ARTIFACT_MANIFEST.json"\nassert manifest.is_file()\ncfg = build_config(\n    overrides={\n        "BIOHUB_DEEPCENTER_CHECKPOINT": str(_deepcenter_materialized_path),\n        "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT": str(_deepcenter_materialized_path),\n        "BIOHUB_DEEPCENTER_MANIFEST": str(manifest),\n        "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT": str(manifest),\n    },\n    test_dir=TEST_DIR,\n    profile="e23",\n)\nassert cfg.OUTPUT_MOTION_RELINK and cfg.USE_DEEPCENTER_VETO and cfg.REQUIRE_DEEPCENTER_VETO\nbundle, deepcenter_receipt = load_deepcenter_veto_detector_strict(cfg, _deepcenter_materialized_path, manifest)\n\nserializer = ScreenNodeSerializer(shapes)\ncsv_path = WORKING_DIR / "submission.csv"\nrun_stats_path = WORKING_DIR / "run_stats.csv"\nrun_postproc_core(\n    geffs,\n    csv_path,\n    cfg,\n    run_stats_path=run_stats_path,\n    predict_seconds=elapsed,\n    node_serializer=serializer,\n    consensus_loader=observer.frames_for,\n    exclusive_output=True,\n    deepcenter_loader=lambda config: bundle,\n)\n\nbounds = serializer.snapshot()\nverify_screen_output_bounds(bounds, csv_path, shapes=shapes)\nbounds_path = WORKING_DIR / "e31_output_bounds.json"\nwith open(bounds_path, "x", encoding="utf-8") as f:\n    json.dump(bounds, f)\n\nEXPECTED_COLS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\nnode_times = {}\nrow_counter = 0\nwith open(csv_path, newline="", encoding="utf-8") as f:\n    reader = csv.DictReader(f)\n    assert reader.fieldnames == EXPECTED_COLS\n    for row in reader:\n        assert int(row["id"]) == row_counter\n        row_counter += 1\n        ds = row["dataset"]\n        rt = row["row_type"]\n        assert ds in shapes, f"unknown dataset {ds}"\n        if rt == "node":\n            nid = int(row["node_id"])\n            assert nid >= 0\n            assert nid not in node_times.get(ds, {})\n            assert int(row["source_id"]) == -1\n            assert int(row["target_id"]) == -1\n            ti = int(row["t"])\n            zi = int(row["z"])\n            yi = int(row["y"])\n            xi = int(row["x"])\n            sh = shapes[ds]\n            assert 0 <= ti < sh[0] and 0 <= zi < sh[1] and 0 <= yi < sh[2] and 0 <= xi < sh[3]\n            node_times.setdefault(ds, {})[nid] = ti\n        elif rt == "edge":\n            assert int(row["node_id"]) == -1 and int(row["t"]) == -1\n            assert int(row["z"]) == -1 and int(row["y"]) == -1 and int(row["x"]) == -1\n        else:\n            raise ValueError(f"unknown row_type {rt}")\n\nedges_seen = set()\nin_deg = {}\nout_deg = {}\nwith open(csv_path, newline="", encoding="utf-8") as f:\n    reader = csv.DictReader(f)\n    for row in reader:\n        if row["row_type"] != "edge":\n            continue\n        ds = row["dataset"]\n        src = int(row["source_id"])\n        tgt = int(row["target_id"])\n        assert src >= 0 and tgt >= 0\n        nt = node_times.get(ds, {})\n        assert src in nt and tgt in nt\n        assert nt[tgt] == nt[src] + 1\n        key = (ds, src, tgt)\n        assert key not in edges_seen\n        edges_seen.add(key)\n        out_deg[(ds, src)] = out_deg.get((ds, src), 0) + 1\n        in_deg[(ds, tgt)] = in_deg.get((ds, tgt), 0) + 1\n        assert out_deg[(ds, src)] <= 2 and in_deg[(ds, tgt)] <= 1\n\nfor ds in stems:\n    assert ds in node_times and node_times[ds]\n\nassert csv_path.stat().st_size > 0\ncsv_sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()\nreceipt = {\n    "csv_sha256": csv_sha,\n    "stems": stems,\n    "shapes": shapes,\n    "insertion_receipt": insertion_receipt,\n    "deepcenter_receipt": deepcenter_receipt,\n    "E31_HASHES": E31_HASHES,\n    "elapsed": elapsed,\n    "hypothesis": "primary-only reciprocal consensus",\n}\nwith open(WORKING_DIR / "e31_submission_receipt.json", "x", encoding="utf-8") as f:\n    json.dump(receipt, f)\nprint("VALIDATED_E31_SUBMISSION")\n', 'src/biohub/public_postproc/__init__.py': '"""Local, torch-free port of the public Kaggle notebook\'s post-processing stack.\n\nPorts (near-verbatim) the notebook cells that turn raw UNet+transformer+ILP\nprediction graphs (``*.geff``) into the competition submission CSV: motion\nrelink, gap close (+ density-adaptive threshold, synthetic-midpoint\nrefinement), single-parent repair, safe-division post-linking, short-track\nfilter/rescue, and linefit smoothing. See ``scripts/postproc_geffs.py`` for\nthe CLI entry point and ``config.py`` for the preset that reproduces\n``outputs/kaggle/base1_v1/submission.csv``.\n"""\nfrom __future__ import annotations\n', 'src/biohub/public_postproc/config.py': '"""Config for the ported public-notebook post-processing stack.\n\nThis mirrors two notebook cells almost verbatim:\n\n* the "constants" cell, which reads ``os.environ.get(NAME, default)`` into\n  typed Python constants -- captured here as :data:`CODE_DEFAULTS` (the\n  string default for every ``BIOHUB_*`` variable) plus :func:`build_config`,\n  which does the same ``os.environ.get`` parsing but against an in-memory\n  env map instead of the real process environment;\n* the "preset" cell, which unconditionally overwrites a subset of those\n  env vars for one named experiment -- captured here as :data:`PRESET`.\n\n:data:`PRESET` is also exported as :data:`BASE1_PRESET`. Named profiles\n(see :data:`PROFILES`) select which preset is layered on top of\n``CODE_DEFAULTS`` by :func:`build_config`:\n\n* ``base1`` -- ``CODE_DEFAULTS`` + ``PRESET``; the exact config that\n  produced ``outputs/kaggle/base1_v1/submission.csv`` (score 0.8890) via\n  the ``biohub_132_clean_short_track_rescue_lightcv_nohack`` preset with\n  no further ``--set``. This is the default and stays byte-for-byte\n  identical to the previous behavior.\n* ``e23`` -- ``CODE_DEFAULTS`` + :data:`E23_PRESET`; the submitted E23\n  notebook\'s effective post-processing settings (public LB 0.924), per the\n  E23 parity contract in ``analysis/gold_loop_protocol.md``. It is built\n  directly on ``CODE_DEFAULTS`` and never inherits the base1 preset.\n* ``e23_twin_only_v1`` -- ``CODE_DEFAULTS`` + :data:`TWIN_ONLY_V1_PRESET`;\n  the frozen twin-only experiment configuration layered on E23.\n\nOnly the variables that ``filter_output_graph`` and the CSV/run_stats\nwriters actually read are modelled here. Variables that only affect the\nGPU inference stage (``BIOHUB_DET_THRESHOLD``, ``BIOHUB_UNET_BATCH_SIZE``,\n``BIOHUB_ILP_*``, ...) are intentionally out of scope for local\npost-processing and are not part of this dataclass; presets that set them\nare still accepted (the keys are simply ignored) so a real preset file can\nbe pasted into ``--set`` without editing it first.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, fields\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# "constants" cell: BIOHUB_<NAME> -> hardcoded string default.\n# Verbatim from the notebook\'s config cell (values only; env var names kept\n# so presets stay copy-paste compatible).\n# ---------------------------------------------------------------------------\nCODE_DEFAULTS: dict[str, str] = {\n    "BIOHUB_OUTPUT_EDGE_MAX_UM": "14.0",\n    "BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME": "1",\n    "BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR": "1",\n    "BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR": "0",\n    "BIOHUB_OUTPUT_PRUNE_ISOLATED": "1",\n    "BIOHUB_OUTPUT_MOTION_RELINK": "1",\n    "BIOHUB_MOTION_RELINK_TIGHT_UM": "6.0",\n    "BIOHUB_MOTION_RELINK_RELAXED_UM": "10.0",\n    "BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT": "0.5",\n    "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "0.75",\n    "BIOHUB_MOTION_RELINK_MAX_FRAME_NODES": "2600",\n    "BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER": "0",\n    "BIOHUB_DIV_PARENT_MAX_UM": "10.5",\n    "BIOHUB_DIV_SISTER_MAX_UM": "8.0",\n    "BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD": "1",\n    "BIOHUB_OUTPUT_GAP_CLOSE": "1",\n    "BIOHUB_GAP_CLOSE_MAX_GAP": "1",\n    "BIOHUB_GAP_CLOSE_UM": "6.0",\n    "BIOHUB_GAP_DENSITY_ADAPTIVE": "0",\n    "BIOHUB_GAP_DENSITY_REFERENCE_UM": "6.5",\n    "BIOHUB_GAP_DENSITY_GAIN": "0.040",\n    "BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM": "0.125",\n    "BIOHUB_GAP_DENSITY_NEIGHBORS": "3",\n    "BIOHUB_GAP_CLOSE_REUSE_EXISTING": "1",\n    "BIOHUB_GAP_CLOSE_REUSE_UM": "3.2",\n    "BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC": "0.05",\n    "BIOHUB_GAP_CLOSE_MAX_ADDED_ABS": "2000",\n    "BIOHUB_GAP_REFINE_SYNTHETIC": "1",\n    "BIOHUB_GAP_REFINE_WIN_Z": "1",\n    "BIOHUB_GAP_REFINE_WIN_YX": "3",\n    "BIOHUB_GAP_REFINE_MAX_SHIFT_UM": "3.2",\n    "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "1",\n    "BIOHUB_OUTPUT_MIN_TRACK_LEN": "6",\n    "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",\n    "BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE": "0",\n    "BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC": "0.10",\n    "BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN": "4",\n    "BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB": "0.82",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM": "3.25",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC": "0.018",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS": "180",\n    "BIOHUB_OUTPUT_LINEFIT_SMOOTH": "1",\n    "BIOHUB_OUTPUT_LINEFIT_WEIGHT": "0.8",\n    "BIOHUB_OUTPUT_LINEFIT_WINDOW": "2",\n    "BIOHUB_OUTPUT_GAP2_RECOVERY": "0",\n    "BIOHUB_GAP2_MAX_TOTAL_UM": "10.2",\n    "BIOHUB_GAP2_MAX_STEP_UM": "4.4",\n    "BIOHUB_GAP2_MAX_LINKS_FRAC": "0.0045",\n    "BIOHUB_GAP2_MAX_LINKS_ABS": "180",\n    "BIOHUB_GAP2_REQUIRE_CONTEXT": "1",\n    "BIOHUB_GAP2_FRAME_FRAC_CAP": "0.006",\n    "BIOHUB_OUTPUT_SAFE_DIVISIONS": "1",\n    "BIOHUB_SAFE_DIV_MAX_UM": "4.7",\n    "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "7.2",\n    "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "7.8",\n    "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.008",\n    "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.004",\n    "BIOHUB_USE_DEEPCENTER_VETO": "1",\n    "BIOHUB_REQUIRE_DEEPCENTER_VETO": "1",\n    "BIOHUB_DEEPCENTER_GAP_VETO": "1",\n    "BIOHUB_DEEPCENTER_SAFE_DIV_VETO": "1",\n    "BIOHUB_DEEPCENTER_GAP_THRESHOLD": "0.10",\n    "BIOHUB_DEEPCENTER_EXPECTED_EPOCH": "0",\n    "BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM": "0",\n    "BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD": "0.12",\n    "BIOHUB_DEEPCENTER_SCORE_WIN_Z": "1",\n    "BIOHUB_DEEPCENTER_SCORE_WIN_YX": "2",\n    "BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES": "8",\n    "BIOHUB_DEEPCENTER_RELATIVE": "weights/full_frame_center/checkpoint_last.pt",\n    "BIOHUB_DEEPCENTER_CHECKPOINT": "",\n    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT": (\n        "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt"\n    ),\n    "BIOHUB_DEEPCENTER_MANIFEST": "",\n    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT": (\n        "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json"\n    ),\n    # Phase-1 (E23 parity) additions: not part of the original notebook\'s\n    # constants cell. Defaults keep base1 behavior: refinement off, legacy\n    # safe-division mode, structural gates off.\n    "BIOHUB_REFINE_ALL_CENTROIDS": "0",\n    "BIOHUB_REFINE_CENTROIDS_WIN_Z": "1",\n    "BIOHUB_REFINE_CENTROIDS_WIN_YX": "3",\n    "BIOHUB_REFINE_CENTROIDS_BASELINE_PERCENTILE": "20.0",\n    "BIOHUB_REFINE_CENTROIDS_MAX_SHIFT_UM": "2.8",\n    "BIOHUB_SAFE_DIV_MODE": "legacy",\n    "BIOHUB_SAFE_DIV_REQUIRE_MID_TRACK_PARENT": "0",\n    "BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN": "0",\n    "BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE": "0",\n    "BIOHUB_SAFE_DIV_DIVERGE_UM": "2.25",\n    "BIOHUB_OUTPUT_STEAL_TWIN_REWIRE": "0",\n    "BIOHUB_STEAL_TWIN_MODE": "twin_only_v1",\n    "BIOHUB_STEAL_TWIN_DRY_RUN": "0",\n    "BIOHUB_STEAL_TWIN_PARENT_MAX_UM": "8.0",\n    "BIOHUB_STEAL_TWIN_EXISTING_CHILD_MAX_UM": "10.0",\n    "BIOHUB_STEAL_TWIN_SISTER_MIN_UM": "5.5",\n    "BIOHUB_STEAL_TWIN_SISTER_MAX_UM": "11.0",\n    "BIOHUB_STEAL_TWIN_DIVERGE_UM": "2.25",\n    "BIOHUB_STEAL_TWIN_TWIN_MAX_UM": "5.0",\n    "BIOHUB_STEAL_TWIN_REQUIRE_TWO_SUCCESSORS": "1",\n    "BIOHUB_STEAL_TWIN_REJECT_SYNTHETIC": "1",\n    "BIOHUB_STEAL_TWIN_DEEPCENTER_VETO": "1",\n    "BIOHUB_STEAL_TWIN_FRAME_CAP_ABS": "1",\n    "BIOHUB_STEAL_TWIN_VIDEO_CAP_ABS": "2",\n    "BIOHUB_STEAL_TWIN_DEBUG_JSONL": "",\n    "BIOHUB_STEAL_TWIN_DEBUG_MAX_RECORDS": "200",\n    "BIOHUB_EXPERIMENT_TAG": "biohub_132_clean_short_track_rescue_lightcv_nohack",\n}\n\n# ---------------------------------------------------------------------------\n# "preset" cell: biohub_132_clean_short_track_rescue_lightcv_nohack.\n# Verbatim from the notebook preset -- values that are the same as\n# CODE_DEFAULTS are kept too, so this dict alone documents the full preset.\n# Only the keys relevant to post-processing are included (the source cell\n# also sets BIOHUB_DET_THRESHOLD, BIOHUB_ILP_*, BIOHUB_RUN_VISUAL_EDA, ...,\n# which affect the GPU/ILP stage or notebook display only).\n# ---------------------------------------------------------------------------\nPRESET: dict[str, str] = {\n    "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "1",\n    "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "1.0",\n    "BIOHUB_MOTION_RELINK_TIGHT_UM": "6.0",\n    "BIOHUB_MOTION_RELINK_RELAXED_UM": "9.5",\n    "BIOHUB_GAP_CLOSE_MAX_GAP": "2",\n    "BIOHUB_GAP_CLOSE_UM": "5.8",\n    "BIOHUB_GAP_DENSITY_ADAPTIVE": "1",\n    "BIOHUB_GAP_DENSITY_REFERENCE_UM": "6.5",\n    "BIOHUB_GAP_DENSITY_GAIN": "0.040",\n    "BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM": "0.125",\n    "BIOHUB_GAP_DENSITY_NEIGHBORS": "3",\n    "BIOHUB_OUTPUT_MIN_TRACK_LEN": "6",\n    "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",\n    "BIOHUB_OUTPUT_GAP2_RECOVERY": "0",\n    "BIOHUB_SAFE_DIV_MAX_UM": "4.66",\n    "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "8.5",\n    "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "7.65",\n    "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.0076",\n    "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.00375",\n    "BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE": "1",\n    "BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC": "0.10",\n    "BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN": "5",\n    "BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB": "0.90",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM": "2.75",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC": "0.006",\n    "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS": "60",\n    "BIOHUB_USE_DEEPCENTER_VETO": "0",\n    "BIOHUB_REQUIRE_DEEPCENTER_VETO": "0",\n    "BIOHUB_DEEPCENTER_EXPECTED_EPOCH": "0",\n    "BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM": "8.0",\n    "BIOHUB_DEEPCENTER_CHECKPOINT": "",\n    "BIOHUB_DEEPCENTER_GAP_VETO": "0",\n    "BIOHUB_DEEPCENTER_GAP_THRESHOLD": "0.20",\n    "BIOHUB_DEEPCENTER_SAFE_DIV_VETO": "0",\n    "BIOHUB_EXPERIMENT_TAG": "biohub_132_clean_short_track_rescue_lightcv_nohack",\n}\n\n# Alias so the base1 profile reads explicitly next to the other profiles.\nBASE1_PRESET: dict[str, str] = PRESET\n\n# ---------------------------------------------------------------------------\n# E23 preset: the submitted E23 notebook\'s effective post-processing\n# settings (public LB 0.924), per the E23 parity contract in\n# analysis/gold_loop_protocol.md. Layered on CODE_DEFAULTS only -- never on\n# the base1 preset -- so anything unset here falls back to the notebook\'s\n# "constants" cell values, not to base1\'s overrides. Values that equal\n# CODE_DEFAULTS are kept explicit so this dict alone documents the profile.\n# ---------------------------------------------------------------------------\nE23_PRESET: dict[str, str] = {\n    "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "1.0",\n    "BIOHUB_GAP_CLOSE_MAX_GAP": "2",\n    "BIOHUB_GAP_CLOSE_UM": "5.8",\n    "BIOHUB_GAP_DENSITY_ADAPTIVE": "1",\n    "BIOHUB_GAP_DENSITY_REFERENCE_UM": "6.5",\n    "BIOHUB_GAP_DENSITY_GAIN": "0.040",\n    "BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM": "0.125",\n    "BIOHUB_GAP_DENSITY_NEIGHBORS": "3",\n    "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "1",\n    "BIOHUB_OUTPUT_MIN_TRACK_LEN": "6",\n    "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",\n    "BIOHUB_OUTPUT_GAP2_RECOVERY": "0",\n    "BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE": "0",\n    "BIOHUB_SAFE_DIV_MAX_UM": "8.0",\n    "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "11.0",\n    "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "10.0",\n    "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.0076",\n    "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.00375",\n    "BIOHUB_SAFE_DIV_MODE": "e23",\n    "BIOHUB_SAFE_DIV_REQUIRE_MID_TRACK_PARENT": "1",\n    "BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN": "1",\n    "BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE": "1",\n    "BIOHUB_SAFE_DIV_DIVERGE_UM": "2.25",\n    "BIOHUB_REFINE_ALL_CENTROIDS": "1",\n    "BIOHUB_REFINE_CENTROIDS_WIN_Z": "1",\n    "BIOHUB_REFINE_CENTROIDS_WIN_YX": "3",\n    "BIOHUB_REFINE_CENTROIDS_BASELINE_PERCENTILE": "20.0",\n    "BIOHUB_REFINE_CENTROIDS_MAX_SHIFT_UM": "2.8",\n    "BIOHUB_USE_DEEPCENTER_VETO": "1",\n    "BIOHUB_REQUIRE_DEEPCENTER_VETO": "1",\n    "BIOHUB_DEEPCENTER_EXPECTED_EPOCH": "2",\n    "BIOHUB_DEEPCENTER_GAP_VETO": "1",\n    "BIOHUB_DEEPCENTER_GAP_THRESHOLD": "0.25",\n    "BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM": "8.5",\n    "BIOHUB_DEEPCENTER_SAFE_DIV_VETO": "1",\n    "BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD": "0.12",\n    "BIOHUB_DEEPCENTER_RELATIVE": "weights/full_frame_center/best.pt",\n    "BIOHUB_DEEPCENTER_CHECKPOINT": (\n        "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"\n    ),\n    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT": (\n        "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"\n    ),\n    "BIOHUB_EXPERIMENT_TAG": "e23_pub923_parity",\n}\n\nTWIN_ONLY_V1_PRESET: dict[str, str] = {\n    **E23_PRESET,\n    "BIOHUB_OUTPUT_STEAL_TWIN_REWIRE": "1",\n    "BIOHUB_STEAL_TWIN_MODE": "twin_only_v1",\n    "BIOHUB_STEAL_TWIN_DRY_RUN": "0",\n    "BIOHUB_STEAL_TWIN_PARENT_MAX_UM": "8.0",\n    "BIOHUB_STEAL_TWIN_EXISTING_CHILD_MAX_UM": "10.0",\n    "BIOHUB_STEAL_TWIN_SISTER_MIN_UM": "5.5",\n    "BIOHUB_STEAL_TWIN_SISTER_MAX_UM": "11.0",\n    "BIOHUB_STEAL_TWIN_DIVERGE_UM": "2.25",\n    "BIOHUB_STEAL_TWIN_TWIN_MAX_UM": "5.0",\n    "BIOHUB_STEAL_TWIN_REQUIRE_TWO_SUCCESSORS": "1",\n    "BIOHUB_STEAL_TWIN_REJECT_SYNTHETIC": "1",\n    "BIOHUB_STEAL_TWIN_DEEPCENTER_VETO": "1",\n    "BIOHUB_STEAL_TWIN_FRAME_CAP_ABS": "1",\n    "BIOHUB_STEAL_TWIN_VIDEO_CAP_ABS": "2",\n    "BIOHUB_STEAL_TWIN_DEBUG_JSONL": "",\n    "BIOHUB_STEAL_TWIN_DEBUG_MAX_RECORDS": "200",\n    "BIOHUB_EXPERIMENT_TAG": "e23_twin_only_v1",\n}\n\n# Named post-processing profiles: profile name -> preset layered on top of\n# CODE_DEFAULTS by build_config. Adding a profile never changes base1.\nPROFILES: dict[str, dict[str, str]] = {\n    "base1": BASE1_PRESET,\n    "e23": E23_PRESET,\n    "e23_twin_only_v1": TWIN_ONLY_V1_PRESET,\n}\n\n\ndef _get_bool(env: dict[str, str], name: str) -> bool:\n    return env.get(name, CODE_DEFAULTS[name]) != "0"\n\n\ndef _get_float(env: dict[str, str], name: str) -> float:\n    return float(env.get(name, CODE_DEFAULTS[name]))\n\n\ndef _get_int(env: dict[str, str], name: str) -> int:\n    return int(env.get(name, CODE_DEFAULTS[name]))\n\n\ndef _get_str(env: dict[str, str], name: str) -> str:\n    return env.get(name, CODE_DEFAULTS[name])\n\n\n@dataclass(frozen=True)\nclass PostprocConfig:\n    """Typed, parsed view of the ``BIOHUB_*`` post-processing knobs.\n\n    Field names match the notebook\'s constant names (minus the ``BIOHUB_``\n    prefix) so the port stays traceable line-for-line against the source\n    cells.\n    """\n\n    TEST_DIR: Path\n\n    OUTPUT_EDGE_MAX_UM: float\n    OUTPUT_ENFORCE_NEXT_FRAME: bool\n    OUTPUT_SINGLE_PARENT_REPAIR: bool\n    OUTPUT_SINGLE_CHILD_REPAIR: bool\n    OUTPUT_PRUNE_ISOLATED: bool\n    OUTPUT_MOTION_RELINK: bool\n    MOTION_RELINK_TIGHT_UM: float\n    MOTION_RELINK_RELAXED_UM: float\n    MOTION_RELINK_VELOCITY_WEIGHT: float\n    MOTION_RELINK_LEARNED_BONUS: float\n    MOTION_RELINK_MAX_FRAME_NODES: int\n\n    OUTPUT_DIVISION_GEOMETRY_FILTER: bool\n    DIV_PARENT_MAX_UM: float\n    DIV_SISTER_MAX_UM: float\n    DIV_DROP_TO_SINGLE_IF_BAD: bool\n\n    OUTPUT_GAP_CLOSE: bool\n    GAP_CLOSE_MAX_GAP: int\n    GAP_CLOSE_UM: float\n    GAP_DENSITY_ADAPTIVE: bool\n    GAP_DENSITY_REFERENCE_UM: float\n    GAP_DENSITY_GAIN: float\n    GAP_DENSITY_MAX_STEP_DELTA_UM: float\n    GAP_DENSITY_NEIGHBORS: int\n    GAP_CLOSE_REUSE_EXISTING: bool\n    GAP_CLOSE_REUSE_UM: float\n    GAP_CLOSE_MAX_ADDED_FRAC: float\n    GAP_CLOSE_MAX_ADDED_ABS: int\n    GAP_REFINE_SYNTHETIC: bool\n    GAP_REFINE_WIN_Z: int\n    GAP_REFINE_WIN_YX: int\n    GAP_REFINE_MAX_SHIFT_UM: float\n\n    # All-node intensity-centroid refinement (E23 parity; inert in base1).\n    REFINE_ALL_CENTROIDS: bool\n    REFINE_CENTROIDS_WIN_Z: int\n    REFINE_CENTROIDS_WIN_YX: int\n    REFINE_CENTROIDS_BASELINE_PERCENTILE: float\n    REFINE_CENTROIDS_MAX_SHIFT_UM: float\n\n    OUTPUT_FILTER_SHORT_TRACKS: bool\n    OUTPUT_MIN_TRACK_LEN: int\n    OUTPUT_KEEP_DIVISION_COMPONENTS: bool\n    ADAPTIVE_SHORT_TRACK_RESCUE: bool\n    SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC: float\n    SHORT_TRACK_RESCUE_MIN_LEN: int\n    SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB: float\n    SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM: float\n    SHORT_TRACK_RESCUE_MAX_NODES_FRAC: float\n    SHORT_TRACK_RESCUE_MAX_NODES_ABS: int\n\n    OUTPUT_LINEFIT_SMOOTH: bool\n    OUTPUT_LINEFIT_WEIGHT: float\n    OUTPUT_LINEFIT_WINDOW: int\n\n    OUTPUT_GAP2_RECOVERY: bool\n    GAP2_MAX_TOTAL_UM: float\n    GAP2_MAX_STEP_UM: float\n    GAP2_MAX_LINKS_FRAC: float\n    GAP2_MAX_LINKS_ABS: int\n    GAP2_REQUIRE_CONTEXT: bool\n    GAP2_FRAME_FRAC_CAP: float\n\n    OUTPUT_SAFE_DIVISIONS: bool\n    SAFE_DIV_MAX_UM: float\n    SAFE_DIV_SISTER_MAX_UM: float\n    SAFE_DIV_EXISTING_CHILD_MAX_UM: float\n    SAFE_DIV_FRAME_FRAC_CAP: float\n    SAFE_DIV_GLOBAL_FRAC_CAP: float\n    # Structural safe-division gates (E23 parity; inert in base1).\n    # SAFE_DIV_MODE selects the candidate-generation semantics: "legacy"\n    # (current port) or "e23" (notebook structural gates).\n    # REQUIRE_MUTUAL_NN / REQUIRE_DIVERGENCE mirror the notebook env vars\n    # BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN / BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE\n    # exactly; the submitted notebook hardwires both gates, so the e23\n    # preset enables them (and REQUIRE_MID_TRACK_PARENT, its C1 gate).\n    SAFE_DIV_MODE: str\n    SAFE_DIV_REQUIRE_MID_TRACK_PARENT: bool\n    SAFE_DIV_REQUIRE_MUTUAL_NN: bool\n    SAFE_DIV_REQUIRE_DIVERGENCE: bool\n    SAFE_DIV_DIVERGE_UM: float\n\n    OUTPUT_STEAL_TWIN_REWIRE: bool\n    STEAL_TWIN_MODE: str\n    STEAL_TWIN_DRY_RUN: bool\n    STEAL_TWIN_PARENT_MAX_UM: float\n    STEAL_TWIN_EXISTING_CHILD_MAX_UM: float\n    STEAL_TWIN_SISTER_MIN_UM: float\n    STEAL_TWIN_SISTER_MAX_UM: float\n    STEAL_TWIN_DIVERGE_UM: float\n    STEAL_TWIN_TWIN_MAX_UM: float\n    STEAL_TWIN_REQUIRE_TWO_SUCCESSORS: bool\n    STEAL_TWIN_REJECT_SYNTHETIC: bool\n    STEAL_TWIN_DEEPCENTER_VETO: bool\n    STEAL_TWIN_FRAME_CAP_ABS: int\n    STEAL_TWIN_VIDEO_CAP_ABS: int\n    STEAL_TWIN_DEBUG_JSONL: str\n    STEAL_TWIN_DEBUG_MAX_RECORDS: int\n\n    USE_DEEPCENTER_VETO: bool\n    REQUIRE_DEEPCENTER_VETO: bool\n    DEEPCENTER_GAP_VETO: bool\n    DEEPCENTER_SAFE_DIV_VETO: bool\n    DEEPCENTER_GAP_THRESHOLD: float\n    DEEPCENTER_EXPECTED_EPOCH: int\n    DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM: float\n    DEEPCENTER_SAFE_DIV_THRESHOLD: float\n    DEEPCENTER_SCORE_WIN_Z: int\n    DEEPCENTER_SCORE_WIN_YX: int\n    DEEPCENTER_SCORE_CACHE_MAX_FRAMES: int\n    DEEPCENTER_RELATIVE: str\n    DEEPCENTER_CHECKPOINT: str\n    DEEPCENTER_CHECKPOINT_DEFAULT: str\n    DEEPCENTER_MANIFEST: str\n    DEEPCENTER_MANIFEST_DEFAULT: str\n\n    EXPERIMENT_TAG: str\n\n\ndef build_config(\n    overrides: dict[str, str] | None = None,\n    test_dir: Path | str = Path("data/test"),\n    profile: str = "base1",\n) -> PostprocConfig:\n    """Build a :class:`PostprocConfig` the same way the notebook builds its constants.\n\n    Resolution order (last wins), mirroring "preset cell overwrites\n    os.environ, constants cell reads os.environ.get(name, hardcoded)":\n    ``CODE_DEFAULTS`` -> the named profile\'s preset (:data:`PROFILES`;\n    ``base1`` is :data:`PRESET`, ``e23`` is :data:`E23_PRESET`) ->\n    ``overrides`` (``--set``). Positional arguments are unchanged, so\n    existing callers keep building the byte-for-byte ``base1`` config.\n\n    Raises :class:`ValueError` for an unknown ``profile`` or an invalid\n    ``BIOHUB_SAFE_DIV_MODE`` override, and :class:`KeyError` for unknown\n    override keys.\n    """\n    try:\n        preset = PROFILES[profile]\n    except KeyError:\n        raise ValueError(f"unknown profile {profile!r}; expected one of {sorted(PROFILES)}") from None\n\n    env: dict[str, str] = dict(CODE_DEFAULTS)\n    env.update(preset)\n    if overrides:\n        unknown = sorted(set(overrides) - set(CODE_DEFAULTS))\n        if unknown:\n            raise KeyError(f"unknown BIOHUB_* override key(s): {unknown}")\n        env.update(overrides)\n\n    safe_div_mode = _get_str(env, "BIOHUB_SAFE_DIV_MODE")\n    if safe_div_mode not in ("legacy", "e23"):\n        raise ValueError(f"BIOHUB_SAFE_DIV_MODE must be \'legacy\' or \'e23\', got {safe_div_mode!r}")\n\n    output_steal_twin_rewire = _get_bool(env, "BIOHUB_OUTPUT_STEAL_TWIN_REWIRE")\n    steal_twin_values: dict[str, object] = {\n        "STEAL_TWIN_MODE": _get_str(env, "BIOHUB_STEAL_TWIN_MODE"),\n        "STEAL_TWIN_PARENT_MAX_UM": _get_float(env, "BIOHUB_STEAL_TWIN_PARENT_MAX_UM"),\n        "STEAL_TWIN_EXISTING_CHILD_MAX_UM": _get_float(env, "BIOHUB_STEAL_TWIN_EXISTING_CHILD_MAX_UM"),\n        "STEAL_TWIN_SISTER_MIN_UM": _get_float(env, "BIOHUB_STEAL_TWIN_SISTER_MIN_UM"),\n        "STEAL_TWIN_SISTER_MAX_UM": _get_float(env, "BIOHUB_STEAL_TWIN_SISTER_MAX_UM"),\n        "STEAL_TWIN_DIVERGE_UM": _get_float(env, "BIOHUB_STEAL_TWIN_DIVERGE_UM"),\n        "STEAL_TWIN_TWIN_MAX_UM": _get_float(env, "BIOHUB_STEAL_TWIN_TWIN_MAX_UM"),\n        "STEAL_TWIN_REQUIRE_TWO_SUCCESSORS": _get_bool(env, "BIOHUB_STEAL_TWIN_REQUIRE_TWO_SUCCESSORS"),\n        "STEAL_TWIN_REJECT_SYNTHETIC": _get_bool(env, "BIOHUB_STEAL_TWIN_REJECT_SYNTHETIC"),\n        "STEAL_TWIN_DEEPCENTER_VETO": _get_bool(env, "BIOHUB_STEAL_TWIN_DEEPCENTER_VETO"),\n        "STEAL_TWIN_FRAME_CAP_ABS": _get_int(env, "BIOHUB_STEAL_TWIN_FRAME_CAP_ABS"),\n        "STEAL_TWIN_VIDEO_CAP_ABS": _get_int(env, "BIOHUB_STEAL_TWIN_VIDEO_CAP_ABS"),\n        "STEAL_TWIN_DEBUG_MAX_RECORDS": _get_int(env, "BIOHUB_STEAL_TWIN_DEBUG_MAX_RECORDS"),\n    }\n    frozen_steal_twin_values: dict[str, object] = {\n        "STEAL_TWIN_MODE": "twin_only_v1",\n        "STEAL_TWIN_PARENT_MAX_UM": 8.0,\n        "STEAL_TWIN_EXISTING_CHILD_MAX_UM": 10.0,\n        "STEAL_TWIN_SISTER_MIN_UM": 5.5,\n        "STEAL_TWIN_SISTER_MAX_UM": 11.0,\n        "STEAL_TWIN_DIVERGE_UM": 2.25,\n        "STEAL_TWIN_TWIN_MAX_UM": 5.0,\n        "STEAL_TWIN_REQUIRE_TWO_SUCCESSORS": True,\n        "STEAL_TWIN_REJECT_SYNTHETIC": True,\n        "STEAL_TWIN_DEEPCENTER_VETO": True,\n        "STEAL_TWIN_FRAME_CAP_ABS": 1,\n        "STEAL_TWIN_VIDEO_CAP_ABS": 2,\n        "STEAL_TWIN_DEBUG_MAX_RECORDS": 200,\n    }\n    if output_steal_twin_rewire:\n        for name, expected in frozen_steal_twin_values.items():\n            actual = steal_twin_values[name]\n            if actual != expected:\n                raise ValueError(f"BIOHUB_{name} must equal frozen twin_only_v1 value {expected!r}, got {actual!r}")\n\n    return PostprocConfig(\n        TEST_DIR=Path(test_dir),\n        OUTPUT_EDGE_MAX_UM=_get_float(env, "BIOHUB_OUTPUT_EDGE_MAX_UM"),\n        OUTPUT_ENFORCE_NEXT_FRAME=_get_bool(env, "BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME"),\n        OUTPUT_SINGLE_PARENT_REPAIR=_get_bool(env, "BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR"),\n        OUTPUT_SINGLE_CHILD_REPAIR=_get_bool(env, "BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR"),\n        OUTPUT_PRUNE_ISOLATED=_get_bool(env, "BIOHUB_OUTPUT_PRUNE_ISOLATED"),\n        OUTPUT_MOTION_RELINK=_get_bool(env, "BIOHUB_OUTPUT_MOTION_RELINK"),\n        MOTION_RELINK_TIGHT_UM=_get_float(env, "BIOHUB_MOTION_RELINK_TIGHT_UM"),\n        MOTION_RELINK_RELAXED_UM=_get_float(env, "BIOHUB_MOTION_RELINK_RELAXED_UM"),\n        MOTION_RELINK_VELOCITY_WEIGHT=_get_float(env, "BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT"),\n        MOTION_RELINK_LEARNED_BONUS=_get_float(env, "BIOHUB_MOTION_RELINK_LEARNED_BONUS"),\n        MOTION_RELINK_MAX_FRAME_NODES=_get_int(env, "BIOHUB_MOTION_RELINK_MAX_FRAME_NODES"),\n        OUTPUT_DIVISION_GEOMETRY_FILTER=_get_bool(env, "BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER"),\n        DIV_PARENT_MAX_UM=_get_float(env, "BIOHUB_DIV_PARENT_MAX_UM"),\n        DIV_SISTER_MAX_UM=_get_float(env, "BIOHUB_DIV_SISTER_MAX_UM"),\n        DIV_DROP_TO_SINGLE_IF_BAD=_get_bool(env, "BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD"),\n        OUTPUT_GAP_CLOSE=_get_bool(env, "BIOHUB_OUTPUT_GAP_CLOSE"),\n        GAP_CLOSE_MAX_GAP=_get_int(env, "BIOHUB_GAP_CLOSE_MAX_GAP"),\n        GAP_CLOSE_UM=_get_float(env, "BIOHUB_GAP_CLOSE_UM"),\n        GAP_DENSITY_ADAPTIVE=_get_bool(env, "BIOHUB_GAP_DENSITY_ADAPTIVE"),\n        GAP_DENSITY_REFERENCE_UM=_get_float(env, "BIOHUB_GAP_DENSITY_REFERENCE_UM"),\n        GAP_DENSITY_GAIN=_get_float(env, "BIOHUB_GAP_DENSITY_GAIN"),\n        GAP_DENSITY_MAX_STEP_DELTA_UM=_get_float(env, "BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"),\n        GAP_DENSITY_NEIGHBORS=_get_int(env, "BIOHUB_GAP_DENSITY_NEIGHBORS"),\n        GAP_CLOSE_REUSE_EXISTING=_get_bool(env, "BIOHUB_GAP_CLOSE_REUSE_EXISTING"),\n        GAP_CLOSE_REUSE_UM=_get_float(env, "BIOHUB_GAP_CLOSE_REUSE_UM"),\n        GAP_CLOSE_MAX_ADDED_FRAC=_get_float(env, "BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC"),\n        GAP_CLOSE_MAX_ADDED_ABS=_get_int(env, "BIOHUB_GAP_CLOSE_MAX_ADDED_ABS"),\n        GAP_REFINE_SYNTHETIC=_get_bool(env, "BIOHUB_GAP_REFINE_SYNTHETIC"),\n        GAP_REFINE_WIN_Z=_get_int(env, "BIOHUB_GAP_REFINE_WIN_Z"),\n        GAP_REFINE_WIN_YX=_get_int(env, "BIOHUB_GAP_REFINE_WIN_YX"),\n        GAP_REFINE_MAX_SHIFT_UM=_get_float(env, "BIOHUB_GAP_REFINE_MAX_SHIFT_UM"),\n        REFINE_ALL_CENTROIDS=_get_bool(env, "BIOHUB_REFINE_ALL_CENTROIDS"),\n        REFINE_CENTROIDS_WIN_Z=_get_int(env, "BIOHUB_REFINE_CENTROIDS_WIN_Z"),\n        REFINE_CENTROIDS_WIN_YX=_get_int(env, "BIOHUB_REFINE_CENTROIDS_WIN_YX"),\n        REFINE_CENTROIDS_BASELINE_PERCENTILE=_get_float(env, "BIOHUB_REFINE_CENTROIDS_BASELINE_PERCENTILE"),\n        REFINE_CENTROIDS_MAX_SHIFT_UM=_get_float(env, "BIOHUB_REFINE_CENTROIDS_MAX_SHIFT_UM"),\n        OUTPUT_FILTER_SHORT_TRACKS=_get_bool(env, "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"),\n        OUTPUT_MIN_TRACK_LEN=_get_int(env, "BIOHUB_OUTPUT_MIN_TRACK_LEN"),\n        OUTPUT_KEEP_DIVISION_COMPONENTS=_get_bool(env, "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"),\n        ADAPTIVE_SHORT_TRACK_RESCUE=_get_bool(env, "BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"),\n        SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC=_get_float(env, "BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC"),\n        SHORT_TRACK_RESCUE_MIN_LEN=_get_int(env, "BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN"),\n        SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB=_get_float(env, "BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB"),\n        SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM=_get_float(env, "BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM"),\n        SHORT_TRACK_RESCUE_MAX_NODES_FRAC=_get_float(env, "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC"),\n        SHORT_TRACK_RESCUE_MAX_NODES_ABS=_get_int(env, "BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS"),\n        OUTPUT_LINEFIT_SMOOTH=_get_bool(env, "BIOHUB_OUTPUT_LINEFIT_SMOOTH"),\n        OUTPUT_LINEFIT_WEIGHT=_get_float(env, "BIOHUB_OUTPUT_LINEFIT_WEIGHT"),\n        OUTPUT_LINEFIT_WINDOW=_get_int(env, "BIOHUB_OUTPUT_LINEFIT_WINDOW"),\n        OUTPUT_GAP2_RECOVERY=_get_bool(env, "BIOHUB_OUTPUT_GAP2_RECOVERY"),\n        GAP2_MAX_TOTAL_UM=_get_float(env, "BIOHUB_GAP2_MAX_TOTAL_UM"),\n        GAP2_MAX_STEP_UM=_get_float(env, "BIOHUB_GAP2_MAX_STEP_UM"),\n        GAP2_MAX_LINKS_FRAC=_get_float(env, "BIOHUB_GAP2_MAX_LINKS_FRAC"),\n        GAP2_MAX_LINKS_ABS=_get_int(env, "BIOHUB_GAP2_MAX_LINKS_ABS"),\n        GAP2_REQUIRE_CONTEXT=_get_bool(env, "BIOHUB_GAP2_REQUIRE_CONTEXT"),\n        GAP2_FRAME_FRAC_CAP=_get_float(env, "BIOHUB_GAP2_FRAME_FRAC_CAP"),\n        OUTPUT_SAFE_DIVISIONS=_get_bool(env, "BIOHUB_OUTPUT_SAFE_DIVISIONS"),\n        SAFE_DIV_MAX_UM=_get_float(env, "BIOHUB_SAFE_DIV_MAX_UM"),\n        SAFE_DIV_SISTER_MAX_UM=_get_float(env, "BIOHUB_SAFE_DIV_SISTER_MAX_UM"),\n        SAFE_DIV_EXISTING_CHILD_MAX_UM=_get_float(env, "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"),\n        SAFE_DIV_FRAME_FRAC_CAP=_get_float(env, "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"),\n        SAFE_DIV_GLOBAL_FRAC_CAP=_get_float(env, "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"),\n        SAFE_DIV_MODE=safe_div_mode,\n        SAFE_DIV_REQUIRE_MID_TRACK_PARENT=_get_bool(env, "BIOHUB_SAFE_DIV_REQUIRE_MID_TRACK_PARENT"),\n        SAFE_DIV_REQUIRE_MUTUAL_NN=_get_bool(env, "BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN"),\n        SAFE_DIV_REQUIRE_DIVERGENCE=_get_bool(env, "BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE"),\n        SAFE_DIV_DIVERGE_UM=_get_float(env, "BIOHUB_SAFE_DIV_DIVERGE_UM"),\n        OUTPUT_STEAL_TWIN_REWIRE=output_steal_twin_rewire,\n        STEAL_TWIN_MODE=str(steal_twin_values["STEAL_TWIN_MODE"]),\n        STEAL_TWIN_DRY_RUN=_get_bool(env, "BIOHUB_STEAL_TWIN_DRY_RUN"),\n        STEAL_TWIN_PARENT_MAX_UM=float(steal_twin_values["STEAL_TWIN_PARENT_MAX_UM"]),\n        STEAL_TWIN_EXISTING_CHILD_MAX_UM=float(steal_twin_values["STEAL_TWIN_EXISTING_CHILD_MAX_UM"]),\n        STEAL_TWIN_SISTER_MIN_UM=float(steal_twin_values["STEAL_TWIN_SISTER_MIN_UM"]),\n        STEAL_TWIN_SISTER_MAX_UM=float(steal_twin_values["STEAL_TWIN_SISTER_MAX_UM"]),\n        STEAL_TWIN_DIVERGE_UM=float(steal_twin_values["STEAL_TWIN_DIVERGE_UM"]),\n        STEAL_TWIN_TWIN_MAX_UM=float(steal_twin_values["STEAL_TWIN_TWIN_MAX_UM"]),\n        STEAL_TWIN_REQUIRE_TWO_SUCCESSORS=bool(steal_twin_values["STEAL_TWIN_REQUIRE_TWO_SUCCESSORS"]),\n        STEAL_TWIN_REJECT_SYNTHETIC=bool(steal_twin_values["STEAL_TWIN_REJECT_SYNTHETIC"]),\n        STEAL_TWIN_DEEPCENTER_VETO=bool(steal_twin_values["STEAL_TWIN_DEEPCENTER_VETO"]),\n        STEAL_TWIN_FRAME_CAP_ABS=int(steal_twin_values["STEAL_TWIN_FRAME_CAP_ABS"]),\n        STEAL_TWIN_VIDEO_CAP_ABS=int(steal_twin_values["STEAL_TWIN_VIDEO_CAP_ABS"]),\n        STEAL_TWIN_DEBUG_JSONL=_get_str(env, "BIOHUB_STEAL_TWIN_DEBUG_JSONL"),\n        STEAL_TWIN_DEBUG_MAX_RECORDS=int(steal_twin_values["STEAL_TWIN_DEBUG_MAX_RECORDS"]),\n        USE_DEEPCENTER_VETO=_get_bool(env, "BIOHUB_USE_DEEPCENTER_VETO"),\n        REQUIRE_DEEPCENTER_VETO=_get_bool(env, "BIOHUB_REQUIRE_DEEPCENTER_VETO"),\n        DEEPCENTER_GAP_VETO=_get_bool(env, "BIOHUB_DEEPCENTER_GAP_VETO"),\n        DEEPCENTER_SAFE_DIV_VETO=_get_bool(env, "BIOHUB_DEEPCENTER_SAFE_DIV_VETO"),\n        DEEPCENTER_GAP_THRESHOLD=_get_float(env, "BIOHUB_DEEPCENTER_GAP_THRESHOLD"),\n        DEEPCENTER_EXPECTED_EPOCH=_get_int(env, "BIOHUB_DEEPCENTER_EXPECTED_EPOCH"),\n        DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM=_get_float(env, "BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"),\n        DEEPCENTER_SAFE_DIV_THRESHOLD=_get_float(env, "BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD"),\n        DEEPCENTER_SCORE_WIN_Z=_get_int(env, "BIOHUB_DEEPCENTER_SCORE_WIN_Z"),\n        DEEPCENTER_SCORE_WIN_YX=_get_int(env, "BIOHUB_DEEPCENTER_SCORE_WIN_YX"),\n        DEEPCENTER_SCORE_CACHE_MAX_FRAMES=_get_int(env, "BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES"),\n        DEEPCENTER_RELATIVE=_get_str(env, "BIOHUB_DEEPCENTER_RELATIVE"),\n        DEEPCENTER_CHECKPOINT=_get_str(env, "BIOHUB_DEEPCENTER_CHECKPOINT"),\n        DEEPCENTER_CHECKPOINT_DEFAULT=_get_str(env, "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT"),\n        DEEPCENTER_MANIFEST=_get_str(env, "BIOHUB_DEEPCENTER_MANIFEST"),\n        DEEPCENTER_MANIFEST_DEFAULT=_get_str(env, "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT"),\n        EXPERIMENT_TAG=_get_str(env, "BIOHUB_EXPERIMENT_TAG"),\n    )\n\n\ndef config_field_names() -> list[str]:\n    return [f.name for f in fields(PostprocConfig)]\n\n\ndef parse_set_overrides(pairs: list[str]) -> dict[str, str]:\n    """Parse repeated ``--set NAME=VALUE`` CLI args into a ``BIOHUB_*`` override dict.\n\n    ``NAME`` may omit the ``BIOHUB_`` prefix for brevity.\n    """\n    overrides: dict[str, str] = {}\n    for pair in pairs:\n        if "=" not in pair:\n            raise SystemExit(f"--set expects NAME=VALUE, got: {pair!r}")\n        name, value = pair.split("=", 1)\n        name = name.strip()\n        if not name.startswith("BIOHUB_"):\n            name = f"BIOHUB_{name}"\n        overrides[name] = value.strip()\n    return overrides\n', 'src/biohub/public_postproc/csv_out.py': '"""Streaming CSV writer + run_stats assembly, ported verbatim from the notebook cell."""\nfrom __future__ import annotations\n\nimport csv\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom typing import TextIO\n\nimport pandas as pd\n\nSUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\nCSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]\nNodeSerializer = Callable[[dict[str, object], str, int], dict[str, object]]\n\n\nclass SubmissionCsvWriter:\n    """Streaming writer with a single ``id`` counter running across all datasets.\n\n    Row order (and therefore the ``id`` values) must match the notebook: for\n    each dataset, node rows sorted by ``node_id`` then edge rows in the\n    filtered-graph\'s edge-list order.\n    """\n\n    def __init__(self, handle: TextIO, *, node_serializer: NodeSerializer | None = None) -> None:\n        self._writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)\n        self._writer.writeheader()\n        self._node_serializer = node_serializer\n        self.row_id = 0\n\n    def write_nodes(self, dataset: str, nodes_by_id: dict[int, dict[str, object]]) -> None:\n        for node_id in sorted(nodes_by_id):\n            node = nodes_by_id[node_id]\n            row = self._node_serializer(node, dataset, self.row_id) if self._node_serializer is not None else {\n                "id": self.row_id,\n                "dataset": dataset,\n                "row_type": "node",\n                "node_id": int(node["node_id"]),\n                "t": int(node["t"]),\n                "z": max(0, int(round(float(node["z"])))),\n                "y": max(0, int(round(float(node["y"])))),\n                "x": max(0, int(round(float(node["x"])))),\n                "source_id": -1,\n                "target_id": -1,\n            }\n            self._writer.writerow(row)\n            self.row_id += 1\n\n    def write_edges(self, dataset: str, nodes_by_id: dict[int, dict[str, object]], edges: list[dict[str, object]]) -> dict[int, int]:\n        """Write edge rows in list order; returns {source_id: out_degree} for division stats."""\n        division_sources: dict[int, int] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            target_id = int(edge["target_id"])\n            if source_id not in nodes_by_id or target_id not in nodes_by_id:\n                raise AssertionError(f"{dataset}: dangling edge after filtering")\n            self._writer.writerow({\n                "id": self.row_id,\n                "dataset": dataset,\n                "row_type": "edge",\n                "node_id": -1,\n                "t": -1,\n                "z": -1,\n                "y": -1,\n                "x": -1,\n                "source_id": source_id,\n                "target_id": target_id,\n            })\n            self.row_id += 1\n            division_sources[source_id] = division_sources.get(source_id, 0) + 1\n        return division_sources\n\n\ndef build_run_stats_frame(stats_rows: list[dict[str, object]]) -> pd.DataFrame:\n    return pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)\n\n\ndef write_run_stats(stats_rows: list[dict[str, object]], path: Path) -> pd.DataFrame:\n    stats = build_run_stats_frame(stats_rows)\n    stats.to_csv(path, index=False)\n    return stats\n', 'src/biohub/public_postproc/deepcenter.py': '"""DeepCenter add-only repair gate, ported verbatim from the notebook cell.\n\nDisabled by the ``biohub_132_clean_short_track_rescue_lightcv_nohack`` preset\n(``BIOHUB_USE_DEEPCENTER_VETO=0``) that produced ``outputs/kaggle/base1_v1``,\nso none of this needs to fire for the port to reproduce that submission.\nKept so the code path -- and any future preset that re-enables it -- still\nworks; ``torch`` is optional and only imported when the gate is actually\nrequested (it is not a project dependency).\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport stat\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom types import MappingProxyType, SimpleNamespace\nfrom typing import BinaryIO\n\nimport numpy as np\n\nfrom biohub.public_postproc.config import PostprocConfig\nfrom biohub.public_postproc.frames import read_test_frame\nfrom biohub.public_postproc.geometry import node_point  # noqa: F401  (re-exported for callers)\n\ntry:\n    import torch\nexcept Exception:  # pragma: no cover - torch is not a project dependency\n    torch = None\n\n\nSTRICT_DEEPCENTER_CHECKPOINT_SHA256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"\nSTRICT_DEEPCENTER_MANIFEST_SHA256 = "1eedc1af72b10c464c6995013075310510b6f6e634450ff2bc170c67b89ce911"\nSTRICT_DEEPCENTER_EPOCH = 2\nSTRICT_DEEPCENTER_MANIFEST_MODEL_CONFIG = MappingProxyType(\n    {\n        "base_channels": 24,\n        "batch_size": 8,\n        "bg_quantile": 0.4,\n        "brightness_jitter": 0.0,\n        "epochs": 1000,\n        "frames_per_movie": 0,\n        "gauss_sigma": 1.0,\n        "grad_clip_norm": None,\n        "learning_rate": 0.001,\n        "movie_limit": None,\n        "norm_clip_hi": 6.0,\n        "norm_clip_lo": -0.5,\n        "norm_hi_pct": 99.5,\n        "norm_lo_pct": 50.0,\n        "num_workers": 4,\n        "pool_factor": 4,\n        "pos_thresh": 0.05,\n        "random_flip": True,\n        "seed": 2026,\n        "val_fraction": 0.1,\n        "w_bg": 1.0,\n        "w_ignore": 0.05,\n        "w_pos": 12.0,\n        "weight_decay": 0.0,\n    }\n)\nSTRICT_DEEPCENTER_CHECKPOINT_MODEL_CONFIG = MappingProxyType(\n    {\n        **STRICT_DEEPCENTER_MANIFEST_MODEL_CONFIG,\n        "epochs": 50,\n    }\n)\nSTRICT_DEEPCENTER_CONFIG_DISCREPANCY = MappingProxyType(\n    {\n        "field": "epochs",\n        "manifest": 1000,\n        "checkpoint": 50,\n    }\n)\n\n\ndef _strict_stat_identity(value: os.stat_result) -> tuple[int, int, int, int, int, int]:\n    return (\n        value.st_dev,\n        value.st_ino,\n        value.st_mode,\n        value.st_size,\n        value.st_mtime_ns,\n        value.st_ctime_ns,\n    )\n\n\ndef _strict_stat_plain(value: os.stat_result) -> dict[str, int]:\n    return {\n        "device": value.st_dev,\n        "inode": value.st_ino,\n        "mode": value.st_mode,\n        "size": value.st_size,\n        "mtime_ns": value.st_mtime_ns,\n        "ctime_ns": value.st_ctime_ns,\n    }\n\n\ndef _strict_config_matches(actual: object, expected: dict[str, object]) -> bool:\n    if type(actual) is not dict or set(actual) != set(expected):\n        return False\n    for key, expected_value in expected.items():\n        actual_value = actual[key]\n        if type(actual_value) is not type(expected_value):\n            return False\n        if type(expected_value) is float:\n            if not np.isfinite(actual_value) or actual_value.hex() != expected_value.hex():\n                return False\n        elif actual_value != expected_value:\n            return False\n    return True\n\n\ndef _strict_read_one_file[T](\n    path: Path,\n    expected_sha256: str,\n    reader: Callable[[BinaryIO], T],\n) -> tuple[T, dict[str, object]]:\n    """Hash and parse one regular, nonsymlink file through one retained FD."""\n    if type(path) is not type(Path()):\n        raise TypeError("strict DeepCenter paths must be exact pathlib.Path instances")\n    if type(expected_sha256) is not str or len(expected_sha256) != 64:\n        raise ValueError("expected SHA-256 must be 64 lowercase hexadecimal characters")\n    try:\n        bytes.fromhex(expected_sha256)\n    except ValueError as exc:\n        raise ValueError("expected SHA-256 must be hexadecimal") from exc\n    if expected_sha256 != expected_sha256.lower():\n        raise ValueError("expected SHA-256 must be lowercase")\n\n    if not hasattr(os, "O_NOFOLLOW"):\n        raise RuntimeError("strict DeepCenter loading requires O_NOFOLLOW support")\n    flags = os.O_RDONLY | getattr(os, "O_CLOEXEC", 0) | os.O_NOFOLLOW\n    fd = os.open(path, flags)\n    try:\n        before = os.fstat(fd)\n        if not stat.S_ISREG(before.st_mode):\n            raise ValueError(f"strict DeepCenter input is not a regular file: {path.name}")\n        with os.fdopen(fd, "rb", closefd=False) as handle:\n            digest = hashlib.sha256()\n            while True:\n                chunk = handle.read(1024 * 1024)\n                if not chunk:\n                    break\n                digest.update(chunk)\n            after_hash = os.fstat(fd)\n            if _strict_stat_identity(after_hash) != _strict_stat_identity(before):\n                raise RuntimeError(f"strict DeepCenter input changed while hashing: {path.name}")\n            actual_sha256 = digest.hexdigest()\n            if actual_sha256 != expected_sha256:\n                raise ValueError(\n                    f"strict DeepCenter SHA-256 mismatch for {path.name}: "\n                    f"expected {expected_sha256}, got {actual_sha256}"\n                )\n            handle.seek(0)\n            value = reader(handle)\n            after_read = os.fstat(fd)\n            if _strict_stat_identity(after_read) != _strict_stat_identity(before):\n                raise RuntimeError(f"strict DeepCenter input changed while reading: {path.name}")\n        return value, {\n            "sha256": actual_sha256,\n            "pre": _strict_stat_plain(before),\n            "post_hash": _strict_stat_plain(after_hash),\n            "post_read": _strict_stat_plain(after_read),\n            "open_count": 1,\n        }\n    finally:\n        os.close(fd)\n\n\ndef _load_deepcenter_veto_detector_strict_verified(\n    cfg: PostprocConfig,\n    checkpoint_path: Path,\n    manifest_path: Path,\n    *,\n    expected_checkpoint_sha256: str,\n    expected_manifest_sha256: str,\n    expected_epoch: int,\n    expected_manifest_model_config: object,\n    expected_checkpoint_model_config: object,\n) -> tuple[dict[str, object], dict[str, object]]:\n    """Private injected-pin seam; production uses the non-overridable wrapper."""\n    if not cfg.USE_DEEPCENTER_VETO or not cfg.REQUIRE_DEEPCENTER_VETO:\n        raise ValueError("strict DeepCenter requires USE_DEEPCENTER_VETO and REQUIRE_DEEPCENTER_VETO")\n    if torch is None:\n        raise ImportError("torch is required for strict DeepCenter loading")\n    if type(expected_epoch) is not int or expected_epoch != 2:\n        raise ValueError("strict DeepCenter expected epoch must be exact built-in int 2")\n    if not isinstance(expected_manifest_model_config, (dict, MappingProxyType)):\n        raise TypeError("expected DeepCenter manifest model config must be a mapping")\n    if not isinstance(expected_checkpoint_model_config, (dict, MappingProxyType)):\n        raise TypeError("expected DeepCenter checkpoint model config must be a mapping")\n    frozen_manifest_config = dict(expected_manifest_model_config)\n    frozen_checkpoint_config = dict(expected_checkpoint_model_config)\n\n    def read_manifest(handle: BinaryIO) -> object:\n        return json.load(handle)\n\n    manifest, manifest_file = _strict_read_one_file(\n        manifest_path,\n        expected_manifest_sha256,\n        read_manifest,\n    )\n    if type(manifest) is not dict:\n        raise ValueError("strict DeepCenter manifest must be a JSON object")\n    try:\n        manifest_model = manifest["model"]\n        manifest_checkpoint = manifest_model["best_checkpoint"]\n        manifest_summary = manifest_model["best_checkpoint_summary"]\n    except (KeyError, TypeError) as exc:\n        raise ValueError("strict DeepCenter manifest has no registered best checkpoint") from exc\n    if (\n        type(manifest_model) is not dict\n        or type(manifest_checkpoint) is not dict\n        or type(manifest_summary) is not dict\n        or manifest_checkpoint.get("sha256") != expected_checkpoint_sha256\n        or type(manifest_summary.get("epoch")) is not int\n        or manifest_summary.get("epoch") != expected_epoch\n        or not _strict_config_matches(manifest_model.get("config"), frozen_manifest_config)\n    ):\n        raise ValueError("strict DeepCenter manifest identity/config mismatch")\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    def read_checkpoint(handle: BinaryIO) -> object:\n        return torch.load(handle, map_location=device, weights_only=False)\n\n    checkpoint, checkpoint_file = _strict_read_one_file(\n        checkpoint_path,\n        expected_checkpoint_sha256,\n        read_checkpoint,\n    )\n    if type(checkpoint) is not dict or "model_state" not in checkpoint:\n        raise ValueError("strict DeepCenter checkpoint has no model_state")\n    if type(checkpoint.get("epoch")) is not int or checkpoint["epoch"] != expected_epoch:\n        raise ValueError("strict DeepCenter checkpoint epoch mismatch")\n    if not _strict_config_matches(checkpoint.get("config"), frozen_checkpoint_config):\n        raise ValueError("strict DeepCenter checkpoint config mismatch")\n    model_cfg = SimpleNamespace(**checkpoint["config"])\n    model = _DCDeepCenterUNet3D(base_channels=model_cfg.base_channels)\n    model.load_state_dict(checkpoint["model_state"])\n    model.to(device)\n    model.eval()\n    bundle = {\n        "model": model,\n        "cfg": model_cfg,\n        "device": device,\n        "path": checkpoint_path,\n        "torch": torch,\n        "checkpoint_epoch": expected_epoch,\n    }\n    receipt = {\n        "schema_version": "biohub.st_r3.deepcenter_receipt.v1",\n        "registered": {\n            "checkpoint_name": checkpoint_path.name,\n            "manifest_name": manifest_path.name,\n        },\n        "checkpoint": checkpoint_file,\n        "manifest": manifest_file,\n        "chosen_artifact": checkpoint_path.name,\n        "verified_epoch": expected_epoch,\n        "verified_configs": {\n            "manifest": frozen_manifest_config,\n            "checkpoint": frozen_checkpoint_config,\n        },\n        "known_config_discrepancy": {\n            "field": "epochs",\n            "manifest": frozen_manifest_config.get("epochs"),\n            "checkpoint": frozen_checkpoint_config.get("epochs"),\n        },\n        "inference_relevant_config_equal": all(\n            type(frozen_manifest_config[key]) is type(frozen_checkpoint_config.get(key))\n            and frozen_manifest_config[key] == frozen_checkpoint_config[key]\n            for key in frozen_manifest_config\n            if key != "epochs"\n        )\n        and set(frozen_manifest_config) == set(frozen_checkpoint_config),\n        "device": str(device),\n        "dtype": "float32",\n        "open_count": 2,\n        "fallback_candidates": 0,\n    }\n    return bundle, receipt\n\n\ndef load_deepcenter_veto_detector_strict(\n    cfg: PostprocConfig,\n    checkpoint_path: Path,\n    manifest_path: Path,\n) -> tuple[dict[str, object], dict[str, object]]:\n    """Load only the exact preregistered DeepCenter artifact, without fallback."""\n    return _load_deepcenter_veto_detector_strict_verified(\n        cfg,\n        checkpoint_path,\n        manifest_path,\n        expected_checkpoint_sha256=STRICT_DEEPCENTER_CHECKPOINT_SHA256,\n        expected_manifest_sha256=STRICT_DEEPCENTER_MANIFEST_SHA256,\n        expected_epoch=STRICT_DEEPCENTER_EPOCH,\n        expected_manifest_model_config=STRICT_DEEPCENTER_MANIFEST_MODEL_CONFIG,\n        expected_checkpoint_model_config=STRICT_DEEPCENTER_CHECKPOINT_MODEL_CONFIG,\n    )\n\n\ndef load_deepcenter_veto_detector_e31_target(\n    cfg: PostprocConfig,\n    checkpoint_path: Path,\n    manifest_path: Path,\n) -> tuple[dict[str, object], dict[str, object]]:\n    """Load E31 target ep500 DeepCenter artifact (package unchanged bestepoch2).\n\n    Uses the updated manifest hash for the 500-epoch metadata while keeping\n    all other expected constants identical to the strict loader. Never falls\n    back to alternate artifacts or relaxed verification.\n    """\n    return _load_deepcenter_veto_detector_strict_verified(\n        cfg,\n        checkpoint_path,\n        manifest_path,\n        expected_checkpoint_sha256=STRICT_DEEPCENTER_CHECKPOINT_SHA256,\n        expected_manifest_sha256="3bfe97304e9bbc3b3481a095392a1e83937315325bac8b987eb527d2951b96f3",\n        expected_epoch=STRICT_DEEPCENTER_EPOCH,\n        expected_manifest_model_config=STRICT_DEEPCENTER_MANIFEST_MODEL_CONFIG,\n        expected_checkpoint_model_config=STRICT_DEEPCENTER_CHECKPOINT_MODEL_CONFIG,\n    )\n\n\ndef _dc_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:\n    if factor <= 1:\n        return volume.astype(np.float32, copy=False)\n    z, y, x = volume.shape\n    y2 = (y // factor) * factor\n    x2 = (x // factor) * factor\n    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)\n    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))\n\n\ndef _dc_normalize_dynamic_range(volume: np.ndarray, model_cfg: object) -> np.ndarray:\n    vol = np.asarray(volume, dtype=np.float32)\n    lo = float(np.percentile(vol, float(getattr(model_cfg, "norm_lo_pct", 50.0))))\n    hi = float(np.percentile(vol, float(getattr(model_cfg, "norm_hi_pct", 99.5))))\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\n        return np.zeros_like(vol, dtype=np.float32)\n    ratio = (vol - lo) / (hi - lo)\n    return np.clip(\n        ratio,\n        float(getattr(model_cfg, "norm_clip_lo", -0.5)),\n        float(getattr(model_cfg, "norm_clip_hi", 6.0)),\n    ).astype(np.float32)\n\n\ndef _dc_manifest_weight_paths(manifest_path: Path) -> list[Path]:\n    if not manifest_path.exists():\n        return []\n    import json\n\n    try:\n        manifest = json.loads(manifest_path.read_text())\n    except Exception as exc:\n        print("Could not read DeepCenter manifest:", manifest_path, type(exc).__name__, exc)\n        return []\n    root = manifest_path.parent\n    sections: list[dict[str, object]] = []\n    for section in [\n        manifest.get("model", {}),\n        manifest.get("models", {}).get("full_frame_center", {}) if isinstance(manifest.get("models", {}), dict) else {},\n        manifest.get("full_frame_center", {}),\n    ]:\n        if isinstance(section, dict):\n            sections.append(section)\n    candidates: list[Path] = []\n    for section in sections:\n        for key in ("weight_path", "path"):\n            rel = section.get(key)\n            if isinstance(rel, str) and rel:\n                candidates.append(root / rel)\n        for key in ("last_checkpoint", "best_checkpoint"):\n            item = section.get(key)\n            if isinstance(item, dict):\n                rel = item.get("path")\n                if isinstance(rel, str) and rel:\n                    candidates.append(root / rel)\n    for name in ("checkpoint_last.pt", "best.pt", "last.pt"):\n        candidates.append(root / "weights" / "full_frame_center" / name)\n        candidates.append(root / name)\n    return candidates\n\n\ndef _dc_checkpoint_candidates(cfg: PostprocConfig) -> list[Path]:\n    candidates: list[Path] = []\n    explicit = (cfg.DEEPCENTER_CHECKPOINT or cfg.DEEPCENTER_CHECKPOINT_DEFAULT).strip()\n    if explicit:\n        candidates.append(Path(explicit))\n    manifest_explicit = (cfg.DEEPCENTER_MANIFEST or cfg.DEEPCENTER_MANIFEST_DEFAULT).strip()\n    if manifest_explicit:\n        candidates.extend(_dc_manifest_weight_paths(Path(manifest_explicit)))\n    candidates.append(Path(cfg.DEEPCENTER_CHECKPOINT_DEFAULT).parent / cfg.DEEPCENTER_RELATIVE)\n\n    seen: set[Path] = set()\n    out: list[Path] = []\n    for path in candidates:\n        path = path.expanduser()\n        try:\n            key = path.resolve() if path.exists() else path\n        except Exception:\n            key = path\n        if key in seen:\n            continue\n        seen.add(key)\n        out.append(path)\n    return out\n\n\nif torch is not None:\n\n    class _DCConvBlock3d(torch.nn.Module):\n        def __init__(self, in_channels: int, out_channels: int) -> None:\n            super().__init__()\n            groups = min(8, out_channels)\n            self.block = torch.nn.Sequential(\n                torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n                torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n            )\n\n        def forward(self, x):\n            return self.block(x)\n\n    class _DCDeepCenterUNet3D(torch.nn.Module):\n        def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:\n            super().__init__()\n            c = int(base_channels)\n            self.enc1 = _DCConvBlock3d(in_channels, c)\n            self.down1 = torch.nn.MaxPool3d(2, 2)\n            self.enc2 = _DCConvBlock3d(c, c * 2)\n            self.down2 = torch.nn.MaxPool3d(2, 2)\n            self.enc3 = _DCConvBlock3d(c * 2, c * 4)\n            self.down3 = torch.nn.MaxPool3d(2, 2)\n            self.bottleneck = _DCConvBlock3d(c * 4, c * 8)\n            self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)\n            self.dec3 = _DCConvBlock3d(c * 8, c * 4)\n            self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)\n            self.dec2 = _DCConvBlock3d(c * 4, c * 2)\n            self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)\n            self.dec1 = _DCConvBlock3d(c * 2, c)\n            self.head = torch.nn.Conv3d(c, 1, 1)\n\n        def forward(self, x):\n            e1 = self.enc1(x)\n            e2 = self.enc2(self.down1(e1))\n            e3 = self.enc3(self.down2(e2))\n            b = self.bottleneck(self.down3(e3))\n            d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\n            d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\n            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\n            return self.head(d1)\nelse:  # pragma: no cover - exercised only when torch is unavailable\n    _DCConvBlock3d = None\n    _DCDeepCenterUNet3D = None\n\n\ndef load_deepcenter_veto_detector(cfg: PostprocConfig) -> dict[str, object] | None:\n    if not cfg.USE_DEEPCENTER_VETO:\n        print("DeepCenter add-only repair gate disabled by configuration.")\n        return None\n    if torch is None:\n        if cfg.REQUIRE_DEEPCENTER_VETO:\n            raise ImportError("torch is required for DeepCenter add-only repair gate")\n        print("DeepCenter add-only repair gate skipped because torch is unavailable.")\n        return None\n    from types import SimpleNamespace\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    load_errors: list[str] = []\n    for checkpoint_path in _dc_checkpoint_candidates(cfg):\n        if not checkpoint_path.exists():\n            continue\n        try:\n            print("Trying DeepCenter add-only gate checkpoint:", checkpoint_path)\n            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n            if not isinstance(checkpoint, dict) or "model_state" not in checkpoint:\n                raise ValueError("checkpoint has no model_state")\n            checkpoint_epoch = int(checkpoint.get("epoch", -1))\n            if cfg.DEEPCENTER_EXPECTED_EPOCH > 0 and checkpoint_epoch != cfg.DEEPCENTER_EXPECTED_EPOCH:\n                raise ValueError(f"expected DeepCenter epoch {cfg.DEEPCENTER_EXPECTED_EPOCH}, got {checkpoint_epoch}")\n            model_cfg = SimpleNamespace(**checkpoint.get("config", {}))\n            model = _DCDeepCenterUNet3D(base_channels=int(getattr(model_cfg, "base_channels", 24)))\n            model.load_state_dict(checkpoint["model_state"])\n            model.to(device)\n            model.eval()\n            print("Loaded DeepCenter add-only gate checkpoint:", checkpoint_path)\n            print("DeepCenter checkpoint epoch:", checkpoint.get("epoch"), "best_score:", checkpoint.get("best_score"))\n            return {\n                "model": model,\n                "cfg": model_cfg,\n                "device": device,\n                "path": checkpoint_path,\n                "torch": torch,\n                "checkpoint_epoch": checkpoint_epoch,\n            }\n        except Exception as exc:\n            load_errors.append(f"{checkpoint_path}: {type(exc).__name__}: {exc}")\n            print("Skipping incompatible DeepCenter checkpoint:", checkpoint_path, "|", type(exc).__name__, exc)\n    message = "No usable DeepCenter checkpoint found for add-only repair gate."\n    if cfg.REQUIRE_DEEPCENTER_VETO:\n        checked = "\\n".join(str(p) for p in _dc_checkpoint_candidates(cfg)[:80])\n        errors = "\\n".join(load_errors[-20:])\n        raise FileNotFoundError(message + "\\nChecked:\\n" + checked + ("\\nLoad errors:\\n" + errors if errors else ""))\n    print(message)\n    return None\n\n\ndef _dc_cache_trim(cache: dict[tuple[str, int], np.ndarray], cfg: PostprocConfig) -> None:\n    limit = max(1, int(cfg.DEEPCENTER_SCORE_CACHE_MAX_FRAMES))\n    while len(cache) > limit:\n        cache.pop(next(iter(cache)))\n\n\ndef deepcenter_heatmap_for_frame(\n    cfg: PostprocConfig,\n    dataset: str,\n    t: int,\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> np.ndarray | None:\n    if detector_bundle is None:\n        return None\n    key = (dataset, int(t))\n    cached = heatmap_cache.get(key)\n    if cached is not None:\n        return cached\n    model = detector_bundle["model"]\n    model_cfg = detector_bundle["cfg"]\n    device = detector_bundle["device"]\n    torch_mod = detector_bundle["torch"]\n    pool_factor = int(getattr(model_cfg, "pool_factor", 4))\n    volume = read_test_frame(cfg.TEST_DIR, dataset, int(t), frame_cache)\n    pooled = _dc_pool_frame_xy(volume, pool_factor)\n    image = _dc_normalize_dynamic_range(pooled, model_cfg)\n    with torch_mod.no_grad():\n        tensor = torch_mod.from_numpy(image[None, None, ...]).to(device=device, dtype=torch_mod.float32)\n        heatmap = torch_mod.sigmoid(model(tensor))[0, 0].detach().cpu().numpy().astype(np.float32, copy=False)\n    heatmap_cache[key] = heatmap\n    _dc_cache_trim(heatmap_cache, cfg)\n    return heatmap\n\n\ndef deepcenter_score_point(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> float | None:\n    if not cfg.USE_DEEPCENTER_VETO or detector_bundle is None or dataset is None:\n        return None\n    heatmap = deepcenter_heatmap_for_frame(cfg, dataset, int(t), detector_bundle, frame_cache, heatmap_cache)\n    if heatmap is None or heatmap.size == 0:\n        return None\n    model_cfg = detector_bundle["cfg"]\n    pool_factor = int(getattr(model_cfg, "pool_factor", 4))\n    z = int(round(float(point[0])))\n    y = int(round(float(point[1]) / max(pool_factor, 1)))\n    x = int(round(float(point[2]) / max(pool_factor, 1)))\n    z0, z1 = max(0, z - cfg.DEEPCENTER_SCORE_WIN_Z), min(heatmap.shape[0], z + cfg.DEEPCENTER_SCORE_WIN_Z + 1)\n    y0, y1 = max(0, y - cfg.DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[1], y + cfg.DEEPCENTER_SCORE_WIN_YX + 1)\n    x0, x1 = max(0, x - cfg.DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[2], x + cfg.DEEPCENTER_SCORE_WIN_YX + 1)\n    patch = heatmap[z0:z1, y0:y1, x0:x1]\n    if patch.size == 0:\n        return None\n    score = float(np.max(patch))\n    return score if np.isfinite(score) else None\n\n\ndef deepcenter_accept_repair_point(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n    stats: dict[str, int],\n    prefix: str,\n    threshold: float,\n) -> bool:\n    if not cfg.USE_DEEPCENTER_VETO:\n        return True\n    if detector_bundle is None or dataset is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    stats[f"deepcenter_{prefix}_checked"] += 1\n    score = deepcenter_score_point(cfg, dataset, int(t), point, detector_bundle, frame_cache, heatmap_cache)\n    if score is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    if score < float(threshold):\n        stats[f"deepcenter_{prefix}_rejected"] += 1\n        return False\n    stats[f"deepcenter_{prefix}_accepted"] += 1\n    return True\n', 'src/biohub/public_postproc/divisions.py': '"""Division-related post-processing passes, ported verbatim from the notebook cell.\n\nTwo candidate-generation modes share the exact same proposal acceptance loop\n(sorting, frame/global caps, used-target conflicts, output edge fields):\n\n* ``legacy`` -- the pre-E23 port: pure distance gates, no structural\n  constraints. Output is byte-for-byte identical to the Phase-1 behavior.\n* ``e23`` -- the submitted E23 notebook\'s ``add_safe_divisions_postlink``\n  (pub923_repro): all-orphan cKDTree, mid-track parent (C1), mutual nearest\n  orphan (C2), DeepCenter veto before t+2 divergence growth (C3), accepting\n  exactly the notebook\'s candidate set.\n\nBoth modes share the exact notebook proposal-acceptance loop: sort by\n``parent_dist + 0.15 * sister_dist``, frame cap, global cap, and the\nused-target conflict check, appending accepted edges after the originals.\n"""\nfrom __future__ import annotations\n\nimport math\nimport struct\nfrom collections import Counter\nfrom collections.abc import Callable, Iterator, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom numbers import Integral, Real\nfrom typing import Literal\n\nimport numpy as np\nfrom scipy.spatial import cKDTree\n\nfrom biohub.public_postproc.config import PostprocConfig\nfrom biohub.public_postproc.deepcenter import (\n    deepcenter_accept_repair_point,\n    deepcenter_heatmap_for_frame,\n)\nfrom biohub.public_postproc.frames import read_test_frame\nfrom biohub.public_postproc.geometry import VOXEL_SCALE_UM, edge_distance_um\n\n\n@dataclass(frozen=True)\nclass TwinDeepCenterDecision:\n    accepted: bool\n    raw_score: float | None\n    reason: str | None\n\n\n@dataclass(frozen=True)\nclass TwinFrozenMapping(Mapping[object, object]):\n    """Small immutable, insertion-order-preserving mapping snapshot."""\n\n    items_snapshot: tuple[tuple[object, object], ...]\n\n    def __getitem__(self, key: object) -> object:\n        for item_key, value in self.items_snapshot:\n            if item_key == key:\n                return value\n        raise KeyError(key)\n\n    def __iter__(self) -> Iterator[object]:\n        return (key for key, _ in self.items_snapshot)\n\n    def __len__(self) -> int:\n        return len(self.items_snapshot)\n\n\n@dataclass(frozen=True)\nclass TwinFrozenDType:\n    """Immutable dtype description, including structured fields and metadata."""\n\n    string: str\n    descriptor: object\n    metadata: TwinFrozenMapping | None\n    itemsize: int\n    alignment: int\n    byteorder: str\n    names: tuple[str, ...] | None\n    hasobject: bool\n    aligned_struct: bool\n\n\n@dataclass(frozen=True)\nclass TwinFrozenStructuredScalar:\n    fields: tuple[tuple[str, object], ...]\n\n\n@dataclass(frozen=True)\nclass TwinFrozenNumpyScalar:\n    dtype: TwinFrozenDType\n    content: bytes\n\n\n@dataclass(frozen=True)\nclass TwinFrozenArray:\n    """Immutable, lossless semantic snapshot of an ndarray."""\n\n    dtype: TwinFrozenDType\n    shape: tuple[int, ...]\n    strides: tuple[int, ...]\n    c_contiguous: bool\n    f_contiguous: bool\n    content: bytes | tuple[object, ...]\n    object_content: bool\n\n\n@dataclass(frozen=True)\nclass TwinFrozenBuffer:\n    """Immutable snapshot of a bytearray or memoryview."""\n\n    kind: str\n    format: str\n    itemsize: int\n    shape: tuple[int, ...]\n    strides: tuple[int, ...]\n    readonly: bool\n    content: bytes\n\n\n@dataclass(frozen=True)\nclass TwinSnapshotNode:\n    node_id: int\n    t: int\n    z: float\n    y: float\n    x: float\n    gap_synthetic: bool\n\n\n@dataclass(frozen=True)\nclass TwinSnapshotEdge:\n    source_id: int\n    target_id: int\n    input_position: int\n    metadata: TwinFrozenMapping\n\n\n@dataclass(frozen=True)\nclass TwinEdgeRecord:\n    """Public edge snapshot; the internal caller-row ordinal is intentionally absent."""\n\n    source_id: int\n    target_id: int\n    metadata: TwinFrozenMapping\n\n\n@dataclass(frozen=True)\nclass TwinPlannedEdge:\n    source_id: int\n    target_id: int\n    distance_um: float\n    edge_prob: None = None\n\n\ntype TwinScoreCallback = Callable[[TwinSnapshotNode], TwinDeepCenterDecision]\n\n\n@dataclass(frozen=True)\nclass TwinCandidate:\n    frame: int\n    p: int\n    q: int\n    a: int\n    b: int\n    a2: int\n    b2: int\n    d_pq: float\n    d_pa: float\n    d_pb: float\n    d_ab: float\n    d_a2b2: float\n    divergence_growth: float\n    raw_deepcenter_score: float\n    deepcenter_decision: TwinDeepCenterDecision\n    sort_key: tuple[float, float, float, int, int, int, int, int, int]\n    removed_edge: TwinEdgeRecord\n    planned_edge: TwinPlannedEdge\n\n\n@dataclass(frozen=True)\nclass TwinResolutionDecision:\n    candidate: TwinCandidate\n    accepted: bool\n    reason: str | None\n\n\n@dataclass(frozen=True)\nclass TwinDebugRecord:\n    dataset: str | None\n    decision: str\n    reason: str | None\n    p: int\n    q: int\n    a: int\n    b: int\n    a2: int\n    b2: int\n    sort_key: tuple[float, float, float, int, int, int, int, int, int]\n    d_pq: float\n    d_pa: float\n    d_pb: float\n    d_ab: float\n    d_a2b2: float\n    divergence_growth: float\n    raw_deepcenter_score: float\n    deepcenter_threshold: float\n    deepcenter_decision: TwinDeepCenterDecision\n    removed_edge: TwinEdgeRecord\n    planned_edge: TwinPlannedEdge\n\n\n@dataclass(frozen=True)\nclass TwinPlan:\n    validation_reason: str | None\n    nodes: tuple[TwinSnapshotNode, ...]\n    edges: tuple[TwinSnapshotEdge, ...]\n    candidates: tuple[TwinCandidate, ...]\n    accepted_candidates: tuple[TwinCandidate, ...]\n    decisions: tuple[TwinResolutionDecision, ...]\n    counters: TwinFrozenMapping\n    debug_records: tuple[TwinDebugRecord, ...]\n\n\n@dataclass(frozen=True)\nclass TwinMutationSummary:\n    status: Literal["applied", "already_applied", "no_changes"]\n    accepted_count: int\n    nodes_before: int\n    nodes_after: int\n    edges_before: int\n    edges_after: int\n    edges_removed: int\n    edges_added: int\n    edge_symmetric_difference: int\n    isolated_donors: int\n\n\nclass TwinMutationError(RuntimeError):\n    reason: str\n\n    def __init__(self, reason: str) -> None:\n        self.reason = reason\n        super().__init__(reason)\n\n\n_TWIN_COUNTER_KEYS = (\n    "steal_twin_examined_frames",\n    "steal_twin_p_pool",\n    "steal_twin_q_pool",\n    "steal_twin_enumerated",\n    "steal_twin_rejected_distance_twin",\n    "steal_twin_rejected_ambiguous_p_nn",\n    "steal_twin_rejected_ambiguous_q_nn",\n    "steal_twin_rejected_not_mutual_parent_nn",\n    "steal_twin_rejected_distance_existing_child",\n    "steal_twin_rejected_distance_parent",\n    "steal_twin_rejected_distance_sister_low",\n    "steal_twin_rejected_distance_sister_high",\n    "steal_twin_rejected_time",\n    "steal_twin_rejected_missing_successor",\n    "steal_twin_rejected_shared_successor",\n    "steal_twin_rejected_divergence",\n    "steal_twin_rejected_synthetic",\n    "steal_twin_rejected_deepcenter_bundle",\n    "steal_twin_rejected_deepcenter_dataset",\n    "steal_twin_rejected_deepcenter_frame",\n    "steal_twin_rejected_deepcenter_heatmap",\n    "steal_twin_rejected_deepcenter_nonfinite",\n    "steal_twin_rejected_deepcenter_threshold",\n    "steal_twin_eligible",\n    "steal_twin_rejected_conflict",\n    "steal_twin_rejected_frame_cap",\n    "steal_twin_rejected_video_cap",\n    "steal_twin_accepted",\n    "steal_twin_planned_edges_removed",\n    "steal_twin_planned_edges_added",\n    "steal_twin_edges_removed",\n    "steal_twin_edges_added",\n    "steal_twin_isolated_donors",\n    "steal_twin_validation_failed",\n    "steal_twin_validation_missing_node_field",\n    "steal_twin_validation_invalid_node_id",\n    "steal_twin_validation_duplicate_node_id",\n    "steal_twin_validation_invalid_node_time",\n    "steal_twin_validation_nonfinite_node_coordinate",\n    "steal_twin_validation_invalid_edge_endpoint",\n    "steal_twin_validation_dangling_edge",\n    "steal_twin_validation_duplicate_edge",\n    "steal_twin_validation_nonconsecutive_edge",\n    "steal_twin_validation_indegree",\n    "steal_twin_validation_outdegree",\n    "steal_twin_validation_nonfinite_edge_distance",\n    "steal_twin_debug_records_written",\n    "steal_twin_debug_records_dropped",\n)\n\n_TWIN_DEEPCENTER_REASONS = (\n    "deepcenter_bundle",\n    "deepcenter_dataset",\n    "deepcenter_frame",\n    "deepcenter_heatmap",\n    "deepcenter_nonfinite",\n    "deepcenter_threshold",\n)\n\n_TWIN_METADATA_MAX_DEPTH = 64\n\n_TWIN_VALIDATION_REASONS = (\n    "missing_node_field",\n    "invalid_node_id",\n    "duplicate_node_id",\n    "invalid_node_time",\n    "nonfinite_node_coordinate",\n    "invalid_edge_endpoint",\n    "dangling_edge",\n    "duplicate_edge",\n    "nonconsecutive_edge",\n    "indegree",\n    "outdegree",\n    "nonfinite_edge_distance",\n)\n\n\ndef _freeze_twin_dtype(\n    dtype: np.dtype,\n    *,\n    _depth: int = 0,\n    _active: set[int] | None = None,\n) -> TwinFrozenDType:\n    if _depth > _TWIN_METADATA_MAX_DEPTH:\n        raise TypeError("twin metadata exceeds maximum depth")\n    active = set() if _active is None else _active\n    metadata = (\n        _freeze_twin_value(dtype.metadata, _depth=_depth + 1, _active=active)\n        if dtype.metadata is not None\n        else None\n    )\n    if metadata is not None and not isinstance(metadata, TwinFrozenMapping):\n        raise TypeError("numpy dtype metadata must be a mapping")\n    return TwinFrozenDType(\n        dtype.str,\n        _freeze_twin_value(dtype.descr, _depth=_depth + 1, _active=active),\n        metadata,\n        int(dtype.itemsize),\n        int(dtype.alignment),\n        dtype.byteorder,\n        tuple(dtype.names) if dtype.names is not None else None,\n        bool(dtype.hasobject),\n        bool(dtype.isalignedstruct),\n    )\n\n\ndef _freeze_twin_structured_scalar(\n    value: np.void,\n    dtype: np.dtype,\n    *,\n    _depth: int = 0,\n    _active: set[int] | None = None,\n) -> TwinFrozenStructuredScalar:\n    if _depth > _TWIN_METADATA_MAX_DEPTH:\n        raise TypeError("twin metadata exceeds maximum depth")\n    if dtype.names is None:\n        raise TypeError("unstructured numpy void metadata is unsupported")\n    active = set() if _active is None else _active\n    return TwinFrozenStructuredScalar(\n        tuple(\n            (name, _freeze_twin_value(value[name], _depth=_depth + 1, _active=active))\n            for name in dtype.names\n        )\n    )\n\n\ndef _freeze_twin_value(\n    value: object,\n    *,\n    _depth: int = 0,\n    _active: set[int] | None = None,\n) -> object:\n    """Freeze supported edge metadata or reject it before a plan is returned.\n\n    Supported values are standard immutable scalars, recursively frozen\n    mappings/sequences/sets, NumPy dtypes/scalars/arrays, bytearrays, and\n    memoryviews. Arbitrary objects are rejected rather than retained behind a\n    frozen dataclass or ambiguously deep-copied.\n    """\n    if _depth > _TWIN_METADATA_MAX_DEPTH:\n        raise TypeError("twin metadata exceeds maximum depth")\n    active = set() if _active is None else _active\n    recursive = isinstance(\n        value,\n        (np.dtype, np.ndarray, np.void, Mapping, list, tuple, set, frozenset),\n    )\n    value_id = id(value)\n    if recursive:\n        if value_id in active:\n            raise TypeError("cyclic twin metadata")\n        active.add(value_id)\n    try:\n        return _freeze_twin_value_inner(value, _depth, active)\n    finally:\n        if recursive:\n            active.remove(value_id)\n\n\ndef _freeze_twin_value_inner(value: object, depth: int, active: set[int]) -> object:\n    if isinstance(value, np.dtype):\n        return _freeze_twin_dtype(value, _depth=depth, _active=active)\n    if isinstance(value, np.ndarray):\n        original_shape = tuple(value.shape)\n        original_strides = tuple(value.strides)\n        copied_array = np.array(value, copy=True, subok=False, order="K")\n        if value.dtype.hasobject:\n            if value.dtype.names is not None:\n                content: bytes | tuple[object, ...] = tuple(\n                    _freeze_twin_structured_scalar(\n                        copied_array[index], value.dtype, _depth=depth + 1, _active=active\n                    )\n                    for index in np.ndindex(original_shape)\n                )\n            else:\n                content = tuple(\n                    _freeze_twin_value(\n                        copied_array[index], _depth=depth + 1, _active=active\n                    )\n                    for index in np.ndindex(original_shape)\n                )\n        else:\n            content = copied_array.tobytes(order="C")\n        return TwinFrozenArray(\n            _freeze_twin_dtype(value.dtype, _depth=depth + 1, _active=active),\n            original_shape,\n            original_strides,\n            bool(value.flags.c_contiguous),\n            bool(value.flags.f_contiguous),\n            content,\n            bool(value.dtype.hasobject),\n        )\n    if isinstance(value, bytearray):\n        return TwinFrozenBuffer("bytearray", "B", 1, (len(value),), (1,), False, bytes(value))\n    if isinstance(value, memoryview):\n        shape = tuple(value.shape) if value.shape is not None else (value.nbytes // value.itemsize,)\n        strides = tuple(value.strides) if value.strides is not None else ()\n        return TwinFrozenBuffer(\n            "memoryview",\n            value.format,\n            value.itemsize,\n            shape,\n            strides,\n            value.readonly,\n            value.tobytes(),\n        )\n    if isinstance(value, Mapping):\n        return TwinFrozenMapping(\n            tuple(\n                (\n                    _freeze_twin_value(key, _depth=depth + 1, _active=active),\n                    _freeze_twin_value(item, _depth=depth + 1, _active=active),\n                )\n                for key, item in value.items()\n            )\n        )\n    if isinstance(value, (list, tuple)):\n        return tuple(\n            _freeze_twin_value(item, _depth=depth + 1, _active=active) for item in value\n        )\n    if isinstance(value, (set, frozenset)):\n        return frozenset(\n            _freeze_twin_value(item, _depth=depth + 1, _active=active) for item in value\n        )\n    if isinstance(value, np.void):\n        return _freeze_twin_structured_scalar(value, value.dtype, _depth=depth, _active=active)\n    if isinstance(value, np.generic):\n        if value.dtype.hasobject:\n            raise TypeError(f"unsupported numpy object scalar metadata: {value.dtype!r}")\n        return TwinFrozenNumpyScalar(\n            _freeze_twin_dtype(value.dtype, _depth=depth + 1, _active=active),\n            value.tobytes(),\n        )\n    if value is None or type(value) in (bool, int, float, complex, str, bytes):\n        return value\n    raise TypeError(f"unsupported mutable edge metadata type: {type(value).__qualname__}")\n\n\ndef _freeze_twin_edge_row(\n    row: Mapping[str, object],\n    source_id: int,\n    target_id: int,\n    *,\n    _source_key: object | None = None,\n    _target_key: object | None = None,\n) -> TwinFrozenMapping:\n    """Freeze an edge row, canonicalizing its validated integer endpoints.\n\n    Endpoint validation deliberately accepts every non-boolean\n    :class:`numbers.Integral`, while the planner\'s endpoint contract is the\n    canonical integer value used for graph identity.  Converting only the two\n    endpoint fields to built-in ``int`` therefore preserves that contract and\n    detaches mutable ``int`` subclasses without broadening the supported\n    types for arbitrary edge metadata.\n    """\n    active = {id(row)}\n    frozen_items: list[tuple[object, object]] = []\n    try:\n        for key, value in row.items():\n            if _source_key is not None and key is _source_key:\n                frozen_items.append((str(key), source_id))\n            elif _target_key is not None and key is _target_key:\n                frozen_items.append((str(key), target_id))\n            elif _source_key is None and isinstance(key, str) and key == "source_id":\n                frozen_items.append((str(key), source_id))\n            elif _target_key is None and isinstance(key, str) and key == "target_id":\n                frozen_items.append((str(key), target_id))\n            else:\n                frozen_items.append(\n                    (\n                        _freeze_twin_value(key, _depth=1, _active=active),\n                        _freeze_twin_value(value, _depth=1, _active=active),\n                    )\n                )\n    finally:\n        active.remove(id(row))\n    return TwinFrozenMapping(tuple(frozen_items))\n\n\ndef _twin_counters() -> dict[str, int]:\n    return dict.fromkeys(_TWIN_COUNTER_KEYS, 0)\n\n\ndef _frozen_twin_counters(counters: Mapping[str, int]) -> TwinFrozenMapping:\n    return TwinFrozenMapping(tuple((key, int(counters[key])) for key in _TWIN_COUNTER_KEYS))\n\n\ndef _invalid_twin_plan(reason: str) -> TwinPlan:\n    counters = _twin_counters()\n    counters["steal_twin_validation_failed"] = 1\n    counters[f"steal_twin_validation_{reason}"] = 1\n    return TwinPlan(reason, (), (), (), (), (), _frozen_twin_counters(counters), ())\n\n\ndef _is_twin_integer(value: object) -> bool:\n    return not isinstance(value, (bool, np.bool_)) and isinstance(value, Integral)\n\n\ndef _is_twin_finite_real(value: object) -> bool:\n    if isinstance(value, (bool, np.bool_)) or not isinstance(value, Real):\n        return False\n    try:\n        return bool(np.isfinite(value))\n    except (TypeError, ValueError, OverflowError):\n        return False\n\n\ndef _twin_distance(first: TwinSnapshotNode, second: TwinSnapshotNode) -> float:\n    try:\n        with np.errstate(all="ignore"):\n            delta = np.asarray(\n                [\n                    (first.z - second.z) * VOXEL_SCALE_UM[0],\n                    (first.y - second.y) * VOXEL_SCALE_UM[1],\n                    (first.x - second.x) * VOXEL_SCALE_UM[2],\n                ],\n                dtype=np.float64,\n            )\n            distance = float(np.sqrt(np.sum(delta * delta)))\n    except (ArithmeticError, TypeError, ValueError, OverflowError):\n        return float("nan")\n    return distance\n\n\ndef _validate_twin_snapshot(\n    node_rows: Sequence[Mapping[str, object]],\n    edge_rows: Sequence[Mapping[str, object]],\n) -> tuple[str | None, tuple[TwinSnapshotNode, ...], tuple[TwinSnapshotEdge, ...]]:\n    required = ("node_id", "t", "z", "y", "x")\n    if any(not isinstance(row, Mapping) or any(name not in row for name in required) for row in node_rows):\n        return "missing_node_field", (), ()\n    if any(not _is_twin_integer(row["node_id"]) for row in node_rows):\n        return "invalid_node_id", (), ()\n    node_ids = [int(row["node_id"]) for row in node_rows]\n    if len(node_ids) != len(set(node_ids)):\n        return "duplicate_node_id", (), ()\n    if any(not _is_twin_integer(row["t"]) for row in node_rows):\n        return "invalid_node_time", (), ()\n    if any(not all(_is_twin_finite_real(row[name]) for name in ("z", "y", "x")) for row in node_rows):\n        return "nonfinite_node_coordinate", (), ()\n\n    nodes = tuple(\n        sorted(\n            (\n                TwinSnapshotNode(\n                    int(row["node_id"]),\n                    int(row["t"]),\n                    float(row["z"]),\n                    float(row["y"]),\n                    float(row["x"]),\n                    bool(row.get("gap_synthetic", False) == 1),\n                )\n                for row in node_rows\n            ),\n            key=lambda node: node.node_id,\n        )\n    )\n    by_id = {node.node_id: node for node in nodes}\n\n    if any(\n        not isinstance(row, Mapping)\n        or "source_id" not in row\n        or "target_id" not in row\n        or not _is_twin_integer(row["source_id"])\n        or not _is_twin_integer(row["target_id"])\n        for row in edge_rows\n    ):\n        return "invalid_edge_endpoint", (), ()\n    endpoints = [(int(row["source_id"]), int(row["target_id"])) for row in edge_rows]\n    if any(source not in by_id or target not in by_id for source, target in endpoints):\n        return "dangling_edge", (), ()\n    if len(endpoints) != len(set(endpoints)):\n        return "duplicate_edge", (), ()\n    if any(by_id[target].t != by_id[source].t + 1 for source, target in endpoints):\n        return "nonconsecutive_edge", (), ()\n    indegree: dict[int, int] = {}\n    outdegree: dict[int, int] = {}\n    for source, target in endpoints:\n        outdegree[source] = outdegree.get(source, 0) + 1\n        indegree[target] = indegree.get(target, 0) + 1\n    if any(value > 1 for value in indegree.values()):\n        return "indegree", (), ()\n    if any(value > 2 for value in outdegree.values()):\n        return "outdegree", (), ()\n    if any(not np.isfinite(_twin_distance(by_id[source], by_id[target])) for source, target in endpoints):\n        return "nonfinite_edge_distance", (), ()\n\n    edges = tuple(\n        sorted(\n            (\n                TwinSnapshotEdge(\n                    source,\n                    target,\n                    position,\n                    _freeze_twin_edge_row(row, source, target),\n                )\n                for position, (row, (source, target)) in enumerate(\n                    zip(edge_rows, endpoints, strict=True)\n                )\n            ),\n            key=lambda edge: (edge.source_id, edge.target_id, edge.input_position),\n        )\n    )\n    return None, nodes, edges\n\n\ndef _normalize_twin_score(result: object) -> TwinDeepCenterDecision:\n    fallback = TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n    if not isinstance(result, TwinDeepCenterDecision) or type(result.accepted) is not bool:\n        return fallback\n    score = result.raw_score\n    valid_score = _is_twin_finite_real(score) if score is not None else False\n    if result.accepted:\n        if not valid_score or result.reason is not None:\n            return fallback\n        return TwinDeepCenterDecision(True, float(score), None)\n    if result.reason not in _TWIN_DEEPCENTER_REASONS:\n        return fallback\n    if result.reason == "deepcenter_threshold":\n        if not valid_score or float(score) >= 0.12:\n            return fallback\n        return TwinDeepCenterDecision(False, float(score), result.reason)\n    if score is not None:\n        return fallback\n    return TwinDeepCenterDecision(False, None, result.reason)\n\n\ndef plan_twin_only_v1(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    node_rows: Sequence[Mapping[str, object]],\n    edge_rows: Sequence[Mapping[str, object]],\n    score_callback: TwinScoreCallback,\n) -> TwinPlan:\n    """Build the immutable, deterministic ST-R1b twin-only rewire plan."""\n    validation_reason, nodes, edges = _validate_twin_snapshot(node_rows, edge_rows)\n    if validation_reason is not None:\n        return _invalid_twin_plan(validation_reason)\n\n    counters = _twin_counters()\n    by_id = {node.node_id: node for node in nodes}\n    incoming: dict[int, list[TwinSnapshotEdge]] = {}\n    outgoing: dict[int, list[TwinSnapshotEdge]] = {}\n    for edge in edges:\n        outgoing.setdefault(edge.source_id, []).append(edge)\n        incoming.setdefault(edge.target_id, []).append(edge)\n    for values in outgoing.values():\n        values.sort(key=lambda edge: edge.target_id)\n\n    p_pool: dict[int, list[int]] = {}\n    q_pool: dict[int, list[int]] = {}\n    for node in nodes:\n        indegree = len(incoming.get(node.node_id, ()))\n        outdegree = len(outgoing.get(node.node_id, ()))\n        if indegree == 1 and outdegree == 1:\n            p_pool.setdefault(node.t, []).append(node.node_id)\n        if indegree == 0 and outdegree == 1:\n            q_pool.setdefault(node.t, []).append(node.node_id)\n    for pool in (p_pool, q_pool):\n        for values in pool.values():\n            values.sort()\n    frames = sorted(set(p_pool) | set(q_pool))\n    counters["steal_twin_examined_frames"] = len(frames)\n    counters["steal_twin_p_pool"] = sum(map(len, p_pool.values()))\n    counters["steal_twin_q_pool"] = sum(map(len, q_pool.values()))\n\n    eligible: list[TwinCandidate] = []\n\n    def reject(reason: str) -> None:\n        counters[f"steal_twin_rejected_{reason}"] += 1\n\n    for frame in frames:\n        p_ids = p_pool.get(frame, ())\n        q_ids = q_pool.get(frame, ())\n        if not p_ids or not q_ids:\n            continue\n        union_ids = sorted((*p_ids, *q_ids))\n        frame_origin = by_id[union_ids[0]]\n        positions = np.asarray(\n            [\n                (\n                    (by_id[node_id].z - frame_origin.z) * VOXEL_SCALE_UM[0],\n                    (by_id[node_id].y - frame_origin.y) * VOXEL_SCALE_UM[1],\n                    (by_id[node_id].x - frame_origin.x) * VOXEL_SCALE_UM[2],\n                )\n                for node_id in union_ids\n            ],\n            dtype=np.float64,\n        )\n        union_index = {node_id: index for index, node_id in enumerate(union_ids)}\n        tree_coordinate_scale = max(cfg.STEAL_TWIN_TWIN_MAX_UM, 1.0, *np.abs(positions).flat)\n        # The tree sees translated/scaled coordinates and performs more float\n        # operations internally. Query a conservative roundoff superset, then\n        # enforce the exact frozen radius using _twin_distance below.\n        query_roundoff = 32.0 * np.finfo(np.float64).eps * np.sqrt(3.0) * tree_coordinate_scale\n        query_radius = float(\n            np.nextafter(cfg.STEAL_TWIN_TWIN_MAX_UM + query_roundoff, np.inf)\n        )\n        if np.all(np.isfinite(positions)) and np.isfinite(query_radius):\n            try:\n                tree = cKDTree(positions)\n            except Exception:\n                tree = None\n        else:\n            tree = None\n        role_p = set(p_ids)\n        role_q = set(q_ids)\n\n        def neighbors(\n            node_id: int,\n            opposite: set[int],\n            frame_tree=tree,\n            frame_positions=positions,\n            frame_union_ids=tuple(union_ids),\n            frame_union_index=union_index,\n            frame_query_radius=query_radius,\n        ) -> list[tuple[float, int]] | None:\n            if frame_tree is None:\n                return None\n            point = frame_positions[frame_union_index[node_id]]\n            try:\n                indices = frame_tree.query_ball_point(point, r=frame_query_radius)\n                result = []\n                for index in indices:\n                    other_id = frame_union_ids[int(index)]\n                    if other_id not in opposite:\n                        continue\n                    distance = _twin_distance(by_id[node_id], by_id[other_id])\n                    if not np.isfinite(distance):\n                        return None\n                    if distance <= cfg.STEAL_TWIN_TWIN_MAX_UM:\n                        result.append((distance, other_id))\n            except Exception:\n                return None\n            result.sort(key=lambda item: (item[0], item[1]))\n            return result\n\n        for p_id in p_ids:\n            counters["steal_twin_enumerated"] += 1\n            p_neighbors = neighbors(p_id, role_q)\n            if not p_neighbors:\n                reject("distance_twin")\n                continue\n            minimum = p_neighbors[0][0]\n            if sum(distance - minimum <= 1e-9 for distance, _ in p_neighbors) != 1:\n                reject("ambiguous_p_nn")\n                continue\n            d_pq, q_id = p_neighbors[0]\n            q_neighbors = neighbors(q_id, role_p)\n            if not q_neighbors:\n                reject("distance_twin")\n                continue\n            q_minimum = q_neighbors[0][0]\n            if sum(distance - q_minimum <= 1e-9 for distance, _ in q_neighbors) != 1:\n                reject("ambiguous_q_nn")\n                continue\n            if q_neighbors[0][1] != p_id:\n                reject("not_mutual_parent_nn")\n                continue\n\n            a_id = outgoing[p_id][0].target_id\n            a = by_id[a_id]\n            d_pa = _twin_distance(by_id[p_id], a)\n            if not np.isfinite(d_pa) or d_pa > cfg.STEAL_TWIN_EXISTING_CHILD_MAX_UM:\n                reject("distance_existing_child")\n                continue\n            b_edge = outgoing[q_id][0]\n            b_id = b_edge.target_id\n            b = by_id[b_id]\n            d_pb = _twin_distance(by_id[p_id], b)\n            if not np.isfinite(d_pb) or d_pb > cfg.STEAL_TWIN_PARENT_MAX_UM:\n                reject("distance_parent")\n                continue\n            d_ab = _twin_distance(a, b)\n            if np.isfinite(d_ab) and d_ab < cfg.STEAL_TWIN_SISTER_MIN_UM:\n                reject("distance_sister_low")\n                continue\n            if not np.isfinite(d_ab) or d_ab > cfg.STEAL_TWIN_SISTER_MAX_UM:\n                reject("distance_sister_high")\n                continue\n            if len(outgoing.get(a_id, ())) != 1 or len(outgoing.get(b_id, ())) != 1:\n                reject("missing_successor")\n                continue\n            a2_id = outgoing[a_id][0].target_id\n            b2_id = outgoing[b_id][0].target_id\n            if a2_id == b2_id:\n                reject("shared_successor")\n                continue\n            d_a2b2 = _twin_distance(by_id[a2_id], by_id[b2_id])\n            with np.errstate(all="ignore"):\n                growth = d_a2b2 - d_ab\n            if not np.isfinite(d_a2b2) or not np.isfinite(growth) or growth < cfg.STEAL_TWIN_DIVERGE_UM:\n                reject("divergence")\n                continue\n            role_ids = (p_id, q_id, a_id, b_id, a2_id, b2_id)\n            if any(by_id[node_id].gap_synthetic for node_id in role_ids):\n                reject("synthetic")\n                continue\n            try:\n                deepcenter = _normalize_twin_score(score_callback(b))\n            except Exception:\n                deepcenter = TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n            if not deepcenter.accepted:\n                reject(deepcenter.reason or "deepcenter_bundle")\n                continue\n            sort_key = (\n                d_pb + 0.15 * d_ab,\n                -growth,\n                d_pq,\n                p_id,\n                q_id,\n                a_id,\n                b_id,\n                a2_id,\n                b2_id,\n            )\n            if not all(np.isfinite(value) for value in sort_key[:3]):\n                reject("divergence")\n                continue\n            candidate = TwinCandidate(\n                frame,\n                p_id,\n                q_id,\n                a_id,\n                b_id,\n                a2_id,\n                b2_id,\n                d_pq,\n                d_pa,\n                d_pb,\n                d_ab,\n                d_a2b2,\n                float(growth),\n                deepcenter.raw_score,\n                deepcenter,\n                sort_key,\n                TwinEdgeRecord(b_edge.source_id, b_edge.target_id, b_edge.metadata),\n                TwinPlannedEdge(p_id, b_id, d_pb),\n            )\n            eligible.append(candidate)\n            counters["steal_twin_eligible"] += 1\n\n    eligible.sort(key=lambda candidate: candidate.sort_key)\n    decisions: list[TwinResolutionDecision] = []\n    accepted: list[TwinCandidate] = []\n    debug_records: list[TwinDebugRecord] = []\n    used_ids: set[int] = set()\n    accepted_by_frame: dict[int, int] = {}\n    for candidate in eligible:\n        candidate_ids = {candidate.p, candidate.q, candidate.a, candidate.b, candidate.a2, candidate.b2}\n        if candidate_ids & used_ids:\n            reason = "conflict"\n        elif accepted_by_frame.get(candidate.frame, 0) >= cfg.STEAL_TWIN_FRAME_CAP_ABS:\n            reason = "frame_cap"\n        elif len(accepted) >= cfg.STEAL_TWIN_VIDEO_CAP_ABS:\n            reason = "video_cap"\n        else:\n            reason = None\n            accepted.append(candidate)\n            used_ids.update(candidate_ids)\n            accepted_by_frame[candidate.frame] = accepted_by_frame.get(candidate.frame, 0) + 1\n        is_accepted = reason is None\n        decisions.append(TwinResolutionDecision(candidate, is_accepted, reason))\n        if is_accepted:\n            counters["steal_twin_accepted"] += 1\n        else:\n            counters[f"steal_twin_rejected_{reason}"] += 1\n        debug_records.append(\n            TwinDebugRecord(\n                dataset,\n                "accepted" if is_accepted else "rejected",\n                reason,\n                candidate.p,\n                candidate.q,\n                candidate.a,\n                candidate.b,\n                candidate.a2,\n                candidate.b2,\n                candidate.sort_key,\n                candidate.d_pq,\n                candidate.d_pa,\n                candidate.d_pb,\n                candidate.d_ab,\n                candidate.d_a2b2,\n                candidate.divergence_growth,\n                candidate.raw_deepcenter_score,\n                0.12,\n                candidate.deepcenter_decision,\n                candidate.removed_edge,\n                candidate.planned_edge,\n            )\n        )\n    accepted_count = len(accepted)\n    counters["steal_twin_planned_edges_removed"] = accepted_count\n    counters["steal_twin_planned_edges_added"] = accepted_count\n    counters["steal_twin_isolated_donors"] = accepted_count\n    return TwinPlan(\n        None,\n        nodes,\n        edges,\n        tuple(eligible),\n        tuple(accepted),\n        tuple(decisions),\n        _frozen_twin_counters(counters),\n        tuple(debug_records),\n    )\n\n\nclass _TwinMutationFault(Exception):\n    def __init__(self, reason: str) -> None:\n        self.reason = reason\n\n\ndef _twin_exact_tuple(value: object, *, length: int | None = None) -> bool:\n    return type(value) is tuple and (length is None or len(value) == length)\n\n\ndef _twin_shape_product(shape: tuple[int, ...]) -> int:\n    result = 1\n    for size in shape:\n        result *= size\n    return result\n\n\ndef _twin_frozen_token(value: object) -> tuple[object, ...]:\n    """Return a bit-exact, recursively hashable token for an R1 frozen value."""\n    try:\n        return _twin_frozen_token_inner(value, 0, set())\n    except TypeError:\n        raise\n    except Exception as exc:\n        raise TypeError("invalid frozen twin value") from exc\n\n\ndef _twin_frozen_token_inner(\n    value: object,\n    depth: int,\n    active: set[int],\n) -> tuple[object, ...]:\n    if depth > _TWIN_METADATA_MAX_DEPTH:\n        raise TypeError("twin metadata exceeds maximum depth")\n    value_type = type(value)\n    if value is None:\n        return ("none",)\n    if value_type is bool:\n        return ("bool", value)\n    if value_type is int:\n        return ("int", value)\n    if value_type is float:\n        return ("float", struct.pack(">d", value))\n    if value_type is complex:\n        return ("complex", struct.pack(">d", value.real), struct.pack(">d", value.imag))\n    if value_type is str:\n        return ("str", value)\n    if value_type is bytes:\n        return ("bytes", value)\n\n    recursive_types = (\n        tuple,\n        frozenset,\n        TwinFrozenMapping,\n        TwinFrozenDType,\n        TwinFrozenStructuredScalar,\n        TwinFrozenNumpyScalar,\n        TwinFrozenArray,\n        TwinFrozenBuffer,\n    )\n    if value_type not in recursive_types:\n        raise TypeError("unsupported frozen twin value")\n    value_id = id(value)\n    if value_id in active:\n        raise TypeError("cyclic frozen twin value")\n    active.add(value_id)\n    try:\n        if value_type is tuple:\n            return (\n                "tuple",\n                tuple(_twin_frozen_token_inner(item, depth + 1, active) for item in value),\n            )\n        if value_type is frozenset:\n            counts = Counter(\n                _twin_frozen_token_inner(item, depth + 1, active) for item in value\n            )\n            return ("frozenset", frozenset((token, count) for token, count in counts.items()))\n        if value_type is TwinFrozenMapping:\n            if not _twin_exact_tuple(value.items_snapshot):\n                raise TypeError("invalid frozen mapping items")\n            items: list[tuple[tuple[object, ...], tuple[object, ...]]] = []\n            for pair in value.items_snapshot:\n                if not _twin_exact_tuple(pair, length=2):\n                    raise TypeError("invalid frozen mapping item")\n                items.append(\n                    (\n                        _twin_frozen_token_inner(pair[0], depth + 1, active),\n                        _twin_frozen_token_inner(pair[1], depth + 1, active),\n                    )\n                )\n            return ("mapping", tuple(items))\n        if value_type is TwinFrozenDType:\n            if (\n                type(value.string) is not str\n                or type(value.itemsize) is not int\n                or value.itemsize < 0\n                or type(value.alignment) is not int\n                or value.alignment < 0\n                or type(value.byteorder) is not str\n                or type(value.hasobject) is not bool\n                or type(value.aligned_struct) is not bool\n                or (value.metadata is not None and type(value.metadata) is not TwinFrozenMapping)\n            ):\n                raise TypeError("invalid frozen dtype fields")\n            if value.names is not None:\n                if (\n                    not _twin_exact_tuple(value.names)\n                    or any(type(name) is not str for name in value.names)\n                    or len(set(value.names)) != len(value.names)\n                ):\n                    raise TypeError("invalid frozen dtype names")\n            return (\n                "dtype",\n                value.string,\n                _twin_frozen_token_inner(value.descriptor, depth + 1, active),\n                None\n                if value.metadata is None\n                else _twin_frozen_token_inner(value.metadata, depth + 1, active),\n                value.itemsize,\n                value.alignment,\n                value.byteorder,\n                value.names,\n                value.hasobject,\n                value.aligned_struct,\n            )\n        if value_type is TwinFrozenStructuredScalar:\n            if not _twin_exact_tuple(value.fields):\n                raise TypeError("invalid structured scalar fields")\n            names: list[str] = []\n            fields: list[tuple[str, tuple[object, ...]]] = []\n            for pair in value.fields:\n                if not _twin_exact_tuple(pair, length=2) or type(pair[0]) is not str:\n                    raise TypeError("invalid structured scalar field")\n                names.append(pair[0])\n                fields.append(\n                    (pair[0], _twin_frozen_token_inner(pair[1], depth + 1, active))\n                )\n            if len(set(names)) != len(names):\n                raise TypeError("duplicate structured scalar field")\n            return ("structured_scalar", tuple(fields))\n        if value_type is TwinFrozenNumpyScalar:\n            if type(value.dtype) is not TwinFrozenDType or type(value.content) is not bytes:\n                raise TypeError("invalid frozen numpy scalar")\n            dtype_token = _twin_frozen_token_inner(value.dtype, depth + 1, active)\n            if value.dtype.hasobject or len(value.content) != value.dtype.itemsize:\n                raise TypeError("invalid frozen numpy scalar length")\n            return ("numpy_scalar", dtype_token, value.content)\n        if value_type is TwinFrozenArray:\n            if (\n                type(value.dtype) is not TwinFrozenDType\n                or not _twin_exact_tuple(value.shape)\n                or not _twin_exact_tuple(value.strides)\n                or any(type(size) is not int or size < 0 for size in value.shape)\n                or any(type(stride) is not int for stride in value.strides)\n                or len(value.shape) != len(value.strides)\n                or type(value.c_contiguous) is not bool\n                or type(value.f_contiguous) is not bool\n                or type(value.object_content) is not bool\n                or value.object_content is not value.dtype.hasobject\n            ):\n                raise TypeError("invalid frozen array fields")\n            dtype_token = _twin_frozen_token_inner(value.dtype, depth + 1, active)\n            count = _twin_shape_product(value.shape)\n            if value.object_content:\n                if not _twin_exact_tuple(value.content) or len(value.content) != count:\n                    raise TypeError("invalid frozen object array content")\n                content_token: object = tuple(\n                    _twin_frozen_token_inner(item, depth + 1, active)\n                    for item in value.content\n                )\n            else:\n                if type(value.content) is not bytes or len(value.content) != count * value.dtype.itemsize:\n                    raise TypeError("invalid frozen array byte content")\n                content_token = value.content\n            return (\n                "array",\n                dtype_token,\n                value.shape,\n                value.strides,\n                value.c_contiguous,\n                value.f_contiguous,\n                content_token,\n                value.object_content,\n            )\n        if (\n            type(value.kind) is not str\n            or value.kind not in ("bytearray", "memoryview")\n            or type(value.format) is not str\n            or type(value.itemsize) is not int\n            or value.itemsize <= 0\n            or not _twin_exact_tuple(value.shape)\n            or not _twin_exact_tuple(value.strides)\n            or any(type(size) is not int or size < 0 for size in value.shape)\n            or any(type(stride) is not int for stride in value.strides)\n            or (value.strides and len(value.strides) != len(value.shape))\n            or type(value.readonly) is not bool\n            or type(value.content) is not bytes\n            or len(value.content) != _twin_shape_product(value.shape) * value.itemsize\n        ):\n            raise TypeError("invalid frozen buffer fields")\n        if value.kind == "bytearray" and (\n            value.format != "B"\n            or value.itemsize != 1\n            or value.shape != (len(value.content),)\n            or value.strides != (1,)\n            or value.readonly is not False\n        ):\n            raise TypeError("invalid frozen bytearray")\n        return (\n            "buffer",\n            value.kind,\n            value.format,\n            value.itemsize,\n            value.shape,\n            value.strides,\n            value.readonly,\n            value.content,\n        )\n    finally:\n        active.remove(value_id)\n\n\ndef _twin_float_token(value: float) -> bytes:\n    return struct.pack(">d", value)\n\n\ndef _twin_plan_fault(reason: str) -> None:\n    raise _TwinMutationFault(reason)\n\n\ndef _twin_edge_adjacency(\n    edge_pairs: set[tuple[int, int]],\n) -> tuple[dict[int, set[int]], dict[int, set[int]]]:\n    outgoing_targets: dict[int, set[int]] = {}\n    incoming_sources: dict[int, set[int]] = {}\n    for source_id, target_id in edge_pairs:\n        outgoing_targets.setdefault(source_id, set()).add(target_id)\n        incoming_sources.setdefault(target_id, set()).add(source_id)\n    return outgoing_targets, incoming_sources\n\n\ndef _validate_twin_plan_for_mutation(\n    plan: object,\n) -> tuple[\n    TwinPlan,\n    dict[str, int],\n    dict[int, TwinSnapshotNode],\n    list[tuple[TwinSnapshotEdge, tuple[object, ...]]],\n    tuple[tuple[object, ...], ...],\n]:\n    if type(plan) is not TwinPlan:\n        _twin_plan_fault("invalid_plan_type")\n    try:\n        validation_reason = plan.validation_reason\n    except Exception:\n        _twin_plan_fault("plan_acceptance")\n    if validation_reason is not None:\n        _twin_plan_fault("plan_validation_failed")\n    try:\n        plan_nodes = plan.nodes\n        plan_edges = plan.edges\n        plan_candidates = plan.candidates\n        plan_accepted = plan.accepted_candidates\n        plan_decisions = plan.decisions\n        plan_debug_records = plan.debug_records\n        plan_counters = plan.counters\n    except Exception:\n        _twin_plan_fault("plan_acceptance")\n    tuple_fields = (\n        plan_nodes,\n        plan_edges,\n        plan_candidates,\n        plan_accepted,\n        plan_decisions,\n        plan_debug_records,\n    )\n    if any(type(value) is not tuple for value in tuple_fields) or type(plan_counters) is not TwinFrozenMapping:\n        _twin_plan_fault("plan_acceptance")\n\n    try:\n        items = plan_counters.items_snapshot\n    except Exception:\n        _twin_plan_fault("plan_counter_schema")\n    if type(items) is not tuple or len(items) != len(_TWIN_COUNTER_KEYS):\n        _twin_plan_fault("plan_counter_schema")\n    counter_values: list[int] = []\n    for index, entry in enumerate(items):\n        if (\n            type(entry) is not tuple\n            or len(entry) != 2\n            or type(entry[0]) is not str\n            or entry[0] != _TWIN_COUNTER_KEYS[index]\n        ):\n            _twin_plan_fault("plan_counter_schema")\n        if type(entry[1]) is not int or entry[1] < 0:\n            _twin_plan_fault("plan_counter_value")\n        counter_values.append(entry[1])\n    counters = dict(zip(_TWIN_COUNTER_KEYS, counter_values, strict=True))\n    eligibility_keys = (\n        "distance_twin",\n        "ambiguous_p_nn",\n        "ambiguous_q_nn",\n        "not_mutual_parent_nn",\n        "distance_existing_child",\n        "distance_parent",\n        "distance_sister_low",\n        "distance_sister_high",\n        "time",\n        "missing_successor",\n        "shared_successor",\n        "divergence",\n        "synthetic",\n        "deepcenter_bundle",\n        "deepcenter_dataset",\n        "deepcenter_frame",\n        "deepcenter_heatmap",\n        "deepcenter_nonfinite",\n        "deepcenter_threshold",\n    )\n    accepted_count = counters["steal_twin_accepted"]\n    conservation = (\n        counters["steal_twin_enumerated"]\n        == sum(counters[f"steal_twin_rejected_{key}"] for key in eligibility_keys)\n        + counters["steal_twin_eligible"]\n        and counters["steal_twin_eligible"]\n        == accepted_count\n        + counters["steal_twin_rejected_conflict"]\n        + counters["steal_twin_rejected_frame_cap"]\n        + counters["steal_twin_rejected_video_cap"]\n        and counters["steal_twin_planned_edges_removed"] == accepted_count\n        and counters["steal_twin_planned_edges_added"] == accepted_count\n        and counters["steal_twin_edges_removed"] == 0\n        and counters["steal_twin_edges_added"] == 0\n        and counters["steal_twin_isolated_donors"] == accepted_count\n        and counters["steal_twin_debug_records_written"] == 0\n        and counters["steal_twin_debug_records_dropped"] == 0\n        and len(plan_candidates) == counters["steal_twin_eligible"]\n        and len(plan_accepted) == accepted_count\n        and accepted_count <= counters["steal_twin_examined_frames"]\n        and accepted_count <= 2\n    )\n    if not conservation:\n        _twin_plan_fault("plan_conservation")\n\n    by_id: dict[int, TwinSnapshotNode] = {}\n    previous_id: int | None = None\n    for node in plan_nodes:\n        if (\n            type(node) is not TwinSnapshotNode\n            or type(node.node_id) is not int\n            or type(node.t) is not int\n            or any(type(value) is not float or not math.isfinite(value) for value in (node.z, node.y, node.x))\n            or type(node.gap_synthetic) is not bool\n            or (previous_id is not None and node.node_id <= previous_id)\n        ):\n            _twin_plan_fault("plan_candidate")\n        previous_id = node.node_id\n        by_id[node.node_id] = node\n\n    edge_pairs: set[tuple[int, int]] = set()\n    positions: set[int] = set()\n    indegree: Counter[int] = Counter()\n    outdegree: Counter[int] = Counter()\n    edge_metadata_tokens: dict[tuple[int, int], tuple[object, ...]] = {}\n    previous_edge_key: tuple[int, int, int] | None = None\n    for edge in plan_edges:\n        if (\n            type(edge) is not TwinSnapshotEdge\n            or type(edge.source_id) is not int\n            or type(edge.target_id) is not int\n            or type(edge.input_position) is not int\n            or type(edge.metadata) is not TwinFrozenMapping\n        ):\n            _twin_plan_fault("plan_candidate")\n        try:\n            metadata_token = _twin_frozen_token(edge.metadata)\n        except Exception:\n            _twin_plan_fault("plan_candidate")\n        metadata_sources = [\n            item\n            for key, item in edge.metadata.items_snapshot\n            if type(key) is str and key == "source_id"\n        ]\n        metadata_targets = [\n            item\n            for key, item in edge.metadata.items_snapshot\n            if type(key) is str and key == "target_id"\n        ]\n        edge_key = (edge.source_id, edge.target_id, edge.input_position)\n        pair = edge_key[:2]\n        if (\n            (previous_edge_key is not None and edge_key <= previous_edge_key)\n            or pair in edge_pairs\n            or edge.input_position in positions\n            or edge.source_id not in by_id\n            or edge.target_id not in by_id\n            or by_id[edge.target_id].t != by_id[edge.source_id].t + 1\n            or len(metadata_sources) != 1\n            or type(metadata_sources[0]) is not int\n            or metadata_sources[0] != edge.source_id\n            or len(metadata_targets) != 1\n            or type(metadata_targets[0]) is not int\n            or metadata_targets[0] != edge.target_id\n        ):\n            _twin_plan_fault("plan_candidate")\n        previous_edge_key = edge_key\n        edge_pairs.add(pair)\n        positions.add(edge.input_position)\n        edge_metadata_tokens[pair] = metadata_token\n        outdegree[edge.source_id] += 1\n        indegree[edge.target_id] += 1\n    if positions != set(range(len(plan_edges))) or any(v > 1 for v in indegree.values()) or any(v > 2 for v in outdegree.values()):\n        _twin_plan_fault("plan_candidate")\n    outgoing_targets, incoming_sources = _twin_edge_adjacency(edge_pairs)\n\n    if len(plan_decisions) != len(plan_candidates):\n        _twin_plan_fault("plan_acceptance")\n    accepted_from_decisions: list[TwinCandidate] = []\n    rejected_counts = Counter()\n    for candidate, decision in zip(plan_candidates, plan_decisions, strict=True):\n        if type(candidate) is not TwinCandidate or type(decision) is not TwinResolutionDecision or decision.candidate is not candidate:\n            _twin_plan_fault("plan_acceptance")\n        if decision.accepted is True and decision.reason is None:\n            accepted_from_decisions.append(candidate)\n        elif decision.accepted is False and type(decision.reason) is str and decision.reason in ("conflict", "frame_cap", "video_cap"):\n            rejected_counts[decision.reason] += 1\n        else:\n            _twin_plan_fault("plan_acceptance")\n    if (\n        len(accepted_from_decisions) != accepted_count\n        or len(accepted_from_decisions) != len(plan_accepted)\n        or any(\n            left is not right\n            for left, right in zip(accepted_from_decisions, plan_accepted, strict=True)\n        )\n        or any(\n            rejected_counts[key] != counters[f"steal_twin_rejected_{key}"]\n            for key in ("conflict", "frame_cap", "video_cap")\n        )\n    ):\n        _twin_plan_fault("plan_acceptance")\n\n    candidate_metadata_tokens = tuple(\n        _validate_twin_candidate_structure(candidate) for candidate in plan_candidates\n    )\n    accepted_roles: set[int] = set()\n    accepted_removed: set[tuple[int, int]] = set()\n    accepted_planned: set[tuple[int, int]] = set()\n    previous_sort_key: tuple[float, float, float, int, int, int, int, int, int] | None = None\n    accepted_frames: set[int] = set()\n    accepted_metadata_tokens: list[tuple[object, ...]] = []\n    for candidate, candidate_metadata_token, decision in zip(\n        plan_candidates, candidate_metadata_tokens, plan_decisions, strict=True\n    ):\n        try:\n            roles = (candidate.p, candidate.q, candidate.a, candidate.b, candidate.a2, candidate.b2)\n            if len(set(roles)) != 6 or any(role not in by_id for role in roles):\n                _twin_plan_fault("plan_candidate")\n            if (\n                by_id[candidate.p].t != candidate.frame\n                or by_id[candidate.q].t != candidate.frame\n                or by_id[candidate.a].t != candidate.frame + 1\n                or by_id[candidate.b].t != candidate.frame + 1\n                or by_id[candidate.a2].t != candidate.frame + 2\n                or by_id[candidate.b2].t != candidate.frame + 2\n                or any(by_id[role].gap_synthetic for role in roles)\n            ):\n                _twin_plan_fault("plan_candidate")\n            distances = (\n                _twin_distance(by_id[candidate.p], by_id[candidate.q]),\n                _twin_distance(by_id[candidate.p], by_id[candidate.a]),\n                _twin_distance(by_id[candidate.p], by_id[candidate.b]),\n                _twin_distance(by_id[candidate.a], by_id[candidate.b]),\n                _twin_distance(by_id[candidate.a2], by_id[candidate.b2]),\n            )\n            stored = (candidate.d_pq, candidate.d_pa, candidate.d_pb, candidate.d_ab, candidate.d_a2b2)\n            if any(not math.isfinite(value) for value in distances) or any(\n                _twin_float_token(left) != _twin_float_token(right)\n                for left, right in zip(distances, stored, strict=True)\n            ):\n                _twin_plan_fault("plan_candidate")\n            growth = distances[4] - distances[3]\n            formula = (\n                candidate.d_pb + 0.15 * candidate.d_ab,\n                -candidate.divergence_growth,\n                candidate.d_pq,\n                *roles,\n            )\n            if (\n                _twin_float_token(growth) != _twin_float_token(candidate.divergence_growth)\n                or any(\n                    _twin_float_token(formula[i]) != _twin_float_token(candidate.sort_key[i])\n                    for i in range(3)\n                )\n                or formula[3:] != candidate.sort_key[3:]\n                or _twin_float_token(distances[2]) != _twin_float_token(candidate.planned_edge.distance_um)\n            ):\n                _twin_plan_fault("plan_candidate")\n            removed_token = edge_metadata_tokens.get((candidate.q, candidate.b))\n            if removed_token is None or removed_token != candidate_metadata_token:\n                _twin_plan_fault("plan_candidate")\n            required = ((candidate.p, candidate.a), (candidate.q, candidate.b), (candidate.a, candidate.a2), (candidate.b, candidate.b2))\n            if any(pair not in edge_pairs for pair in required) or (candidate.p, candidate.b) in edge_pairs:\n                _twin_plan_fault("plan_candidate")\n            if (\n                outgoing_targets.get(candidate.p, set()) != {candidate.a}\n                or outgoing_targets.get(candidate.q, set()) != {candidate.b}\n                or outgoing_targets.get(candidate.a, set()) != {candidate.a2}\n                or outgoing_targets.get(candidate.b, set()) != {candidate.b2}\n                or incoming_sources.get(candidate.q, set())\n            ):\n                _twin_plan_fault("plan_candidate")\n            if previous_sort_key is not None and candidate.sort_key < previous_sort_key:\n                _twin_plan_fault("plan_acceptance")\n            previous_sort_key = candidate.sort_key\n            if decision.accepted is True:\n                role_set = set(roles)\n                removed_pair = (candidate.q, candidate.b)\n                planned_pair = (candidate.p, candidate.b)\n                if role_set & accepted_roles or removed_pair in accepted_removed or planned_pair in accepted_planned:\n                    _twin_plan_fault("plan_candidate")\n                accepted_roles.update(role_set)\n                accepted_removed.add(removed_pair)\n                accepted_planned.add(planned_pair)\n                accepted_frames.add(candidate.frame)\n                accepted_metadata_tokens.append(candidate_metadata_token)\n        except _TwinMutationFault:\n            raise\n        except Exception:\n            _twin_plan_fault("plan_candidate")\n    if len(accepted_frames) != accepted_count:\n        _twin_plan_fault("plan_conservation")\n\n    if len(plan_debug_records) != len(plan_candidates):\n        _twin_plan_fault("plan_acceptance")\n    dataset_value: str | None = None\n    for index, (record, decision, candidate) in enumerate(\n        zip(plan_debug_records, plan_decisions, plan_candidates, strict=True)\n    ):\n        try:\n            if type(record) is not TwinDebugRecord or (record.dataset is not None and type(record.dataset) is not str):\n                _twin_plan_fault("plan_acceptance")\n            if index == 0:\n                dataset_value = record.dataset\n            elif record.dataset != dataset_value:\n                _twin_plan_fault("plan_acceptance")\n            expected_decision = "accepted" if decision.accepted else "rejected"\n            if (\n                type(record.decision) is not str\n                or record.decision != expected_decision\n                or (record.reason is not None and type(record.reason) is not str)\n                or record.reason != decision.reason\n            ):\n                _twin_plan_fault("plan_acceptance")\n            projected = (\n                (record.p, candidate.p), (record.q, candidate.q), (record.a, candidate.a),\n                (record.b, candidate.b), (record.a2, candidate.a2), (record.b2, candidate.b2),\n                (record.sort_key, candidate.sort_key), (record.d_pq, candidate.d_pq),\n                (record.d_pa, candidate.d_pa), (record.d_pb, candidate.d_pb),\n                (record.d_ab, candidate.d_ab), (record.d_a2b2, candidate.d_a2b2),\n                (record.divergence_growth, candidate.divergence_growth),\n                (record.raw_deepcenter_score, candidate.raw_deepcenter_score),\n                (record.deepcenter_decision, candidate.deepcenter_decision),\n                (record.removed_edge, candidate.removed_edge), (record.planned_edge, candidate.planned_edge),\n            )\n            if any(left is not right for left, right in projected):\n                _twin_plan_fault("plan_acceptance")\n            if type(record.deepcenter_threshold) is not float or _twin_float_token(record.deepcenter_threshold) != _twin_float_token(0.12):\n                _twin_plan_fault("plan_acceptance")\n        except _TwinMutationFault:\n            raise\n        except Exception:\n            _twin_plan_fault("plan_acceptance")\n    ordered_edges = sorted(\n        (\n            (edge, edge_metadata_tokens[(edge.source_id, edge.target_id)])\n            for edge in plan_edges\n        ),\n        key=lambda item: item[0].input_position,\n    )\n    return plan, counters, by_id, ordered_edges, tuple(accepted_metadata_tokens)\n\n\ndef _validate_twin_candidate_structure(candidate: TwinCandidate) -> tuple[object, ...]:\n    roles = (candidate.frame, candidate.p, candidate.q, candidate.a, candidate.b, candidate.a2, candidate.b2)\n    floats = (\n        candidate.d_pq, candidate.d_pa, candidate.d_pb, candidate.d_ab,\n        candidate.d_a2b2, candidate.divergence_growth, candidate.raw_deepcenter_score,\n    )\n    if any(type(value) is not int for value in roles) or any(type(value) is not float or not math.isfinite(value) for value in floats):\n        _twin_plan_fault("plan_candidate")\n    if (\n        type(candidate.deepcenter_decision) is not TwinDeepCenterDecision\n        or candidate.deepcenter_decision.accepted is not True\n        or candidate.deepcenter_decision.reason is not None\n        or type(candidate.deepcenter_decision.raw_score) is not float\n        or not math.isfinite(candidate.deepcenter_decision.raw_score)\n        or _twin_float_token(candidate.deepcenter_decision.raw_score) != _twin_float_token(candidate.raw_deepcenter_score)\n        or type(candidate.sort_key) is not tuple\n        or len(candidate.sort_key) != 9\n        or any(type(value) is not float or not math.isfinite(value) for value in candidate.sort_key[:3])\n        or any(type(value) is not int for value in candidate.sort_key[3:])\n        or type(candidate.removed_edge) is not TwinEdgeRecord\n        or type(candidate.removed_edge.source_id) is not int\n        or type(candidate.removed_edge.target_id) is not int\n        or type(candidate.removed_edge.metadata) is not TwinFrozenMapping\n        or (candidate.removed_edge.source_id, candidate.removed_edge.target_id) != (candidate.q, candidate.b)\n        or type(candidate.planned_edge) is not TwinPlannedEdge\n        or type(candidate.planned_edge.source_id) is not int\n        or type(candidate.planned_edge.target_id) is not int\n        or type(candidate.planned_edge.distance_um) is not float\n        or not math.isfinite(candidate.planned_edge.distance_um)\n        or candidate.planned_edge.edge_prob is not None\n        or (candidate.planned_edge.source_id, candidate.planned_edge.target_id) != (candidate.p, candidate.b)\n    ):\n        _twin_plan_fault("plan_candidate")\n    try:\n        return _twin_frozen_token(candidate.removed_edge.metadata)\n    except Exception:\n        _twin_plan_fault("plan_candidate")\n\n\n@dataclass(frozen=True)\nclass _TwinCurrentEdge:\n    source_id: int\n    target_id: int\n    metadata: TwinFrozenMapping\n    token: tuple[object, ...]\n\n\ndef _normalize_twin_current_nodes(\n    nodes_by_id: object,\n    plan_nodes: dict[int, TwinSnapshotNode],\n) -> dict[int, TwinSnapshotNode]:\n    if type(nodes_by_id) is not dict:\n        _twin_plan_fault("node_mapping")\n    locations: list[tuple[int, dict[str, object], dict[str, str]]] = []\n    try:\n        for outer_key, row in nodes_by_id.items():\n            if type(outer_key) is not int or type(row) is not dict:\n                _twin_plan_fault("node_mapping")\n            found: dict[str, str] = {}\n            for key in row:\n                if type(key) is str and key in ("node_id", "t", "z", "y", "x", "gap_synthetic"):\n                    found[key] = key\n            if any(name not in found for name in ("node_id", "t", "z", "y", "x")):\n                _twin_plan_fault("node_mapping")\n            locations.append((outer_key, row, found))\n        if {item[0] for item in locations} != set(plan_nodes):\n            _twin_plan_fault("node_mapping")\n        normalized: dict[int, TwinSnapshotNode] = {}\n        for outer_key, row, found in locations:\n            node_id_value = row[found["node_id"]]\n            time_value = row[found["t"]]\n            coordinate_values = (row[found["z"]], row[found["y"]], row[found["x"]])\n            if (\n                type(node_id_value) is not int\n                or node_id_value != outer_key\n                or not _is_twin_integer(time_value)\n                or any(not _is_twin_finite_real(value) for value in coordinate_values)\n            ):\n                _twin_plan_fault("node_mapping")\n            gap = False\n            if "gap_synthetic" in found:\n                gap = bool(row[found["gap_synthetic"]] == 1)\n            current = TwinSnapshotNode(\n                outer_key,\n                int(time_value),\n                float(coordinate_values[0]),\n                float(coordinate_values[1]),\n                float(coordinate_values[2]),\n                gap,\n            )\n            expected = plan_nodes[outer_key]\n            if (\n                current.t != expected.t\n                or current.gap_synthetic is not expected.gap_synthetic\n                or any(\n                    _twin_float_token(left) != _twin_float_token(right)\n                    for left, right in zip(\n                        (current.z, current.y, current.x),\n                        (expected.z, expected.y, expected.x),\n                        strict=True,\n                    )\n                )\n            ):\n                _twin_plan_fault("node_mapping")\n            normalized[outer_key] = current\n        return normalized\n    except _TwinMutationFault:\n        raise\n    except Exception:\n        _twin_plan_fault("node_mapping")\n\n\ndef _normalize_twin_current_edges(\n    current_edges: object,\n    nodes: dict[int, TwinSnapshotNode],\n    *,\n    failure_reason: str = "current_graph_invalid",\n) -> list[_TwinCurrentEdge]:\n    if type(current_edges) is not list:\n        _twin_plan_fault(failure_reason)\n    result: list[_TwinCurrentEdge] = []\n    pairs: set[tuple[int, int]] = set()\n    try:\n        for row in current_edges:\n            if type(row) is not dict:\n                _twin_plan_fault(failure_reason)\n            source_key: object | None = None\n            target_key: object | None = None\n            source_value: object = None\n            target_value: object = None\n            for key, value in row.items():\n                if isinstance(key, str):\n                    if key == "source_id":\n                        source_key = key\n                        source_value = value\n                    elif key == "target_id":\n                        target_key = key\n                        target_value = value\n            if source_key is None or target_key is None:\n                _twin_plan_fault(failure_reason)\n            if not _is_twin_integer(source_value) or not _is_twin_integer(target_value):\n                _twin_plan_fault(failure_reason)\n            source_id = int(source_value)\n            target_id = int(target_value)\n            pair = (source_id, target_id)\n            if (\n                pair in pairs\n                or source_id not in nodes\n                or target_id not in nodes\n                or nodes[target_id].t != nodes[source_id].t + 1\n            ):\n                _twin_plan_fault(failure_reason)\n            metadata = _freeze_twin_edge_row(\n                row,\n                source_id,\n                target_id,\n                _source_key=source_key,\n                _target_key=target_key,\n            )\n            token = _twin_frozen_token(metadata)\n            result.append(_TwinCurrentEdge(source_id, target_id, metadata, token))\n            pairs.add(pair)\n        return result\n    except _TwinMutationFault:\n        raise\n    except Exception:\n        _twin_plan_fault(failure_reason)\n\n\ndef _twin_edge_semantic_token(\n    source_id: int,\n    target_id: int,\n    metadata: TwinFrozenMapping,\n) -> tuple[object, ...]:\n    return ("edge", source_id, target_id, _twin_frozen_token(metadata))\n\n\ndef _twin_replacement_row(\n    candidate: TwinCandidate,\n    nodes: dict[int, TwinSnapshotNode],\n) -> dict[str, object]:\n    distance = _twin_distance(nodes[candidate.p], nodes[candidate.b])\n    return {\n        "source_id": candidate.p,\n        "target_id": candidate.b,\n        "distance_um": distance,\n        "edge_prob": None,\n    }\n\n\ndef _twin_expected_replacement_token(\n    candidate: TwinCandidate,\n    nodes: dict[int, TwinSnapshotNode],\n) -> tuple[object, ...]:\n    distance = _twin_distance(nodes[candidate.p], nodes[candidate.b])\n    metadata_token = (\n        "mapping",\n        (\n            (("str", "source_id"), ("int", candidate.p)),\n            (("str", "target_id"), ("int", candidate.b)),\n            (("str", "distance_um"), ("float", _twin_float_token(distance))),\n            (("str", "edge_prob"), ("none",)),\n        ),\n    )\n    return ("edge", candidate.p, candidate.b, metadata_token)\n\n\ndef _twin_degrees(edges: Sequence[_TwinCurrentEdge]) -> tuple[Counter[int], Counter[int]]:\n    indegree: Counter[int] = Counter()\n    outdegree: Counter[int] = Counter()\n    for edge in edges:\n        outdegree[edge.source_id] += 1\n        indegree[edge.target_id] += 1\n    return indegree, outdegree\n\n\ndef _twin_summary(\n    status: Literal["applied", "already_applied", "no_changes"],\n    node_count: int,\n    edge_count: int,\n    accepted_count: int,\n) -> TwinMutationSummary:\n    applied = status == "applied"\n    return TwinMutationSummary(\n        status,\n        accepted_count,\n        node_count,\n        node_count,\n        edge_count,\n        edge_count,\n        accepted_count if applied else 0,\n        accepted_count if applied else 0,\n        2 * accepted_count if applied else 0,\n        accepted_count if applied else 0,\n    )\n\n\ndef _snapshot_twin_node_mapping_shallow(\n    nodes_by_id: object,\n) -> tuple[\n    dict[int, dict[str, object]],\n    tuple[\n        tuple[\n            object,\n            object,\n            tuple[tuple[object, object], ...] | None,\n        ],\n        ...,\n    ],\n] | None:\n    """Capture only identities needed by the section-10 write-free proof."""\n    if type(nodes_by_id) is not dict:\n        return None\n    entries: list[\n        tuple[object, object, tuple[tuple[object, object], ...] | None]\n    ] = []\n    for key, row in nodes_by_id.items():\n        bindings = tuple(row.items()) if type(row) is dict else None\n        entries.append((key, row, bindings))\n    return nodes_by_id, tuple(entries)\n\n\ndef _twin_node_mapping_matches_shallow_snapshot(\n    nodes_by_id: object,\n    snapshot: tuple[\n        dict[int, dict[str, object]],\n        tuple[\n            tuple[\n                object,\n                object,\n                tuple[tuple[object, object], ...] | None,\n            ],\n            ...,\n        ],\n    ]\n    | None,\n) -> bool:\n    if snapshot is None or type(nodes_by_id) is not dict or nodes_by_id is not snapshot[0]:\n        return False\n    current_entries = tuple(nodes_by_id.items())\n    expected_entries = snapshot[1]\n    if len(current_entries) != len(expected_entries):\n        return False\n    for (expected_key, expected_row, expected_bindings), (key, row) in zip(\n        expected_entries, current_entries, strict=True\n    ):\n        if key is not expected_key or row is not expected_row or type(row) is not dict:\n            return False\n        if expected_bindings is None:\n            return False\n        current_bindings = tuple(row.items())\n        if len(current_bindings) != len(expected_bindings):\n            return False\n        if any(\n            key is not expected_key or value is not expected_value\n            for (expected_key, expected_value), (key, value) in zip(\n                expected_bindings, current_bindings, strict=True\n            )\n        ):\n            return False\n    return True\n\n\ndef apply_twin_only_v1_plan(\n    nodes_by_id: dict[int, dict[str, object]],\n    current_edges: list[dict[str, object]],\n    plan: TwinPlan,\n) -> tuple[list[dict[str, object]], TwinMutationSummary]:\n    """Atomically apply a validated R1 twin plan to its exact source graph."""\n    fallback_reason = "plan_candidate"\n    try:\n        node_mapping_snapshot = _snapshot_twin_node_mapping_shallow(nodes_by_id)\n        (\n            valid_plan,\n            _counters,\n            plan_nodes,\n            ordered_plan_edges,\n            accepted_metadata_tokens,\n        ) = _validate_twin_plan_for_mutation(plan)\n        fallback_reason = "node_mapping"\n        _normalize_twin_current_nodes(nodes_by_id, plan_nodes)\n        fallback_reason = "current_graph_invalid"\n        current = _normalize_twin_current_edges(current_edges, plan_nodes)\n        accepted = valid_plan.accepted_candidates\n        accepted_count = len(accepted)\n\n        pre_tokens = tuple(\n            ("edge", edge.source_id, edge.target_id, metadata_token)\n            for edge, metadata_token in ordered_plan_edges\n        )\n        current_tokens = tuple(\n            ("edge", edge.source_id, edge.target_id, edge.token)\n            for edge in current\n        )\n        remove_tokens = tuple(\n            ("edge", candidate.q, candidate.b, metadata_token)\n            for candidate, metadata_token in zip(\n                accepted, accepted_metadata_tokens, strict=True\n            )\n        )\n        replacement_tokens = tuple(\n            _twin_expected_replacement_token(candidate, plan_nodes)\n            for candidate in accepted\n        )\n\n        def state_for_mask(mask: int) -> tuple[tuple[object, ...], ...]:\n            removed = {\n                remove_tokens[index]\n                for index in range(accepted_count)\n                if mask & (1 << (2 * index))\n            }\n            state = [token for token in pre_tokens if token not in removed]\n            state.extend(\n                replacement_tokens[index]\n                for index in range(accepted_count)\n                if mask & (1 << (2 * index + 1))\n            )\n            return tuple(state)\n\n        full_mask = (1 << (2 * accepted_count)) - 1\n        post_tokens = state_for_mask(full_mask)\n        exact_pre = current_tokens == pre_tokens\n        exact_post = accepted_count > 0 and current_tokens == post_tokens\n        if accepted_count == 0:\n            if exact_pre:\n                return current_edges, _twin_summary("no_changes", len(nodes_by_id), len(current_edges), 0)\n            indegree, outdegree = _twin_degrees(current)\n            if any(value > 1 for value in indegree.values()) or any(value > 2 for value in outdegree.values()):\n                _twin_plan_fault("current_graph_invalid")\n            _twin_plan_fault("current_graph_mismatch")\n        if exact_post:\n            return current_edges, _twin_summary(\n                "already_applied", len(nodes_by_id), len(current_edges), accepted_count\n            )\n        if not exact_pre:\n            if any(current_tokens == state_for_mask(mask) for mask in range(1, full_mask)):\n                _twin_plan_fault("partial_application")\n            indegree, outdegree = _twin_degrees(current)\n            if any(value > 1 for value in indegree.values()) or any(value > 2 for value in outdegree.values()):\n                _twin_plan_fault("current_graph_invalid")\n            _twin_plan_fault("current_graph_mismatch")\n\n        fallback_reason = "post_invariant"\n        replacement_rows = tuple(\n            _twin_replacement_row(candidate, plan_nodes) for candidate in accepted\n        )\n        for candidate, row in zip(accepted, replacement_rows, strict=True):\n            expected_distance = _twin_distance(\n                plan_nodes[candidate.p], plan_nodes[candidate.b]\n            )\n            if (\n                type(row) is not dict\n                or tuple(row) != (\n                    "source_id",\n                    "target_id",\n                    "distance_um",\n                    "edge_prob",\n                )\n                or any(type(key) is not str for key in row)\n                or type(row["source_id"]) is not int\n                or row["source_id"] != candidate.p\n                or type(row["target_id"]) is not int\n                or row["target_id"] != candidate.b\n                or type(row["distance_um"]) is not float\n                or _twin_float_token(row["distance_um"])\n                != _twin_float_token(expected_distance)\n                or row["edge_prob"] is not None\n            ):\n                _twin_plan_fault("post_invariant")\n        replacement_current = _normalize_twin_current_edges(\n            list(replacement_rows), plan_nodes, failure_reason="post_invariant"\n        )\n\n        remove_token_set = set(remove_tokens)\n        survivors = [\n            (row, edge)\n            for row, edge, token in zip(current_edges, current, current_tokens, strict=True)\n            if token not in remove_token_set\n        ]\n        result = [row for row, _edge in survivors]\n        result.extend(replacement_rows)\n\n        # Prove the complete temporary graph before exposing it to the caller.\n        temporary = [edge for _row, edge in survivors]\n        temporary.extend(replacement_current)\n        temporary_tokens = tuple(("edge", edge.source_id, edge.target_id, edge.token) for edge in temporary)\n        before_pairs = {(edge.source_id, edge.target_id) for edge in current}\n        after_pairs = {(edge.source_id, edge.target_id) for edge in temporary}\n        indegree, outdegree = _twin_degrees(temporary)\n        surviving_count = len(current_edges) - accepted_count\n        if (\n            temporary_tokens != post_tokens\n            or len(result) != len(current_edges)\n            or len(before_pairs) != len(current_edges)\n            or len(after_pairs) != len(result)\n            or len(before_pairs ^ after_pairs) != 2 * accepted_count\n            or any(result[index] is not survivors[index][0] for index in range(surviving_count))\n            or any(value > 1 for value in indegree.values())\n            or any(value > 2 for value in outdegree.values())\n            or any(indegree[candidate.q] + outdegree[candidate.q] != 0 for candidate in accepted)\n        ):\n            _twin_plan_fault("post_invariant")\n        for index, row in enumerate(replacement_rows):\n            result_row = result[surviving_count + index]\n            if result_row is not row or tuple(result_row) != (\n                "source_id", "target_id", "distance_um", "edge_prob"\n            ):\n                _twin_plan_fault("post_invariant")\n        if not _twin_node_mapping_matches_shallow_snapshot(\n            nodes_by_id, node_mapping_snapshot\n        ):\n            _twin_plan_fault("post_invariant")\n        return result, _twin_summary(\n            "applied", len(nodes_by_id), len(current_edges), accepted_count\n        )\n    except _TwinMutationFault as exc:\n        raise TwinMutationError(exc.reason) from exc\n    except TwinMutationError:\n        raise\n    except Exception as exc:\n        raise TwinMutationError(fallback_reason) from exc\n\n\ndef score_twin_deepcenter(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> TwinDeepCenterDecision:\n    """Strict, fail-closed DeepCenter decision for the frozen twin-only planner."""\n    required_bundle_keys = {"model", "cfg", "device", "torch", "path", "checkpoint_epoch"}\n    if not cfg.USE_DEEPCENTER_VETO or not cfg.STEAL_TWIN_DEEPCENTER_VETO or detector_bundle is None:\n        return TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n    if not required_bundle_keys.issubset(detector_bundle):\n        return TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n    checkpoint_epoch = detector_bundle["checkpoint_epoch"]\n    if (\n        isinstance(checkpoint_epoch, bool)\n        or not isinstance(checkpoint_epoch, int)\n        or checkpoint_epoch != cfg.DEEPCENTER_EXPECTED_EPOCH\n    ):\n        return TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n    model_cfg = detector_bundle["cfg"]\n    pool_factor = getattr(model_cfg, "pool_factor", None)\n    if isinstance(pool_factor, bool) or not isinstance(pool_factor, int) or pool_factor < 1:\n        return TwinDeepCenterDecision(False, None, "deepcenter_bundle")\n    if dataset is None or not dataset.strip():\n        return TwinDeepCenterDecision(False, None, "deepcenter_dataset")\n\n    try:\n        frame = read_test_frame(cfg.TEST_DIR, dataset, int(t), frame_cache)\n    except Exception:\n        return TwinDeepCenterDecision(False, None, "deepcenter_frame")\n    if frame is None or np.asarray(frame).size == 0:\n        return TwinDeepCenterDecision(False, None, "deepcenter_frame")\n    frame_array = np.asarray(frame)\n    if not np.all(np.isfinite(frame_array)):\n        return TwinDeepCenterDecision(False, None, "deepcenter_nonfinite")\n    if frame_array.ndim != 3:\n        return TwinDeepCenterDecision(False, None, "deepcenter_frame")\n\n    try:\n        heatmap = deepcenter_heatmap_for_frame(\n            cfg,\n            dataset,\n            int(t),\n            detector_bundle,\n            frame_cache,\n            heatmap_cache,\n        )\n    except Exception:\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    if heatmap is None or np.asarray(heatmap).size == 0:\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    heatmap_array = np.asarray(heatmap)\n    if not np.all(np.isfinite(heatmap_array)):\n        return TwinDeepCenterDecision(False, None, "deepcenter_nonfinite")\n    if heatmap_array.ndim != 3:\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n\n    point_array = np.asarray(point, dtype=np.float64)\n    if point_array.shape != (3,) or not np.all(np.isfinite(point_array)):\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    z = int(round(float(point_array[0])))\n    y = int(round(float(point_array[1]) / pool_factor))\n    x = int(round(float(point_array[2]) / pool_factor))\n    if not (0 <= z < heatmap_array.shape[0] and 0 <= y < heatmap_array.shape[1] and 0 <= x < heatmap_array.shape[2]):\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    z0, z1 = max(0, z - cfg.DEEPCENTER_SCORE_WIN_Z), min(\n        heatmap_array.shape[0], z + cfg.DEEPCENTER_SCORE_WIN_Z + 1\n    )\n    y0, y1 = max(0, y - cfg.DEEPCENTER_SCORE_WIN_YX), min(\n        heatmap_array.shape[1], y + cfg.DEEPCENTER_SCORE_WIN_YX + 1\n    )\n    x0, x1 = max(0, x - cfg.DEEPCENTER_SCORE_WIN_YX), min(\n        heatmap_array.shape[2], x + cfg.DEEPCENTER_SCORE_WIN_YX + 1\n    )\n    patch = heatmap_array[z0:z1, y0:y1, x0:x1]\n    if patch.size == 0:\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    if not np.all(np.isfinite(patch)):\n        return TwinDeepCenterDecision(False, None, "deepcenter_nonfinite")\n    raw_value = np.max(patch)\n    if np.ndim(raw_value) != 0:\n        return TwinDeepCenterDecision(False, None, "deepcenter_heatmap")\n    raw_score = float(raw_value)\n    if not np.isfinite(raw_score):\n        return TwinDeepCenterDecision(False, None, "deepcenter_nonfinite")\n    if raw_score < 0.12:\n        return TwinDeepCenterDecision(False, raw_score, "deepcenter_threshold")\n    return TwinDeepCenterDecision(True, raw_score, None)\n\n\ndef add_safe_divisions_postlink(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> list[dict[str, object]]:\n    if not cfg.OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\n        return edges\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n\n    out_by_source: dict[int, list[dict[str, object]]] = {}\n    incoming: set[int] = set()\n    for edge in edges:\n        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)\n        incoming.add(int(edge["target_id"]))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n\n    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}\n    global_cap = max(1, int(round(max(1, len(edges)) * cfg.SAFE_DIV_GLOBAL_FRAC_CAP)))\n    added: list[dict[str, object]] = []\n    used_targets: set[int] = set()\n\n    for t in sorted(ids_by_t):\n        child_frame_ids = ids_by_t.get(t + 1, [])\n        if not child_frame_ids:\n            continue\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\n        if not source_ids or not candidate_ids:\n            continue\n\n        frame_cap = max(1, int(round(len(source_ids) * cfg.SAFE_DIV_FRAME_FRAC_CAP)))\n        if cfg.SAFE_DIV_MODE == "e23":\n            proposals = _e23_frame_proposals(\n                cfg,\n                nodes_by_id,\n                out_by_source,\n                incoming,\n                existing_edges,\n                source_ids,\n                candidate_ids,\n                t,\n                stats,\n                dataset,\n                deepcenter_bundle,\n                frame_cache,\n                deepcenter_cache,\n            )\n        else:  # "legacy"; build_config rejects every other value\n            proposals = _legacy_frame_proposals(\n                cfg,\n                nodes_by_id,\n                out_by_source,\n                existing_edges,\n                source_ids,\n                candidate_ids,\n                t,\n                stats,\n                dataset,\n                deepcenter_bundle,\n                frame_cache,\n                deepcenter_cache,\n            )\n\n        stats["safe_division_candidates"] += len(proposals)\n        if not proposals:\n            continue\n        proposals.sort(key=lambda item: item[0])\n        added_this_frame = 0\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\n            if len(added) >= global_cap:\n                stats["safe_division_skipped_cap"] += 1\n                break\n            if added_this_frame >= frame_cap:\n                break\n            if candidate_id in used_targets or candidate_id in incoming:\n                continue\n            added.append({\n                "source_id": source_id,\n                "target_id": candidate_id,\n                "edge_prob": None,\n                "distance_um": parent_dist,\n                "safe_division": 1,\n            })\n            used_targets.add(candidate_id)\n            added_this_frame += 1\n\n    if added:\n        stats["safe_divisions_added"] = len(added)\n        return [*edges, *added]\n    return edges\n\n\ndef _legacy_frame_proposals(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    out_by_source: dict[int, list[dict[str, object]]],\n    existing_edges: set[tuple[int, int]],\n    source_ids: list[int],\n    candidate_ids: list[int],\n    t: int,\n    stats: dict[str, int],\n    dataset: str | None,\n    deepcenter_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    deepcenter_cache: dict[tuple[str, int], np.ndarray],\n) -> list[tuple[float, int, int, float, float]]:\n    """Candidate generation exactly as the pre-E23 port: distance gates only.\n\n    No cKDTree and no structural restrictions; the legacy output is\n    output-identical to the Phase-1 behavior.\n    """\n    proposals: list[tuple[float, int, int, float, float]] = []\n    for source_id in source_ids:\n        source = nodes_by_id[source_id]\n        existing_child_edge = out_by_source[source_id][0]\n        existing_child_id = int(existing_child_edge["target_id"])\n        existing_child = nodes_by_id.get(existing_child_id)\n        if existing_child is None or int(existing_child["t"]) != t + 1:\n            continue\n        child_dist = edge_distance_um(source, existing_child)\n        if child_dist > cfg.SAFE_DIV_EXISTING_CHILD_MAX_UM:\n            continue\n        for candidate_id in candidate_ids:\n            if (source_id, candidate_id) in existing_edges:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            parent_dist = edge_distance_um(source, candidate)\n            if parent_dist > cfg.SAFE_DIV_MAX_UM:\n                continue\n            sister_dist = edge_distance_um(existing_child, candidate)\n            if sister_dist > cfg.SAFE_DIV_SISTER_MAX_UM:\n                continue\n            if cfg.DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\n                cfg,\n                dataset,\n                int(candidate["t"]),\n                (float(candidate["z"]), float(candidate["y"]), float(candidate["x"])),\n                deepcenter_bundle,\n                frame_cache,\n                deepcenter_cache,\n                stats,\n                "safe_div",\n                cfg.DEEPCENTER_SAFE_DIV_THRESHOLD,\n            ):\n                continue\n            score = parent_dist + 0.15 * sister_dist\n            proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\n    return proposals\n\n\ndef _e23_frame_proposals(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    out_by_source: dict[int, list[dict[str, object]]],\n    incoming: set[int],\n    existing_edges: set[tuple[int, int]],\n    source_ids: list[int],\n    candidate_ids: list[int],\n    t: int,\n    stats: dict[str, int],\n    dataset: str | None,\n    deepcenter_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    deepcenter_cache: dict[tuple[str, int], np.ndarray],\n) -> list[tuple[float, int, int, float, float]]:\n    """E23 notebook candidate generation (pub923_repro add_safe_divisions_postlink).\n\n    One cKDTree over ALL orphan candidates of the frame, in physical\n    micrometres; the mutual gate queries the GLOBAL nearest orphan of the\n    existing child, so the nearest orphan may sit outside the parent ball.\n    Per-source gates: existing child at t+1 within\n    SAFE_DIV_EXISTING_CHILD_MAX_UM; C1 mid-track parent (the notebook\n    hardwires it; the port exposes SAFE_DIV_REQUIRE_MID_TRACK_PARENT).\n    Per-candidate gates accept exactly the notebook\'s set: existing edge,\n    parent radius (ball membership), sister radius, C2 mutual nearest\n    orphan, DeepCenter safe-division veto (before divergence), C3 t+2\n    divergence (distinct single successors whose t+2 separation grew by at\n    least SAFE_DIV_DIVERGE_UM).\n\n    Parity note: the submitted notebook declares\n    ``safe_division_geometric_candidates``,\n    ``safe_division_mutual_nn_rejected`` and\n    ``safe_division_divergence_rejected`` but never increments them (they\n    always print 0). The port fills all three truthfully: geometric counts\n    every candidate that passes the existing-edge and the distance gates,\n    before mutual-NN, DeepCenter, or divergence; the other two count the\n    respective gate rejections.\n    """\n    positions = np.asarray(\n        [\n            [\n                float(nodes_by_id[candidate_id]["z"]) * VOXEL_SCALE_UM[0],\n                float(nodes_by_id[candidate_id]["y"]) * VOXEL_SCALE_UM[1],\n                float(nodes_by_id[candidate_id]["x"]) * VOXEL_SCALE_UM[2],\n            ]\n            for candidate_id in candidate_ids\n        ],\n        dtype=float,\n    )\n    tree = cKDTree(positions) if len(positions) else None\n\n    def _succ1(node_id: int) -> int | None:\n        out_edges = out_by_source.get(node_id, [])\n        return int(out_edges[0]["target_id"]) if len(out_edges) == 1 else None\n\n    proposals: list[tuple[float, int, int, float, float]] = []\n    for source_id in source_ids:\n        source = nodes_by_id[source_id]\n        existing_child_edge = out_by_source[source_id][0]\n        existing_child_id = int(existing_child_edge["target_id"])\n        existing_child = nodes_by_id.get(existing_child_id)\n        if existing_child is None or int(existing_child["t"]) != t + 1:\n            continue\n        child_dist = edge_distance_um(source, existing_child)\n        if child_dist > cfg.SAFE_DIV_EXISTING_CHILD_MAX_UM:\n            continue\n        # C1: parent must be mid-track (has a predecessor) - not a track start\n        if cfg.SAFE_DIV_REQUIRE_MID_TRACK_PARENT and source_id not in incoming:\n            continue\n        if tree is None:\n            continue\n        source_point = np.asarray(\n            [\n                float(source["z"]) * VOXEL_SCALE_UM[0],\n                float(source["y"]) * VOXEL_SCALE_UM[1],\n                float(source["x"]) * VOXEL_SCALE_UM[2],\n            ],\n            dtype=float,\n        )\n        near = tree.query_ball_point(source_point, r=cfg.SAFE_DIV_MAX_UM)\n        child_point = np.asarray(\n            [\n                float(existing_child["z"]) * VOXEL_SCALE_UM[0],\n                float(existing_child["y"]) * VOXEL_SCALE_UM[1],\n                float(existing_child["x"]) * VOXEL_SCALE_UM[2],\n            ],\n            dtype=float,\n        )\n        # Single nearest query on the ALL-orphan tree at the existing-child\n        # point, exactly the notebook\'s call. SciPy does not specify which of\n        # several exactly equidistant points cKDTree.query returns, and\n        # nonzero-index tie winners occur in practice, so a tie\'s winner is an\n        # unspecified implementation detail here, not a guaranteed contract.\n        nearest_dist, nearest_idx = tree.query(child_point)\n        mutual_id = candidate_ids[int(nearest_idx)] if nearest_dist <= cfg.SAFE_DIV_SISTER_MAX_UM else None\n        for near_idx in near:\n            candidate_id = candidate_ids[int(near_idx)]\n            if (source_id, candidate_id) in existing_edges:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            parent_dist = edge_distance_um(source, candidate)\n            if parent_dist > cfg.SAFE_DIV_MAX_UM:\n                continue\n            sister_dist = edge_distance_um(existing_child, candidate)\n            if sister_dist > cfg.SAFE_DIV_SISTER_MAX_UM:\n                continue\n            stats["safe_division_geometric_candidates"] += 1\n            # C2: sisters must be MUTUAL nearest orphans\n            if cfg.SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_id:\n                stats["safe_division_mutual_nn_rejected"] += 1\n                continue\n            if cfg.DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\n                cfg,\n                dataset,\n                int(candidate["t"]),\n                (float(candidate["z"]), float(candidate["y"]), float(candidate["x"])),\n                deepcenter_bundle,\n                frame_cache,\n                deepcenter_cache,\n                stats,\n                "safe_div",\n                cfg.DEEPCENTER_SAFE_DIV_THRESHOLD,\n            ):\n                continue\n            # C3: DIVERGENCE - both daughters continue at t+2 and separate.\n            if cfg.SAFE_DIV_REQUIRE_DIVERGENCE:\n                successor_1 = _succ1(existing_child_id)\n                successor_2 = _succ1(candidate_id)\n                rejected = successor_1 is None or successor_2 is None or successor_1 == successor_2\n                if not rejected:\n                    node_1 = nodes_by_id.get(successor_1)\n                    node_2 = nodes_by_id.get(successor_2)\n                    rejected = (\n                        node_1 is None\n                        or node_2 is None\n                        or int(node_1["t"]) != t + 2\n                        or int(node_2["t"]) != t + 2\n                        or edge_distance_um(node_1, node_2) - sister_dist < cfg.SAFE_DIV_DIVERGE_UM\n                    )\n                if rejected:\n                    stats["safe_division_divergence_rejected"] += 1\n                    continue\n            score = parent_dist + 0.15 * sister_dist\n            proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\n    return proposals\n', 'src/biohub/public_postproc/frames.py': '"""Test-frame access for the image-intensity refinement steps.\n\nThe notebook cell re-implemented its own blosc2 chunk reader with a\n``zarr``-library fallback; here we reuse ``biohub.io.open_volume(...).frame(t)``\n(same one-chunk-per-timepoint blosc2/zstd layout) as instructed. Any read\nfailure still lands in the caller\'s ``except Exception`` and degrades to\n"keep the unrefined midpoint", matching the notebook\'s behaviour.\n\nThe E23 stack additionally refines *every* node coordinate against the local\nintensity centroid before any edge-distance filtering\n(:func:`refine_centroids` / :func:`refine_all_centroids`); unlike the\ngap-refinement fallback, frame read errors there are raised, never swallowed.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.io import open_volume\nfrom biohub.public_postproc.config import PostprocConfig\nfrom biohub.public_postproc.geometry import point_distance_um\n\n\ndef read_test_frame(test_dir: Path, dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:\n    if t in frame_cache:\n        return frame_cache[t]\n    frame = open_volume(test_dir / f"{dataset}.zarr").frame(t)\n    frame_cache[t] = frame\n    return frame\n\n\ndef refine_synthetic_midpoint(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    t: int,\n    midpoint: tuple[float, float, float],\n    frame_cache: dict[int, np.ndarray],\n    stats: dict[str, int],\n) -> tuple[float, float, float]:\n    if not cfg.GAP_REFINE_SYNTHETIC or dataset is None:\n        return midpoint\n    try:\n        frame = read_test_frame(cfg.TEST_DIR, dataset, t, frame_cache)\n        z, y, x = (int(round(v)) for v in midpoint)\n        z0 = max(0, z - cfg.GAP_REFINE_WIN_Z)\n        z1 = min(frame.shape[0], z + cfg.GAP_REFINE_WIN_Z + 1)\n        y0 = max(0, y - cfg.GAP_REFINE_WIN_YX)\n        y1 = min(frame.shape[1], y + cfg.GAP_REFINE_WIN_YX + 1)\n        x0 = max(0, x - cfg.GAP_REFINE_WIN_YX)\n        x1 = min(frame.shape[2], x + cfg.GAP_REFINE_WIN_YX + 1)\n        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)\n        if patch.size == 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        baseline = float(np.percentile(patch, 20.0))\n        weights = np.maximum(patch - baseline, 0.0)\n        total = float(weights.sum())\n        if total <= 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\n        refined = (\n            float((weights * zz).sum() / total),\n            float((weights * yy).sum() / total),\n            float((weights * xx).sum() / total),\n        )\n        if point_distance_um(refined, midpoint) > cfg.GAP_REFINE_MAX_SHIFT_UM:\n            stats["gap_refine_rejected_shift"] += 1\n            return midpoint\n        stats["gap_refined_synthetic"] += 1\n        return refined\n    except Exception:\n        stats["gap_refine_failed"] += 1\n        return midpoint\n\n\ndef refine_centroids(\n    cfg: PostprocConfig,\n    vol: np.ndarray,\n    coords: list[tuple[float, float, float]],\n    stats: dict[str, int],\n) -> list[tuple[float, float, float]]:\n    """Intensity-centroid refinement of node coordinates within one frame (E23).\n\n    For every ``(z, y, x)``: round to the nearest voxel with Python\n    ``int(round())``, take the clipped ``REFINE_CENTROIDS_WIN_Z`` /\n    ``REFINE_CENTROIDS_WIN_YX`` window, subtract the\n    ``REFINE_CENTROIDS_BASELINE_PERCENTILE`` baseline, and move the point to\n    the weighted centroid in absolute voxel coordinates unless the physical\n    shift exceeds ``REFINE_CENTROIDS_MAX_SHIFT_UM``. Empty or flat windows\n    leave the point unchanged (``centroid_refine_no_signal``).\n    ``centroid_refine_moved`` counts only accepted points whose output\n    coordinates actually differ from the input, matching the notebook\'s\n    ``total_refined`` semantics (an accepted exact no-op is not "moved", so\n    the outcome counters need not partition ``centroid_refine_examined``).\n    """\n    wz = cfg.REFINE_CENTROIDS_WIN_Z\n    wyx = cfg.REFINE_CENTROIDS_WIN_YX\n    depth, height, width = vol.shape\n    refined_coords: list[tuple[float, float, float]] = []\n    for z, y, x in coords:\n        stats["centroid_refine_examined"] += 1\n        zi, yi, xi = int(round(z)), int(round(y)), int(round(x))\n        z0 = max(0, zi - wz)\n        z1 = min(depth, zi + wz + 1)\n        y0 = max(0, yi - wyx)\n        y1 = min(height, yi + wyx + 1)\n        x0 = max(0, xi - wyx)\n        x1 = min(width, xi + wyx + 1)\n        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)\n        if patch.size == 0:\n            stats["centroid_refine_no_signal"] += 1\n            refined_coords.append((z, y, x))\n            continue\n        baseline = float(np.percentile(patch, cfg.REFINE_CENTROIDS_BASELINE_PERCENTILE))\n        weights = np.maximum(patch - baseline, 0.0)\n        total = float(weights.sum())\n        if total <= 0:\n            stats["centroid_refine_no_signal"] += 1\n            refined_coords.append((z, y, x))\n            continue\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\n        refined = (\n            float((weights * zz).sum() / total),\n            float((weights * yy).sum() / total),\n            float((weights * xx).sum() / total),\n        )\n        if point_distance_um(refined, (z, y, x)) > cfg.REFINE_CENTROIDS_MAX_SHIFT_UM:\n            stats["centroid_refine_rejected_shift"] += 1\n            refined_coords.append((z, y, x))\n            continue\n        if refined != (z, y, x):\n            stats["centroid_refine_moved"] += 1\n        refined_coords.append(refined)\n    return refined_coords\n\n\ndef refine_all_centroids(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    dataset: str | None,\n    frame_cache: dict[int, np.ndarray],\n    stats: dict[str, int],\n) -> dict[int, dict[str, object]]:\n    """Run :func:`refine_centroids` over every node, reading each frame ``t`` once.\n\n    Nodes are grouped by ``t`` in dict iteration order (which is preserved;\n    only coordinates are mutated in place). Frames go through\n    :func:`read_test_frame` into the shared ``frame_cache`` so later\n    gap-close / safe-division / DeepCenter passes reuse them. A frame read\n    failure raises ``RuntimeError`` naming the dataset, ``t`` and zarr path,\n    chained from the original exception -- refinement never swallows it.\n    """\n    if dataset is None:\n        raise ValueError("refine_all_centroids needs a dataset name to read test frames")\n\n    nodes_by_t: dict[int, list[dict[str, object]]] = {}\n    for node in nodes_by_id.values():\n        nodes_by_t.setdefault(int(node["t"]), []).append(node)\n\n    zarr_path = cfg.TEST_DIR / f"{dataset}.zarr"\n    for t, nodes in nodes_by_t.items():\n        try:\n            frame = read_test_frame(cfg.TEST_DIR, dataset, t, frame_cache)\n        except Exception as exc:\n            raise RuntimeError(f"centroid refinement cannot read frame: dataset={dataset} t={t} path={zarr_path}") from exc\n        coords = [(float(node["z"]), float(node["y"]), float(node["x"])) for node in nodes]\n        for node, point in zip(nodes, refine_centroids(cfg, frame, coords, stats), strict=True):\n            node["z"], node["y"], node["x"] = point\n    return nodes_by_id\n', 'src/biohub/public_postproc/geometry.py': '"""Geometry helpers ported verbatim from the notebook\'s post-processing cell."""\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\n\nfrom biohub.io import DEFAULT_SCALE\n\nVOXEL_SCALE_UM: tuple[float, float, float] = DEFAULT_SCALE\n\n\ndef edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:\n    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]\n    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]\n    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:\n    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]\n    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]\n    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef node_point(node: dict[str, object]) -> tuple[float, float, float]:\n    return (float(node["z"]), float(node["y"]), float(node["x"]))\n\n\ndef edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:\n    prob = edge.get("edge_prob")\n    prob_value = float(prob) if prob is not None else 0.0\n    return prob_value, -float(edge["distance_um"])\n\n\ndef position_um(node: dict[str, object]) -> np.ndarray:\n    return np.array(\n        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],\n        dtype=np.float64,\n    )\n\n\ndef next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:\n    return max(nodes_by_id) + 1 if nodes_by_id else 1\n', 'src/biohub/public_postproc/graph_ops.py': '"""Graph repair/relink/filter passes, ported verbatim from the notebook\'s\npost-processing cell (``filter_output_graph`` and its helpers).\n\nEvery function takes an explicit ``cfg: PostprocConfig`` instead of reading\nmodule-level ``BIOHUB_*`` globals, and a ``stats: dict[str, int]`` counter\ndict that callers must pre-seed with zeros for every key a pass may touch\n(``pipeline.new_stats()`` does this) -- both notebook behaviours otherwise\nkept unchanged, including iteration order (it determines row order in the\nfinal CSV, so ``dict``/list ordering here must stay exactly as ported).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial import cKDTree\n\nfrom biohub.appearance_cost import frame_appearance_penalty\nfrom biohub.public_postproc.config import PostprocConfig\nfrom biohub.public_postproc.deepcenter import deepcenter_accept_repair_point\nfrom biohub.public_postproc.frames import refine_synthetic_midpoint\nfrom biohub.public_postproc.geometry import edge_distance_um, next_node_id, node_point, point_distance_um, position_um\n\n\ndef _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_source: dict[int, list[int]] = {}\n    for edge in edges:\n        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))\n    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}\n\n\ndef _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_target: dict[int, list[int]] = {}\n    for edge in edges:\n        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))\n    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}\n\n\ndef prepare_consensus_reservations(\n    consensus_pairs_by_t,\n    nodes_by_id,\n    raw_edges,\n    filtered_edges,\n    tight_gate_um,\n):\n    """Select unambiguous reciprocal-consensus pairs for motion reservation."""\n    if type(consensus_pairs_by_t) is not dict:\n        raise ValueError("consensus_pairs_by_t must be an exact dict")\n    if type(nodes_by_id) is not dict:\n        raise ValueError("nodes_by_id must be an exact dict")\n    if type(raw_edges) is not list or type(filtered_edges) is not list:\n        raise ValueError("edge collections must be exact lists")\n    if type(tight_gate_um) is bool or type(tight_gate_um) not in (int, float):\n        raise ValueError("tight_gate_um must be int/float")\n    if not np.isfinite(tight_gate_um) or tight_gate_um <= 0:\n        raise ValueError("tight_gate_um must be finite and > 0")\n    for t, pairs in consensus_pairs_by_t.items():\n        if type(t) is not int or t < 0 or t > 98 or type(pairs) is not list:\n            raise ValueError("bad consensus key or value type")\n        if any(type(p) is not tuple or len(p) != 2 or\n               any(type(x) is not int or x < 0 for x in p) for p in pairs):\n            raise ValueError("bad consensus pair element")\n        if pairs != sorted(pairs) or len(set(pairs)) != len(pairs):\n            raise ValueError("consensus pairs must be sorted and unique")\n        if len({p[0] for p in pairs}) != len(pairs) or len({p[1] for p in pairs}) != len(pairs):\n            raise ValueError("consensus pairs must be one-to-one")\n    for key, node in nodes_by_id.items():\n        if type(key) is not int or type(node) is not dict or node.get("node_id") != key:\n            raise ValueError("node entry mismatch")\n        if type(node.get("t")) is not int:\n            raise ValueError("node time must be builtin int")\n        for field in ("z", "y", "x"):\n            value = node.get(field)\n            if type(value) is bool or type(value) not in (int, float) or not np.isfinite(value):\n                raise ValueError("node field not finite numeric: " + field)\n\n    def edge_set(edges, name):\n        found = set()\n        for edge in edges:\n            if type(edge) is not dict:\n                raise ValueError("edge must be a dict: " + name)\n            source, target = edge.get("source_id"), edge.get("target_id")\n            if type(source) is not int or type(target) is not int or source < 0 or target < 0:\n                raise ValueError("bad edge ids in " + name)\n            if (source, target) in found:\n                raise ValueError("duplicate pair in " + name)\n            found.add((source, target))\n        return found\n\n    raw_set = edge_set(raw_edges, "raw_edges")\n    filtered_set = edge_set(filtered_edges, "filtered_edges")\n\n    def degrees(edges):\n        out, incoming = {}, {}\n        for edge in edges:\n            source, target = edge["source_id"], edge["target_id"]\n            out[source] = out.get(source, 0) + 1\n            incoming[target] = incoming.get(target, 0) + 1\n        return out, incoming\n\n    raw_out, raw_in = degrees(raw_edges)\n    filtered_out, filtered_in = degrees(filtered_edges)\n    result = {}\n    for t in sorted(consensus_pairs_by_t):\n        kept = []\n        for source_id, target_id in consensus_pairs_by_t[t]:\n            source, target = nodes_by_id.get(source_id), nodes_by_id.get(target_id)\n            if source is None or target is None or source["t"] != t or target["t"] != t + 1:\n                continue\n            if (source_id, target_id) not in raw_set or (source_id, target_id) not in filtered_set:\n                continue\n            if (raw_out.get(source_id, 0) != 1 or raw_in.get(target_id, 0) != 1 or\n                    filtered_out.get(source_id, 0) != 1 or filtered_in.get(target_id, 0) != 1):\n                continue\n            if np.linalg.norm(position_um(source) - position_um(target)) > tight_gate_um:\n                continue\n            kept.append((source_id, target_id))\n        if kept:\n            result[t] = kept\n    return result\n\n\ndef motion_relink_edges(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    stats: dict[str, int],\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\n    *,\n    appearance_frames: dict[int, dict] | None = None,\n    consensus_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    consensus_soft_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    bidirectional_motion_consistency: bool = False,\n) -> list[dict[str, object]]:\n    if bidirectional_motion_consistency:\n        stats.setdefault("motion_relink_bidirectional_kept", 0)\n        stats.setdefault("motion_relink_bidirectional_rejected", 0)\n    if appearance_frames is not None and (consensus_pairs_by_t is not None or consensus_soft_pairs_by_t is not None):\n        raise ValueError("appearance_frames cannot be combined with consensus pairs")\n    if consensus_pairs_by_t is not None and consensus_soft_pairs_by_t is not None:\n        raise ValueError("hard and soft consensus pairs cannot be combined")\n    if appearance_frames is not None:\n        if type(appearance_frames) is not dict:\n            raise ValueError("appearance_frames must be a dict keyed by int time")\n        for key in appearance_frames:\n            if type(key) is not int or not 0 <= key <= 98:\n                raise ValueError("appearance_frames keys must be builtin ints in 0..98")\n        if not cfg.OUTPUT_MOTION_RELINK or (not nodes_by_id and appearance_frames):\n            raise ValueError("appearance_frames supplied but motion relink is disabled or has no nodes")\n\n    if not cfg.OUTPUT_MOTION_RELINK or not nodes_by_id:\n        return []\n\n    learned_edge_probs = learned_edge_probs or {}\n\n    def learned_prob(source_id: int, target_id: int) -> float:\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\n        try:\n            value = float(value)\n        except (TypeError, ValueError):\n            return 0.0\n        if not np.isfinite(value):\n            return 0.0\n        if value < 0.0 or value > 1.0:\n            import math\n\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n        return float(np.clip(value, 0.0, 1.0))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    for ids in ids_by_t.values():\n        ids.sort()\n\n    if appearance_frames is not None:\n        active_pairs = {t for t, ids in ids_by_t.items() if ids and ids_by_t.get(t + 1)}\n        if set(active_pairs) != set(appearance_frames):\n            raise ValueError("appearance_frames keys must match every active transition frame")\n\n    for consensus_name, consensus_pairs in (\n        ("consensus_pairs_by_t", consensus_pairs_by_t),\n        ("consensus_soft_pairs_by_t", consensus_soft_pairs_by_t),\n    ):\n        if consensus_pairs is None:\n            continue\n        if type(consensus_pairs) is not dict or not nodes_by_id:\n            raise ValueError(f"{consensus_name} requires enabled motion and nonempty nodes")\n        active_pairs = {t for t, ids in ids_by_t.items() if ids and ids_by_t.get(t + 1)}\n        if not set(consensus_pairs).issubset(active_pairs):\n            raise ValueError(f"{consensus_name} keys must be active transition frames")\n        for t, pairs in consensus_pairs.items():\n            if type(t) is not int or type(pairs) is not list or pairs != sorted(pairs):\n                raise ValueError("invalid consensus pair mapping")\n            if len(set(pairs)) != len(pairs) or len({p[0] for p in pairs}) != len(pairs) or len({p[1] for p in pairs}) != len(pairs):\n                raise ValueError("consensus pairs must be one-to-one")\n            for pair in pairs:\n                if type(pair) is not tuple or len(pair) != 2 or any(type(x) is not int for x in pair):\n                    raise ValueError("consensus pairs must contain integer tuples")\n                source_id, target_id = pair\n                if source_id not in nodes_by_id or target_id not in nodes_by_id:\n                    raise ValueError("consensus pair references unknown node")\n                if nodes_by_id[source_id]["t"] != t or nodes_by_id[target_id]["t"] != t + 1:\n                    raise ValueError("consensus pair has invalid node times")\n\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\n    if frame_sizes and max(frame_sizes) > cfg.MOTION_RELINK_MAX_FRAME_NODES:\n        if appearance_frames is not None:\n            raise ValueError("appearance_frames cannot be used when a large frame is skipped")\n        stats["motion_relink_skipped_large_frame"] = 1\n        return []\n\n    position_lookup = {node_id: position_um(node) for node_id, node in nodes_by_id.items()}\n    frame_penalty: np.ndarray | None = None\n    full_source_index: dict[int, int] = {}\n    full_target_index: dict[int, int] = {}\n    predecessor_position_um: dict[int, np.ndarray] = {}\n    selected_edges: list[dict[str, object]] = []\n    consensus_soft_pairs = (\n        {pair for pairs in consensus_soft_pairs_by_t.values() for pair in pairs}\n        if consensus_soft_pairs_by_t is not None\n        else set()\n    )\n\n    def assign_pass(\n        source_ids: list[int],\n        target_ids: list[int],\n        gate_um: float,\n    ) -> list[tuple[int, int, float, float, float]]:\n        if not source_ids or not target_ids:\n            return []\n        big = gate_um * 1000.0 + 1.0\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\n        raw_dist = np.full_like(cost, np.inf)\n        motion_dist = np.full_like(cost, np.inf)\n        prob_matrix = np.zeros_like(cost)\n        for i, source_id in enumerate(source_ids):\n            source_pos = position_lookup[source_id]\n            prev_pos = predecessor_position_um.get(source_id)\n            if prev_pos is None:\n                predicted = source_pos\n            else:\n                predicted = source_pos + cfg.MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n            for j, target_id in enumerate(target_ids):\n                target_pos = position_lookup[target_id]\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > gate_um:\n                    continue\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                cost_prob = 1.0 if (source_id, target_id) in consensus_soft_pairs else prob\n                raw_dist[i, j] = raw\n                motion_dist[i, j] = motion\n                prob_matrix[i, j] = prob\n                cost[i, j] = (\n                    motion\n                    + 0.05 * raw\n                    - cfg.MOTION_RELINK_LEARNED_BONUS * cost_prob\n                )\n                if frame_penalty is not None:\n                    cost[i, j] += frame_penalty[full_source_index[source_id], full_target_index[target_id]]\n        row_ind, col_ind = linear_sum_assignment(cost)\n        matches: list[tuple[int, int, float, float, float]] = []\n        for r, c in zip(row_ind, col_ind, strict=True):\n            if cost[r, c] >= big:\n                continue\n            matches.append((\n                source_ids[int(r)],\n                target_ids[int(c)],\n                float(raw_dist[r, c]),\n                float(motion_dist[r, c]),\n                float(prob_matrix[r, c]),\n            ))\n        return matches\n\n    times = sorted(ids_by_t)\n    for t in times:\n        source_ids = ids_by_t.get(t, [])\n        target_ids = ids_by_t.get(t + 1, [])\n        if not source_ids or not target_ids:\n            continue\n        if appearance_frames is None:\n            frame_penalty = None\n            full_source_index = {}\n            full_target_index = {}\n        else:\n            frame_penalty = frame_appearance_penalty(appearance_frames[t], t, source_ids, target_ids)\n            full_source_index = {node_id: index for index, node_id in enumerate(source_ids)}\n            full_target_index = {node_id: index for index, node_id in enumerate(target_ids)}\n        unmatched_sources = set(source_ids)\n        unmatched_targets = set(target_ids)\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\n        if consensus_pairs_by_t is not None:\n            for source_id, target_id in consensus_pairs_by_t.get(t, []):\n                source_pos = position_lookup[source_id]\n                target_pos = position_lookup[target_id]\n                prev_pos = predecessor_position_um.get(source_id)\n                predicted = source_pos if prev_pos is None else source_pos + cfg.MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > cfg.MOTION_RELINK_TIGHT_UM:\n                    raise ValueError("consensus reservation exceeds tight gate")\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    raise ValueError("consensus reservation endpoint conflict")\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, "consensus_reserved", prob))\n                stats["motion_relink_consensus_reserved_edges"] = stats.get("motion_relink_consensus_reserved_edges", 0) + 1\n        for pass_name, gate_um in (("tight", cfg.MOTION_RELINK_TIGHT_UM), ("relaxed", cfg.MOTION_RELINK_RELAXED_UM)):\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\n            for source_id, target_id, raw, motion, prob in matches:\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    continue\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\n                if pass_name == "tight":\n                    stats["motion_relink_tight_edges"] += 1\n                else:\n                    stats["motion_relink_relaxed_edges"] += 1\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\n            selected_edges.append({\n                "source_id": source_id,\n                "target_id": target_id,\n                "edge_prob": prob,\n                "distance_um": raw,\n                "motion_distance_um": motion,\n                "motion_relinked": 1,\n                "motion_pass": pass_name,\n            })\n            predecessor_position_um[target_id] = position_lookup[source_id]\n        stats["motion_relink_frames"] += 1\n\n    if bidirectional_motion_consistency and selected_edges:\n        # E34: retain only forward-selected edges that are also the closest\n        # admissible predecessor when the same transition is viewed backward.\n        # This is deliberately fail-closed and never adds an edge.\n        checked: list[dict[str, object]] = []\n        for edge in selected_edges:\n            sid, tid = int(edge["source_id"]), int(edge["target_id"])\n            t = int(nodes_by_id[sid]["t"])\n            candidates = []\n            for cand_sid in ids_by_t.get(t, []):\n                raw = float(np.linalg.norm(position_lookup[tid] - position_lookup[cand_sid]))\n                if np.isfinite(raw) and raw <= cfg.MOTION_RELINK_RELAXED_UM:\n                    candidates.append((raw, cand_sid))\n            if candidates and sid == min(candidates, key=lambda item: (item[0], item[1]))[1]:\n                checked.append(edge)\n        stats["motion_relink_bidirectional_kept"] = len(checked)\n        stats["motion_relink_bidirectional_rejected"] = len(selected_edges) - len(checked)\n        selected_edges = checked\n        stats["motion_relink_tight_edges"] = sum(e.get("motion_pass") == "tight" for e in selected_edges)\n        stats["motion_relink_relaxed_edges"] = sum(e.get("motion_pass") == "relaxed" for e in selected_edges)\n        stats["motion_relink_consensus_reserved_edges"] = sum(\n            e.get("motion_pass") == "consensus_reserved" for e in selected_edges\n        )\n    stats["motion_relink_edges"] = len(selected_edges)\n    return selected_edges\n\n\ndef close_single_frame_gaps(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not cfg.OUTPUT_GAP_CLOSE or cfg.GAP_CLOSE_MAX_GAP < 1 or not edges:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    incident = outgoing | incoming\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    isolated_by_t: dict[int, list[int]] = {}\n    all_ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        all_ids_by_t.setdefault(t, []).append(node_id)\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incident:\n            isolated_by_t.setdefault(t, []).append(node_id)\n\n    max_synthetic = min(\n        cfg.GAP_CLOSE_MAX_ADDED_ABS,\n        max(1, int(round(len(nodes_by_id) * cfg.GAP_CLOSE_MAX_ADDED_FRAC))) if cfg.GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,\n    )\n    next_id = next_node_id(nodes_by_id)\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n    used_starts: set[int] = set()\n    used_isolated: set[int] = set()\n    synthetic_added = 0\n    new_edges: list[dict[str, object]] = []\n\n    density_cache: dict[int, dict[int, float]] = {}\n\n    def frame_local_spacing(t: int) -> dict[int, float]:\n        cached = density_cache.get(t)\n        if cached is not None:\n            return cached\n\n        frame_ids = all_ids_by_t.get(t, [])\n        if len(frame_ids) <= 1:\n            result = {node_id: cfg.GAP_DENSITY_REFERENCE_UM for node_id in frame_ids}\n            density_cache[t] = result\n            return result\n\n        positions = np.stack([position_um(nodes_by_id[node_id]) for node_id in frame_ids])\n        tree = cKDTree(positions)\n        query_k = min(len(frame_ids), max(2, cfg.GAP_DENSITY_NEIGHBORS + 1))\n        distances, _ = tree.query(positions, k=query_k)\n        if distances.ndim == 1:\n            distances = distances[:, None]\n\n        result: dict[int, float] = {}\n        for idx, node_id in enumerate(frame_ids):\n            neighbour_distances = distances[idx, 1:]\n            neighbour_distances = neighbour_distances[np.isfinite(neighbour_distances)]\n            spacing = float(np.median(neighbour_distances)) if neighbour_distances.size else cfg.GAP_DENSITY_REFERENCE_UM\n            result[node_id] = spacing\n\n        density_cache[t] = result\n        stats["gap_density_nodes_scored"] += len(result)\n        return result\n\n    effective_gap_max = min(cfg.GAP_CLOSE_MAX_GAP, 1)\n    stats["gap_close_effective_max_gap"] = effective_gap_max\n    for gap in range(1, effective_gap_max + 1):\n        for t, end_ids in sorted(ends_by_t.items()):\n            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]\n            if not end_ids or not start_ids:\n                continue\n\n            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]\n            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]\n            threshold_um = cfg.GAP_CLOSE_UM * (gap + 1)\n            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)\n            adaptive_threshold = np.full_like(d, threshold_um)\n\n            source_spacing = frame_local_spacing(t)\n            target_spacing = frame_local_spacing(t + gap + 1)\n\n            for i, ep in enumerate(end_points):\n                for j, sp in enumerate(start_points):\n                    d[i, j] = point_distance_um(ep, sp)\n\n                    if cfg.GAP_DENSITY_ADAPTIVE:\n                        local_spacing = 0.5 * (\n                            source_spacing.get(end_ids[i], cfg.GAP_DENSITY_REFERENCE_UM)\n                            + target_spacing.get(start_ids[j], cfg.GAP_DENSITY_REFERENCE_UM)\n                        )\n                        step_delta = float(\n                            np.clip(\n                                cfg.GAP_DENSITY_GAIN * (local_spacing - cfg.GAP_DENSITY_REFERENCE_UM),\n                                -cfg.GAP_DENSITY_MAX_STEP_DELTA_UM,\n                                cfg.GAP_DENSITY_MAX_STEP_DELTA_UM,\n                            )\n                        )\n                        adaptive_threshold[i, j] = threshold_um + step_delta * (gap + 1)\n                        stats["gap_density_step_delta_milli_sum"] += int(round(1000.0 * step_delta))\n\n            base_allowed = d <= threshold_um\n            adaptive_allowed = d <= adaptive_threshold\n\n            stats["gap_density_candidates_expanded"] += int((adaptive_allowed & ~base_allowed).sum())\n            stats["gap_density_candidates_restricted"] += int((base_allowed & ~adaptive_allowed).sum())\n            stats["gap_candidates"] += int(adaptive_allowed.sum())\n\n            if not np.isfinite(d).any():\n                continue\n\n            max_threshold = float(np.max(adaptive_threshold))\n            big = max_threshold * 1000.0 + 1.0\n            cost = np.where(adaptive_allowed, d, big)\n            row_ind, col_ind = linear_sum_assignment(cost)\n\n            for r, c in zip(row_ind, col_ind, strict=True):\n                if not adaptive_allowed[r, c]:\n                    continue\n                if not base_allowed[r, c]:\n                    stats["gap_density_selected_outside_base"] += 1\n                source_id = end_ids[int(r)]\n                target_id = start_ids[int(c)]\n                if source_id in outgoing or target_id in used_starts:\n                    continue\n\n                source = nodes_by_id[source_id]\n                target = nodes_by_id[target_id]\n                mid_t = int(source["t"]) + gap\n                mid_point = (\n                    (float(source["z"]) + float(target["z"])) / 2.0,\n                    (float(source["y"]) + float(target["y"])) / 2.0,\n                    (float(source["x"]) + float(target["x"])) / 2.0,\n                )\n\n                middle_id: int | None = None\n                middle_reused = False\n                if cfg.GAP_CLOSE_REUSE_EXISTING:\n                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]\n                    if candidates:\n                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]\n                        best_idx = int(np.argmin(distances))\n                        if distances[best_idx] <= cfg.GAP_CLOSE_REUSE_UM:\n                            middle_id = candidates[best_idx]\n                            middle_reused = True\n\n                if middle_id is None:\n                    if synthetic_added >= max_synthetic:\n                        stats["gap_skipped_node_cap"] += 1\n                        continue\n                    middle_id = next_id\n                    next_id += 1\n                    refined_point = refine_synthetic_midpoint(cfg, dataset, mid_t, mid_point, frame_cache, stats)\n                    nodes_by_id[middle_id] = {\n                        "node_id": middle_id,\n                        "t": mid_t,\n                        "z": refined_point[0],\n                        "y": refined_point[1],\n                        "x": refined_point[2],\n                        "gap_synthetic": 1,\n                    }\n                    synthetic_added += 1\n                    stats["gap_inserted_synthetic"] += 1\n\n                middle = nodes_by_id[middle_id]\n                gap_span_um = float(d[r, c])\n                marginal_gap = gap_span_um >= cfg.DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM\n                synthetic_middle = int(middle.get("gap_synthetic", 0)) == 1\n                requires_center_confirmation = (\n                    cfg.DEEPCENTER_GAP_VETO and marginal_gap and synthetic_middle\n                )\n                if cfg.DEEPCENTER_GAP_VETO and not marginal_gap:\n                    stats["deepcenter_gap_bypassed_strong_motion"] += 1\n                elif cfg.DEEPCENTER_GAP_VETO and not synthetic_middle:\n                    stats["deepcenter_gap_bypassed_observed_node"] += 1\n                if requires_center_confirmation and not deepcenter_accept_repair_point(\n                    cfg,\n                    dataset,\n                    mid_t,\n                    node_point(middle),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "gap",\n                    cfg.DEEPCENTER_GAP_THRESHOLD,\n                ):\n                    if int(middle.get("gap_synthetic", 0)) == 1:\n                        nodes_by_id.pop(middle_id, None)\n                        synthetic_added = max(0, synthetic_added - 1)\n                        stats["gap_inserted_synthetic"] = max(0, stats["gap_inserted_synthetic"] - 1)\n                    continue\n                if middle_reused:\n                    used_isolated.add(middle_id)\n                    stats["gap_reused_existing"] += 1\n\n                e1 = {\n                    "source_id": source_id,\n                    "target_id": middle_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(source, middle),\n                    "gap_closed": 1,\n                }\n                e2 = {\n                    "source_id": middle_id,\n                    "target_id": target_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(middle, target),\n                    "gap_closed": 1,\n                }\n                new_edges.extend([e1, e2])\n                outgoing.add(source_id)\n                incoming.add(middle_id)\n                outgoing.add(middle_id)\n                incoming.add(target_id)\n                used_starts.add(target_id)\n                stats["gap_pairs_selected"] += 1\n                stats["gap_added_edges"] += 2\n\n    if new_edges:\n        edges = [*edges, *new_edges]\n    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]\n    return nodes_by_id, edges\n\n\ndef recover_strict_gap2(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not cfg.OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    predecessor = _single_predecessor_map(edges)\n    successor = _single_successor_map(edges)\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n\n    cap = min(cfg.GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * cfg.GAP2_MAX_LINKS_FRAC))))\n    proposals: list[tuple[float, int, int, int, float]] = []\n\n    def pos_um(node_id: int) -> np.ndarray:\n        return position_um(nodes_by_id[node_id])\n\n    for t, end_ids in sorted(ends_by_t.items()):\n        start_ids = starts_by_t.get(t + 3, [])\n        if not end_ids or not start_ids:\n            continue\n        for end_id in end_ids:\n            end_pos = pos_um(end_id)\n            for start_id in start_ids:\n                start_pos = pos_um(start_id)\n                dist = float(np.linalg.norm(start_pos - end_pos))\n                if dist > cfg.GAP2_MAX_TOTAL_UM or dist / 3.0 > cfg.GAP2_MAX_STEP_UM:\n                    continue\n                step = (start_pos - end_pos) / 3.0\n                context_penalty = 0.0\n                if cfg.GAP2_REQUIRE_CONTEXT:\n                    ok_context = False\n                    prev_id = predecessor.get(end_id)\n                    if prev_id is not None:\n                        prev_step = end_pos - pos_um(prev_id)\n                        prev_norm = float(np.linalg.norm(prev_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if prev_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    next_id = successor.get(start_id)\n                    if next_id is not None:\n                        next_step = pos_um(next_id) - start_pos\n                        next_norm = float(np.linalg.norm(next_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if next_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    if not ok_context:\n                        continue\n                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))\n\n    proposals.sort(key=lambda item: item[0])\n    stats["gap2_candidates"] = len(proposals)\n    if not proposals:\n        return nodes_by_id, edges\n\n    selected: list[tuple[float, int, int, int, float]] = []\n    used_ends: set[int] = set()\n    used_starts: set[int] = set()\n    per_frame_count: dict[int, int] = {}\n    for proposal in proposals:\n        if len(selected) >= cap:\n            stats["gap2_skipped_cap"] += 1\n            break\n        _, end_id, start_id, t, _ = proposal\n        if end_id in used_ends or start_id in used_starts:\n            continue\n        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * cfg.GAP2_FRAME_FRAC_CAP)))\n        if per_frame_count.get(t, 0) >= frame_cap:\n            continue\n        selected.append(proposal)\n        used_ends.add(end_id)\n        used_starts.add(start_id)\n        per_frame_count[t] = per_frame_count.get(t, 0) + 1\n\n    if not selected:\n        return nodes_by_id, edges\n\n    next_id = next_node_id(nodes_by_id)\n    frame_cache: dict[int, np.ndarray] = {}\n    new_edges: list[dict[str, object]] = []\n    for _, end_id, start_id, _t, _ in selected:\n        source = nodes_by_id[end_id]\n        target = nodes_by_id[start_id]\n        previous_id = end_id\n        inserted_ids: list[int] = []\n        for k in (1, 2):\n            frac = k / 3.0\n            mid_t = int(source["t"]) + k\n            midpoint = (\n                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,\n                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,\n                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,\n            )\n            refined_point = refine_synthetic_midpoint(cfg, dataset, mid_t, midpoint, frame_cache, stats)\n            node_id = next_id\n            next_id += 1\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": mid_t,\n                "z": refined_point[0],\n                "y": refined_point[1],\n                "x": refined_point[2],\n            }\n            inserted_ids.append(node_id)\n            current = nodes_by_id[node_id]\n            new_edges.append({\n                "source_id": previous_id,\n                "target_id": node_id,\n                "edge_prob": None,\n                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),\n                "gap2_recovered": 1,\n            })\n            previous_id = node_id\n        new_edges.append({\n            "source_id": previous_id,\n            "target_id": start_id,\n            "edge_prob": None,\n            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),\n            "gap2_recovered": 1,\n        })\n        stats["gap2_pairs_selected"] += 1\n        stats["gap2_added_nodes"] += len(inserted_ids)\n        stats["gap2_added_edges"] += 3\n\n    return nodes_by_id, [*edges, *new_edges]\n\n\ndef filter_short_track_components(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    consensus_proof_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not cfg.OUTPUT_FILTER_SHORT_TRACKS or cfg.OUTPUT_MIN_TRACK_LEN <= 1 or not edges:\n        return nodes_by_id, edges\n\n    parent = {node_id: node_id for node_id in nodes_by_id}\n\n    def find(node_id: int) -> int:\n        while parent[node_id] != node_id:\n            parent[node_id] = parent[parent[node_id]]\n            node_id = parent[node_id]\n        return node_id\n\n    def union(a: int, b: int) -> None:\n        if a not in parent or b not in parent:\n            return\n        ra = find(a)\n        rb = find(b)\n        if ra != rb:\n            parent[ra] = rb\n\n    out_count: dict[int, int] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        union(source_id, target_id)\n        out_count[source_id] = out_count.get(source_id, 0) + 1\n\n    components: dict[int, list[int]] = {}\n    for node_id in nodes_by_id:\n        components.setdefault(find(node_id), []).append(node_id)\n\n    component_edges: dict[int, list[dict[str, object]]] = {root: [] for root in components}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        if source_id in parent and target_id in parent:\n            component_edges.setdefault(find(source_id), []).append(edge)\n\n    proof_pairs = (set(pair for pairs in consensus_proof_pairs_by_t.values() for pair in pairs)\n                   if consensus_proof_pairs_by_t is not None else set())\n\n    keep: set[int] = set()\n    for _root, members in components.items():\n        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)\n        if len(members) >= cfg.OUTPUT_MIN_TRACK_LEN or (cfg.OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):\n            keep.update(members)\n\n    if consensus_proof_pairs_by_t is not None:\n        for root, members in components.items():\n            if len(members) != 5 or set(members) & keep:\n                continue\n            member_set = set(members)\n            c_edges = component_edges.get(root, [])\n            if len(c_edges) != 4 or any(int(e["source_id"]) == int(e["target_id"]) for e in c_edges):\n                continue\n            ordered = sorted(members, key=lambda node_id: (int(nodes_by_id[node_id]["t"]), node_id))\n            if [int(nodes_by_id[node_id]["t"]) for node_id in ordered] != list(range(int(nodes_by_id[ordered[0]]["t"]), int(nodes_by_id[ordered[0]]["t"]) + 5)):\n                continue\n            expected = {(ordered[i], ordered[i + 1]) for i in range(4)}\n            actual = {(int(e["source_id"]), int(e["target_id"])) for e in c_edges}\n            if actual == expected and actual <= proof_pairs:\n                keep.update(member_set)\n                stats["short_track_rescue_components"] = stats.get("short_track_rescue_components", 0) + 1\n                stats["short_track_rescue_nodes"] = stats.get("short_track_rescue_nodes", 0) + 5\n\n    if not keep:\n        stats["short_track_filter_skipped_all"] += 1\n        return nodes_by_id, edges\n\n    removed_before_rescue = len(nodes_by_id) - len(keep)\n    if removed_before_rescue <= 0:\n        return nodes_by_id, edges\n\n    if cfg.ADAPTIVE_SHORT_TRACK_RESCUE:\n        removed_frac = removed_before_rescue / max(len(nodes_by_id), 1)\n        if removed_frac >= cfg.SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC:\n            budget = min(\n                cfg.SHORT_TRACK_RESCUE_MAX_NODES_ABS,\n                max(0, int(round(len(nodes_by_id) * cfg.SHORT_TRACK_RESCUE_MAX_NODES_FRAC))),\n            )\n            stats["short_track_rescue_triggered"] = 1\n            stats["short_track_rescue_budget"] = budget\n            proposals: list[tuple[float, int, float, int, list[int]]] = []\n            for root, members in components.items():\n                if set(members) & keep:\n                    continue\n                if len(members) < cfg.SHORT_TRACK_RESCUE_MIN_LEN or len(members) >= cfg.OUTPUT_MIN_TRACK_LEN:\n                    continue\n                c_edges = component_edges.get(root, [])\n                if not c_edges:\n                    continue\n                probs: list[float] = []\n                dists: list[float] = []\n                for edge in c_edges:\n                    try:\n                        prob = float(edge.get("edge_prob", 0.0))\n                    except (TypeError, ValueError):\n                        prob = 0.0\n                    if np.isfinite(prob):\n                        probs.append(prob)\n                    try:\n                        dist = float(edge.get("distance_um", np.nan))\n                    except (TypeError, ValueError):\n                        dist = np.nan\n                    if np.isfinite(dist):\n                        dists.append(dist)\n                mean_prob = float(np.mean(probs)) if probs else 0.0\n                mean_dist = float(np.mean(dists)) if dists else float("inf")\n                if mean_prob < cfg.SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB:\n                    continue\n                if mean_dist > cfg.SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM:\n                    continue\n                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)\n                proposals.append((score, len(members), mean_prob, root, members))\n            proposals.sort(reverse=True)\n            rescued_nodes = 0\n            rescued_components = 0\n            for _, size, _, _, members in proposals:\n                if budget <= 0 or rescued_nodes + size > budget:\n                    continue\n                keep.update(members)\n                rescued_nodes += size\n                rescued_components += 1\n            stats["short_track_rescue_components"] = rescued_components\n            stats["short_track_rescue_nodes"] = rescued_nodes\n\n    removed_nodes = len(nodes_by_id) - len(keep)\n    if removed_nodes <= 0:\n        return nodes_by_id, edges\n\n    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}\n    kept_edges = [\n        edge for edge in edges if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes\n    ]\n    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))\n    stats["short_track_nodes_removed"] = removed_nodes\n    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)\n    return kept_nodes, kept_edges\n\n\ndef linefit_smooth_output_graph(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> dict[int, dict[str, object]]:\n    """Smooth linear track interiors without changing graph topology."""\n    if not cfg.OUTPUT_LINEFIT_SMOOTH or cfg.OUTPUT_LINEFIT_WEIGHT <= 0 or cfg.OUTPUT_LINEFIT_WINDOW <= 0 or not edges:\n        return nodes_by_id\n\n    predecessor: dict[int, list[int]] = {}\n    successor: dict[int, list[int]] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        source = nodes_by_id.get(source_id)\n        target = nodes_by_id.get(target_id)\n        if source is None or target is None:\n            continue\n        if int(target["t"]) != int(source["t"]) + 1:\n            continue\n        successor.setdefault(source_id, []).append(target_id)\n        predecessor.setdefault(target_id, []).append(source_id)\n\n    original_pos = {\n        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)\n        for node_id, node in nodes_by_id.items()\n    }\n    updated_pos: dict[int, np.ndarray] = {}\n    weight = float(np.clip(cfg.OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))\n\n    for node_id in sorted(nodes_by_id):\n        neighbourhood: list[tuple[int, int]] = [(0, node_id)]\n\n        current = node_id\n        for step in range(1, cfg.OUTPUT_LINEFIT_WINDOW + 1):\n            prev_ids = predecessor.get(current, [])\n            if len(prev_ids) != 1:\n                break\n            current = prev_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((-step, current))\n\n        current = node_id\n        for step in range(1, cfg.OUTPUT_LINEFIT_WINDOW + 1):\n            next_ids = successor.get(current, [])\n            if len(next_ids) != 1:\n                break\n            current = next_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((step, current))\n\n        if len(neighbourhood) < 3:\n            stats["linefit_skipped_nodes"] += 1\n            continue\n\n        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)\n        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])\n        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)\n        if not np.isfinite(fitted).all():\n            stats["linefit_skipped_nodes"] += 1\n            continue\n        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted\n\n    for node_id, pos in updated_pos.items():\n        nodes_by_id[node_id]["z"] = float(pos[0])\n        nodes_by_id[node_id]["y"] = float(pos[1])\n        nodes_by_id[node_id]["x"] = float(pos[2])\n\n    stats["linefit_smoothed_nodes"] = len(updated_pos)\n    return nodes_by_id\n', 'src/biohub/public_postproc/pipeline.py': '"""Orchestrates the post-processing stack, ported verbatim from the notebook\'s\n``filter_output_graph`` function and the CSV-writing loop that calls it once\nper prediction ``.geff``.\n\nAlso provides a checkpoint split at the linefit-smoothing boundary\n(:func:`filter_output_graph_pre_linefit` / :func:`save_prelinefit_checkpoint`\n/ :func:`run_relinefit`): the stack up to and including\n``filter_short_track_components`` fixes final graph *topology* (node/edge\nsets); ``linefit_smooth_output_graph`` only nudges node coordinates. Sweeping\n``BIOHUB_OUTPUT_LINEFIT_*`` therefore never needs to redo the expensive\nmotion-relink / gap-close / safe-division passes.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport pickle\nimport struct\nimport tempfile\nfrom collections.abc import Callable, Mapping, Sequence\nfrom pathlib import Path\nfrom types import MappingProxyType\n\nimport numpy as np\nimport pandas as pd\n\nfrom biohub.io import load_geff_graph\nfrom biohub.public_postproc.config import PostprocConfig\nfrom biohub.public_postproc.csv_out import NodeSerializer, SubmissionCsvWriter, write_run_stats\nfrom biohub.public_postproc.deepcenter import load_deepcenter_veto_detector\nfrom biohub.public_postproc.divisions import (\n    _TWIN_COUNTER_KEYS,\n    _TWIN_METADATA_MAX_DEPTH,\n    _TWIN_VALIDATION_REASONS,\n    TwinDebugRecord,\n    TwinDeepCenterDecision,\n    TwinEdgeRecord,\n    TwinFrozenArray,\n    TwinFrozenBuffer,\n    TwinFrozenDType,\n    TwinFrozenMapping,\n    TwinFrozenNumpyScalar,\n    TwinFrozenStructuredScalar,\n    TwinPlan,\n    TwinPlannedEdge,\n    TwinSnapshotNode,\n    add_safe_divisions_postlink,\n    apply_twin_only_v1_plan,\n    plan_twin_only_v1,\n    score_twin_deepcenter,\n)\nfrom biohub.public_postproc.frames import refine_all_centroids\nfrom biohub.public_postproc.geometry import edge_distance_um, edge_sort_key\nfrom biohub.public_postproc.graph_ops import (\n    close_single_frame_gaps,\n    filter_short_track_components,\n    linefit_smooth_output_graph,\n    motion_relink_edges,\n    prepare_consensus_reservations,\n    recover_strict_gap2,\n)\n\nCHECKPOINT_MANIFEST_NAME = "manifest.json"\n\nTwinPlanHook = Callable[[str, TwinPlan | None], None]\nRawStatsHook = Callable[[Mapping[str, object]], None]\nDatasetHook = Callable[[int, str], None]\nDeepCenterLoader = Callable[[PostprocConfig], dict[str, object] | None]\n\n_TWIN_R2_COUNTER_KEYS = (\n    "steal_twin_mutations_applied",\n    "steal_twin_pure_nodes",\n    "steal_twin_pure_edges",\n    "steal_twin_pure_fork_sources",\n    "steal_twin_pure_edge_symmetric_difference",\n    "steal_twin_geometry_edges_removed_observed",\n    "steal_twin_prune_nodes_removed_observed",\n    "steal_twin_prune_edges_removed_observed",\n    "steal_twin_short_nodes_removed_observed",\n    "steal_twin_short_edges_removed_observed",\n    "steal_twin_final_nodes",\n    "steal_twin_final_edges",\n    "steal_twin_final_fork_sources",\n    "steal_twin_linefit_coordinate_changed_nodes_observed",\n)\n\n\ndef new_stats() -> dict[str, int]:\n    """Zero-initialised counters filter_output_graph may increment.\n\n    Verbatim from the notebook cell (``gap_close_effective_max_gap`` is\n    deliberately absent: it is only set if ``close_single_frame_gaps``\n    actually runs, same as the source).\n\n    The notebook declares ``safe_division_geometric_candidates``,\n    ``safe_division_mutual_nn_rejected`` and\n    ``safe_division_divergence_rejected`` but never increments them (its\n    telemetry always prints 0 for all three). The port keeps the keys in\n    the notebook\'s position and fills them truthfully inside\n    ``add_safe_divisions_postlink`` -- the documented notebook-only\n    broken-counter exception of the E23 parity contract.\n    """\n    stats = {\n        "raw_edges": 0,\n        "dropped_nonconsecutive_edges": 0,\n        "dropped_long_edges": 0,\n        "dropped_multi_parent_edges": 0,\n        "dropped_multi_child_edges": 0,\n        "dropped_division_edges": 0,\n        "gap_candidates": 0,\n        "gap_pairs_selected": 0,\n        "gap_reused_existing": 0,\n        "gap_inserted_synthetic": 0,\n        "gap_added_nodes": 0,\n        "gap_added_edges": 0,\n        "gap_skipped_node_cap": 0,\n        "gap_density_nodes_scored": 0,\n        "gap_density_candidates_expanded": 0,\n        "gap_density_candidates_restricted": 0,\n        "gap_density_selected_outside_base": 0,\n        "gap_density_step_delta_milli_sum": 0,\n        "gap_refined_synthetic": 0,\n        "gap_refine_failed": 0,\n        "gap_refine_rejected_shift": 0,\n        "centroid_refine_examined": 0,\n        "centroid_refine_moved": 0,\n        "centroid_refine_no_signal": 0,\n        "centroid_refine_rejected_shift": 0,\n        "pruned_isolated_nodes": 0,\n        "motion_relink_edges": 0,\n        "motion_relink_tight_edges": 0,\n        "motion_relink_relaxed_edges": 0,\n        "motion_relink_frames": 0,\n        "motion_relink_replaced_raw_edges": 0,\n        "motion_relink_fallback_raw": 0,\n        "motion_relink_skipped_large_frame": 0,\n        "gap2_candidates": 0,\n        "gap2_pairs_selected": 0,\n        "gap2_added_nodes": 0,\n        "gap2_added_edges": 0,\n        "gap2_skipped_cap": 0,\n        "safe_division_candidates": 0,\n        "safe_division_geometric_candidates": 0,\n        "safe_divisions_added": 0,\n        "safe_division_skipped_cap": 0,\n        "safe_division_mutual_nn_rejected": 0,\n        "safe_division_divergence_rejected": 0,\n        "deepcenter_gap_checked": 0,\n        "deepcenter_gap_bypassed_strong_motion": 0,\n        "deepcenter_gap_bypassed_observed_node": 0,\n        "deepcenter_gap_accepted": 0,\n        "deepcenter_gap_rejected": 0,\n        "deepcenter_gap_missing": 0,\n        "deepcenter_safe_div_checked": 0,\n        "deepcenter_safe_div_accepted": 0,\n        "deepcenter_safe_div_rejected": 0,\n        "deepcenter_safe_div_missing": 0,\n        "short_track_components_removed": 0,\n        "short_track_nodes_removed": 0,\n        "short_track_edges_removed": 0,\n        "short_track_filter_skipped_all": 0,\n        "short_track_rescue_triggered": 0,\n        "short_track_rescue_components": 0,\n        "short_track_rescue_nodes": 0,\n        "short_track_rescue_budget": 0,\n        "linefit_smoothed_nodes": 0,\n        "linefit_skipped_nodes": 0,\n    }\n    stats.update(dict.fromkeys(_TWIN_COUNTER_KEYS, 0))\n    stats.update(dict.fromkeys(_TWIN_R2_COUNTER_KEYS, 0))\n    return stats\n\n\ndef _require_steal_twin_r1_dry_run(cfg: PostprocConfig) -> None:\n    if not cfg.OUTPUT_STEAL_TWIN_REWIRE:\n        return\n    frozen = {\n        "STEAL_TWIN_MODE": "twin_only_v1",\n        "STEAL_TWIN_PARENT_MAX_UM": 8.0,\n        "STEAL_TWIN_EXISTING_CHILD_MAX_UM": 10.0,\n        "STEAL_TWIN_SISTER_MIN_UM": 5.5,\n        "STEAL_TWIN_SISTER_MAX_UM": 11.0,\n        "STEAL_TWIN_DIVERGE_UM": 2.25,\n        "STEAL_TWIN_TWIN_MAX_UM": 5.0,\n        "STEAL_TWIN_REQUIRE_TWO_SUCCESSORS": True,\n        "STEAL_TWIN_REJECT_SYNTHETIC": True,\n        "STEAL_TWIN_DEEPCENTER_VETO": True,\n        "STEAL_TWIN_FRAME_CAP_ABS": 1,\n        "STEAL_TWIN_VIDEO_CAP_ABS": 2,\n        "STEAL_TWIN_DEBUG_MAX_RECORDS": 200,\n    }\n    if any(\n        type(getattr(cfg, name)) is not type(expected) or getattr(cfg, name) != expected\n        for name, expected in frozen.items()\n    ):\n        raise RuntimeError("invalid twin_only_v1 mode/profile lock")\n\n\ndef _twin_float_plain(value: float) -> float | dict[str, str]:\n    if math.isfinite(value):\n        return value\n    if math.isnan(value):\n        label = "nan"\n    elif value > 0:\n        label = "+inf"\n    else:\n        label = "-inf"\n    return {\n        "__twin_type__": "float",\n        "value": label,\n        "bits_hex": struct.pack(">d", value).hex(),\n    }\n\n\ndef _twin_frozen_value_plain(value: object) -> object:\n    return _twin_frozen_value_plain_inner(value, 0, set())\n\n\ndef _twin_frozen_value_plain_inner(value: object, depth: int, active: set[int]) -> object:\n    if depth > _TWIN_METADATA_MAX_DEPTH:\n        raise TypeError("twin metadata exceeds maximum depth")\n    value_type = type(value)\n    if value is None or value_type in (bool, int, str):\n        return value\n    if value_type is float:\n        return _twin_float_plain(value)\n    if value_type is complex:\n        return {\n            "__twin_type__": "complex",\n            "real": _twin_float_plain(value.real),\n            "imag": _twin_float_plain(value.imag),\n        }\n    if value_type is bytes:\n        return {"__twin_type__": "bytes", "hex": value.hex()}\n    recursive_types = (\n        tuple,\n        frozenset,\n        TwinFrozenMapping,\n        TwinFrozenDType,\n        TwinFrozenStructuredScalar,\n        TwinFrozenNumpyScalar,\n        TwinFrozenArray,\n        TwinFrozenBuffer,\n    )\n    if value_type not in recursive_types:\n        raise TypeError(f"unsupported frozen twin value type: {value_type.__qualname__}")\n    value_id = id(value)\n    if value_id in active:\n        raise TypeError("cyclic frozen twin value")\n    active.add(value_id)\n    try:\n        return _twin_frozen_value_plain_recursive(value, depth, active)\n    finally:\n        active.remove(value_id)\n\n\ndef _twin_frozen_value_plain_recursive(value: object, depth: int, active: set[int]) -> object:\n    value_type = type(value)\n    if value_type is tuple:\n        return {\n            "__twin_type__": "tuple",\n            "items": [\n                _twin_frozen_value_plain_inner(item, depth + 1, active) for item in value\n            ],\n        }\n    if value_type is frozenset:\n        items = [\n            _twin_frozen_value_plain_inner(item, depth + 1, active) for item in value\n        ]\n        items.sort(\n            key=lambda item: json.dumps(\n                item,\n                sort_keys=True,\n                separators=(",", ":"),\n                allow_nan=False,\n            )\n        )\n        return {"__twin_type__": "frozenset", "items": items}\n    if value_type is TwinFrozenMapping:\n        return {\n            "__twin_type__": "mapping",\n            "items": [\n                [\n                    _twin_frozen_value_plain_inner(key, depth + 1, active),\n                    _twin_frozen_value_plain_inner(item, depth + 1, active),\n                ]\n                for key, item in value.items_snapshot\n            ],\n        }\n    if value_type is TwinFrozenDType:\n        return {\n            "__twin_type__": "numpy_dtype",\n            "string": value.string,\n            "descriptor": _twin_frozen_value_plain_inner(value.descriptor, depth + 1, active),\n            "metadata": (\n                None\n                if value.metadata is None\n                else _twin_frozen_value_plain_inner(value.metadata, depth + 1, active)\n            ),\n            "itemsize": value.itemsize,\n            "alignment": value.alignment,\n            "byteorder": value.byteorder,\n            "names": None if value.names is None else list(value.names),\n            "hasobject": value.hasobject,\n            "aligned_struct": value.aligned_struct,\n        }\n    if value_type is TwinFrozenStructuredScalar:\n        return {\n            "__twin_type__": "numpy_structured_scalar",\n            "fields": [\n                [name, _twin_frozen_value_plain_inner(item, depth + 1, active)]\n                for name, item in value.fields\n            ],\n        }\n    if value_type is TwinFrozenNumpyScalar:\n        return {\n            "__twin_type__": "numpy_scalar",\n            "dtype": _twin_frozen_value_plain_inner(value.dtype, depth + 1, active),\n            "content": value.content.hex(),\n        }\n    if value_type is TwinFrozenArray:\n        plain = {\n            "__twin_type__": "numpy_array",\n            "dtype": _twin_frozen_value_plain_inner(value.dtype, depth + 1, active),\n            "shape": list(value.shape),\n            "strides": list(value.strides),\n            "c_contiguous": value.c_contiguous,\n            "f_contiguous": value.f_contiguous,\n            "object_content": value.object_content,\n        }\n        if type(value.content) is bytes:\n            plain["content_hex"] = value.content.hex()\n        elif type(value.content) is tuple:\n            plain["content"] = {\n                "__twin_type__": "tuple",\n                "items": [\n                    _twin_frozen_value_plain_inner(item, depth + 1, active)\n                    for item in value.content\n                ],\n            }\n        else:\n            raise TypeError("unsupported TwinFrozenArray content type")\n        return plain\n    if value_type is TwinFrozenBuffer:\n        return {\n            "__twin_type__": "buffer",\n            "kind": value.kind,\n            "format": value.format,\n            "itemsize": value.itemsize,\n            "shape": list(value.shape),\n            "strides": list(value.strides),\n            "readonly": value.readonly,\n            "content_hex": value.content.hex(),\n        }\n    raise TypeError(f"unsupported frozen twin value type: {value_type.__qualname__}")\n\n\ndef _twin_debug_record_plain(record: TwinDebugRecord) -> dict[str, object]:\n    return {\n        "dataset": record.dataset,\n        "decision": record.decision,\n        "reason": record.reason,\n        "p": record.p,\n        "q": record.q,\n        "a": record.a,\n        "b": record.b,\n        "a2": record.a2,\n        "b2": record.b2,\n        "sort_key": list(record.sort_key),\n        "d_pq": record.d_pq,\n        "d_pa": record.d_pa,\n        "d_pb": record.d_pb,\n        "d_ab": record.d_ab,\n        "d_a2b2": record.d_a2b2,\n        "divergence_growth": record.divergence_growth,\n        "raw_deepcenter_score": record.raw_deepcenter_score,\n        "deepcenter_threshold": record.deepcenter_threshold,\n        "deepcenter_decision": {\n            "accepted": record.deepcenter_decision.accepted,\n            "raw_score": record.deepcenter_decision.raw_score,\n            "reason": record.deepcenter_decision.reason,\n        },\n        "removed_edge": {\n            "source_id": record.removed_edge.source_id,\n            "target_id": record.removed_edge.target_id,\n            "metadata": _twin_frozen_value_plain(record.removed_edge.metadata),\n        },\n        "planned_edge": {\n            "source_id": record.planned_edge.source_id,\n            "target_id": record.planned_edge.target_id,\n            "distance_um": record.planned_edge.distance_um,\n            "edge_prob": None,\n        },\n    }\n\n\ndef _twin_candidate_plain(candidate: object) -> dict[str, object]:\n    return {\n        "frame": candidate.frame,\n        "p": candidate.p,\n        "q": candidate.q,\n        "a": candidate.a,\n        "b": candidate.b,\n        "a2": candidate.a2,\n        "b2": candidate.b2,\n        "d_pq": candidate.d_pq,\n        "d_pa": candidate.d_pa,\n        "d_pb": candidate.d_pb,\n        "d_ab": candidate.d_ab,\n        "d_a2b2": candidate.d_a2b2,\n        "divergence_growth": candidate.divergence_growth,\n        "raw_deepcenter_score": candidate.raw_deepcenter_score,\n        "deepcenter_decision": {\n            "accepted": candidate.deepcenter_decision.accepted,\n            "raw_score": candidate.deepcenter_decision.raw_score,\n            "reason": candidate.deepcenter_decision.reason,\n        },\n        "sort_key": list(candidate.sort_key),\n        "removed_edge": {\n            "source_id": candidate.removed_edge.source_id,\n            "target_id": candidate.removed_edge.target_id,\n            "metadata": _twin_frozen_value_plain(candidate.removed_edge.metadata),\n        },\n        "planned_edge": {\n            "source_id": candidate.planned_edge.source_id,\n            "target_id": candidate.planned_edge.target_id,\n            "distance_um": candidate.planned_edge.distance_um,\n            "edge_prob": candidate.planned_edge.edge_prob,\n        },\n    }\n\n\ndef twin_plan_plain(plan: TwinPlan) -> dict[str, object]:\n    """Losslessly expose every validated plan field for the production hook."""\n    plan = _validate_twin_plan_failure_envelope(plan)\n    return {\n        "validation_reason": plan.validation_reason,\n        "nodes": [\n            {\n                "node_id": node.node_id,\n                "t": node.t,\n                "z": node.z,\n                "y": node.y,\n                "x": node.x,\n                "gap_synthetic": node.gap_synthetic,\n            }\n            for node in plan.nodes\n        ],\n        "edges": [\n            {\n                "source_id": edge.source_id,\n                "target_id": edge.target_id,\n                "input_position": edge.input_position,\n                "metadata": _twin_frozen_value_plain(edge.metadata),\n            }\n            for edge in plan.edges\n        ],\n        "candidates": [_twin_candidate_plain(candidate) for candidate in plan.candidates],\n        "accepted_candidates": [\n            _twin_candidate_plain(candidate) for candidate in plan.accepted_candidates\n        ],\n        "decisions": [\n            {\n                "candidate": _twin_candidate_plain(decision.candidate),\n                "accepted": decision.accepted,\n                "reason": decision.reason,\n            }\n            for decision in plan.decisions\n        ],\n        "counters": _twin_frozen_value_plain(plan.counters),\n        "debug_records": [_twin_debug_record_plain(record) for record in plan.debug_records],\n    }\n\n\ndef _twin_record_type_error(index: int, field: str) -> TypeError:\n    return TypeError(f"twin debug record {index}: invalid type for {field}")\n\n\ndef _twin_record_value_error(index: int, field: str) -> ValueError:\n    return ValueError(f"twin debug record {index}: invalid value for {field}")\n\n\ndef _validate_twin_debug_record(\n    record: object,\n    index: int,\n    expected_dataset: str | None,\n    *,\n    first: bool,\n) -> str | None:\n    if type(record) is not TwinDebugRecord:\n        raise _twin_record_type_error(index, "record")\n    if record.dataset is not None and type(record.dataset) is not str:\n        raise _twin_record_type_error(index, "dataset")\n    if not first and record.dataset != expected_dataset:\n        raise _twin_record_value_error(index, "dataset")\n    if type(record.decision) is not str:\n        raise _twin_record_type_error(index, "decision")\n    if record.decision == "accepted":\n        if record.reason is not None:\n            raise _twin_record_value_error(index, "decision/reason")\n    elif record.decision == "rejected":\n        if type(record.reason) is not str:\n            raise _twin_record_type_error(index, "reason")\n        if record.reason not in ("conflict", "frame_cap", "video_cap"):\n            raise _twin_record_value_error(index, "reason")\n    else:\n        raise _twin_record_value_error(index, "decision")\n\n    roles = (record.p, record.q, record.a, record.b, record.a2, record.b2)\n    if any(type(value) is not int for value in roles):\n        raise _twin_record_type_error(index, "roles")\n    if type(record.sort_key) is not tuple:\n        raise _twin_record_type_error(index, "sort_key")\n    if len(record.sort_key) != 9:\n        raise _twin_record_value_error(index, "sort_key length")\n    if any(type(value) is not float for value in record.sort_key[:3]):\n        raise _twin_record_type_error(index, "sort_key numeric prefix")\n    if not all(math.isfinite(value) for value in record.sort_key[:3]):\n        raise _twin_record_value_error(index, "sort_key numeric prefix")\n    if any(type(value) is not int for value in record.sort_key[3:]):\n        raise _twin_record_type_error(index, "sort_key role suffix")\n    if record.sort_key[3:] != roles:\n        raise _twin_record_value_error(index, "sort_key role suffix")\n\n    float_fields = (\n        ("d_pq", record.d_pq),\n        ("d_pa", record.d_pa),\n        ("d_pb", record.d_pb),\n        ("d_ab", record.d_ab),\n        ("d_a2b2", record.d_a2b2),\n        ("divergence_growth", record.divergence_growth),\n        ("raw_deepcenter_score", record.raw_deepcenter_score),\n        ("deepcenter_threshold", record.deepcenter_threshold),\n    )\n    for field, value in float_fields:\n        if type(value) is not float:\n            raise _twin_record_type_error(index, field)\n        if not math.isfinite(value):\n            raise _twin_record_value_error(index, field)\n    if record.deepcenter_threshold != 0.12:\n        raise _twin_record_value_error(index, "deepcenter_threshold")\n\n    decision = record.deepcenter_decision\n    if type(decision) is not TwinDeepCenterDecision:\n        raise _twin_record_type_error(index, "deepcenter_decision")\n    if type(decision.accepted) is not bool:\n        raise _twin_record_type_error(index, "deepcenter_decision.accepted")\n    if decision.accepted is not True or decision.reason is not None:\n        raise _twin_record_value_error(index, "deepcenter_decision")\n    if type(decision.raw_score) is not float:\n        raise _twin_record_type_error(index, "deepcenter_decision.raw_score")\n    if not math.isfinite(decision.raw_score):\n        raise _twin_record_value_error(index, "deepcenter_decision.raw_score")\n    if decision.raw_score != record.raw_deepcenter_score:\n        raise _twin_record_value_error(index, "deepcenter_decision.raw_score")\n\n    removed = record.removed_edge\n    if type(removed) is not TwinEdgeRecord:\n        raise _twin_record_type_error(index, "removed_edge")\n    if type(removed.source_id) is not int or type(removed.target_id) is not int:\n        raise _twin_record_type_error(index, "removed_edge endpoints")\n    if type(removed.metadata) is not TwinFrozenMapping:\n        raise _twin_record_type_error(index, "removed_edge.metadata")\n    if (removed.source_id, removed.target_id) != (record.q, record.b):\n        raise _twin_record_value_error(index, "removed_edge endpoints")\n\n    planned = record.planned_edge\n    if type(planned) is not TwinPlannedEdge:\n        raise _twin_record_type_error(index, "planned_edge")\n    if type(planned.source_id) is not int or type(planned.target_id) is not int:\n        raise _twin_record_type_error(index, "planned_edge endpoints")\n    if type(planned.distance_um) is not float:\n        raise _twin_record_type_error(index, "planned_edge.distance_um")\n    if not math.isfinite(planned.distance_um):\n        raise _twin_record_value_error(index, "planned_edge.distance_um")\n    if planned.edge_prob is not None:\n        raise _twin_record_value_error(index, "planned_edge.edge_prob")\n    if (planned.source_id, planned.target_id, planned.distance_um) != (\n        record.p,\n        record.b,\n        record.d_pb,\n    ):\n        raise _twin_record_value_error(index, "planned_edge")\n    return record.dataset\n\n\nclass _TwinDebugCollector:\n    def __init__(self, max_records: int) -> None:\n        if type(max_records) is not int or max_records < 0:\n            raise ValueError("max_records must be a nonnegative built-in int")\n        self._max_records = max_records\n        self._lines: list[str] = []\n        self._records_written = 0\n        self._records_dropped = 0\n        self._finalized = False\n\n    @property\n    def records_written(self) -> int:\n        return self._records_written\n\n    @property\n    def records_dropped(self) -> int:\n        return self._records_dropped\n\n    def allocate(self, records: Sequence[TwinDebugRecord]) -> tuple[int, int]:\n        if self._finalized:\n            raise RuntimeError("twin debug collector is already finalized")\n        if not isinstance(records, Sequence):\n            raise TypeError("twin debug records must be a Sequence")\n        records_snapshot = tuple(records)\n        expected_dataset: str | None = None\n        for index, record in enumerate(records_snapshot):\n            expected_dataset = _validate_twin_debug_record(\n                record,\n                index,\n                expected_dataset,\n                first=index == 0,\n            )\n\n        written = min(len(records_snapshot), self._max_records - self._records_written)\n        dropped = len(records_snapshot) - written\n        encoded = [\n            json.dumps(\n                _twin_debug_record_plain(records_snapshot[index]),\n                sort_keys=True,\n                separators=(",", ":"),\n                allow_nan=False,\n            )\n            + "\\n"\n            for index in range(written)\n        ]\n        self._lines.extend(encoded)\n        self._records_written += written\n        self._records_dropped += dropped\n        return written, dropped\n\n    def finalize(self, output_path: Path) -> None:\n        if self._finalized:\n            raise RuntimeError("twin debug collector is already finalized")\n        handle = None\n        temp_path: Path | None = None\n        try:\n            output_path.parent.mkdir(parents=True, exist_ok=True)\n            handle = tempfile.NamedTemporaryFile(\n                mode="w",\n                encoding="utf-8",\n                newline="",\n                dir=output_path.parent,\n                prefix=f".{output_path.name}.",\n                suffix=".tmp",\n                delete=False,\n            )\n            temp_path = Path(handle.name)\n            handle.write("".join(self._lines))\n            handle.flush()\n            os.fsync(handle.fileno())\n            handle.close()\n            handle = None\n            os.replace(temp_path, output_path)\n            temp_path = None\n            self._finalized = True\n        except BaseException:\n            if handle is not None:\n                try:\n                    handle.close()\n                except BaseException:\n                    pass\n            if temp_path is not None:\n                try:\n                    temp_path.unlink(missing_ok=True)\n                except BaseException:\n                    pass\n            raise\n\n\ndef _validate_twin_plan_failure_envelope(plan: object) -> TwinPlan:\n    if type(plan) is not TwinPlan:\n        raise RuntimeError("invalid twin planner return type")\n    try:\n        reason = plan.validation_reason\n        snapshots = (\n            plan.nodes,\n            plan.edges,\n            plan.candidates,\n            plan.accepted_candidates,\n            plan.decisions,\n            plan.debug_records,\n        )\n        counters = plan.counters\n    except Exception as exc:\n        raise RuntimeError("invalid twin planner top-level fields") from exc\n    if any(type(value) is not tuple for value in snapshots):\n        raise RuntimeError("invalid twin planner top-level fields")\n    if type(counters) is not TwinFrozenMapping:\n        raise RuntimeError("invalid twin planner failure counters")\n    try:\n        items = counters.items_snapshot\n    except Exception as exc:\n        raise RuntimeError("invalid twin planner failure counters") from exc\n    if type(items) is not tuple or len(items) != len(_TWIN_COUNTER_KEYS):\n        raise RuntimeError("invalid twin planner failure counters")\n    values: list[int] = []\n    for index, entry in enumerate(items):\n        if (\n            type(entry) is not tuple\n            or len(entry) != 2\n            or type(entry[0]) is not str\n            or entry[0] != _TWIN_COUNTER_KEYS[index]\n            or type(entry[1]) is not int\n        ):\n            raise RuntimeError("invalid twin planner failure counters")\n        values.append(entry[1])\n    if reason is None:\n        return plan\n    if type(reason) is not str or reason not in _TWIN_VALIDATION_REASONS:\n        raise RuntimeError("invalid twin planner validation reason")\n    if any(\n        type(value) is not tuple or value\n        for value in snapshots\n    ):\n        raise RuntimeError("invalid twin planner failure snapshot")\n    expected_one = {\n        "steal_twin_validation_failed",\n        f"steal_twin_validation_{reason}",\n    }\n    if any(value != (1 if key in expected_one else 0) for key, value in zip(_TWIN_COUNTER_KEYS, values, strict=True)):\n        raise RuntimeError("invalid twin planner failure counters")\n    return plan\n\n\ndef _run_steal_twin_r1_dry_run(\n    cfg: PostprocConfig,\n    dataset: str | None,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    deepcenter_bundle: dict[str, object] | None,\n    repair_frame_cache: dict[int, np.ndarray],\n    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray],\n    twin_debug_collector: _TwinDebugCollector | None = None,\n    *,\n    defer_debug_allocation: bool = False,\n) -> TwinPlan:\n    def score_callback(node: TwinSnapshotNode) -> TwinDeepCenterDecision:\n        return score_twin_deepcenter(\n            cfg,\n            dataset,\n            node.t,\n            (node.z, node.y, node.x),\n            deepcenter_bundle,\n            repair_frame_cache,\n            deepcenter_heatmap_cache,\n        )\n\n    plan = plan_twin_only_v1(\n        cfg,\n        dataset,\n        tuple(nodes_by_id.values()),\n        tuple(edges),\n        score_callback,\n    )\n    plan = _validate_twin_plan_failure_envelope(plan)\n    try:\n        counter_keys = tuple(plan.counters)\n        counter_values = tuple(plan.counters[key] for key in counter_keys)\n    except Exception as exc:\n        raise RuntimeError("invalid twin planner counter mapping") from exc\n    if counter_keys != _TWIN_COUNTER_KEYS:\n        raise RuntimeError("invalid twin planner counter key schema")\n    if any(type(value) is not int or value < 0 for value in counter_values):\n        raise RuntimeError("invalid twin planner counter value")\n    if any(\n        plan.counters[key] != 0\n        for key in ("steal_twin_debug_records_written", "steal_twin_debug_records_dropped")\n    ):\n        raise RuntimeError("twin planner must not allocate debug counters")\n    if any(key not in stats or type(stats[key]) is not int or stats[key] != 0 for key in _TWIN_COUNTER_KEYS):\n        raise RuntimeError("twin destination counters must be preseeded exact integer zero")\n\n    for key, value in zip(counter_keys, counter_values, strict=True):\n        stats[key] = value\n    if cfg.STEAL_TWIN_DEBUG_JSONL and not defer_debug_allocation:\n        if twin_debug_collector is None:\n            raise RuntimeError("twin debug path requires a run-level collector")\n        written, dropped = twin_debug_collector.allocate(plan.debug_records)\n        stats["steal_twin_debug_records_written"] = written\n        stats["steal_twin_debug_records_dropped"] = dropped\n    return plan\n\n\ndef _twin_fork_sources(edges: list[dict[str, object]]) -> int:\n    counts: dict[int, int] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        counts[source_id] = counts.get(source_id, 0) + 1\n    return sum(value == 2 for value in counts.values())\n\n\ndef _twin_observed_removal(before: int, after: int, stage: str) -> int:\n    removed = before - after\n    if removed < 0:\n        raise RuntimeError(f"{stage} increased twin graph topology")\n    return removed\n\n\ndef _linefit_with_twin_r2_guard(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    *,\n    enabled: bool,\n) -> dict[int, dict[str, object]]:\n    if not enabled:\n        return linefit_smooth_output_graph(cfg, nodes_by_id, edges, stats)\n\n    node_mapping_id = id(nodes_by_id)\n    node_snapshot: list[\n        tuple[\n            object,\n            dict[str, object],\n            tuple[tuple[object, object], ...],\n            tuple[bytes, bytes, bytes],\n        ]\n    ] = []\n    for outer_key, row in nodes_by_id.items():\n        if type(row) is not dict:\n            raise RuntimeError("invalid twin linefit node row")\n        items = tuple(row.items())\n        coordinate_values: dict[str, float] = {}\n        for key, value in items:\n            if type(key) is str and key in ("z", "y", "x"):\n                coordinate_values[key] = value\n        if set(coordinate_values) != {"z", "y", "x"}:\n            raise RuntimeError("invalid twin linefit coordinate schema")\n        coordinates = tuple(coordinate_values[name] for name in ("z", "y", "x"))\n        if any(type(value) is not float or not math.isfinite(value) for value in coordinates):\n            raise RuntimeError("invalid twin linefit pre-coordinate")\n        node_snapshot.append(\n            (\n                outer_key,\n                row,\n                items,\n                tuple(struct.pack(">d", value) for value in coordinates),\n            )\n        )\n    edge_list_id = id(edges)\n    edge_snapshot = [\n        (row, tuple(row.items()))\n        for row in edges\n        if type(row) is dict\n    ]\n    if len(edge_snapshot) != len(edges):\n        raise RuntimeError("invalid twin linefit edge row")\n\n    returned = linefit_smooth_output_graph(cfg, nodes_by_id, edges, stats)\n    if id(returned) != node_mapping_id or id(nodes_by_id) != node_mapping_id or id(edges) != edge_list_id:\n        raise RuntimeError("twin linefit replaced a graph container")\n    if len(nodes_by_id) != len(node_snapshot) or len(edges) != len(edge_snapshot):\n        raise RuntimeError("twin linefit changed graph topology")\n\n    changed = 0\n    for (expected_outer_key, expected_row, expected_items, before), (outer_key, row) in zip(\n        node_snapshot, nodes_by_id.items(), strict=True\n    ):\n        if outer_key is not expected_outer_key or row is not expected_row:\n            raise RuntimeError("twin linefit replaced or reordered a node")\n        current_items = tuple(row.items())\n        if len(current_items) != len(expected_items) or any(\n            current_key is not expected_key\n            for (current_key, _), (expected_key, _) in zip(\n                current_items, expected_items, strict=True\n            )\n        ):\n            raise RuntimeError("twin linefit changed node keys")\n        after_coordinates: dict[str, bytes] = {}\n        for (key, value), (_, expected_value) in zip(\n            current_items, expected_items, strict=True\n        ):\n            if type(key) is str and key in ("z", "y", "x"):\n                if type(value) is not float or not math.isfinite(value):\n                    raise RuntimeError("invalid twin linefit post-coordinate")\n                after_coordinates[key] = struct.pack(">d", value)\n            elif value is not expected_value:\n                raise RuntimeError("twin linefit changed a preserved node binding")\n        if tuple(after_coordinates[name] for name in ("z", "y", "x")) != before:\n            changed += 1\n    for (expected_row, expected_items), row in zip(edge_snapshot, edges, strict=True):\n        if row is not expected_row:\n            raise RuntimeError("twin linefit replaced or reordered an edge")\n        current_items = tuple(row.items())\n        if len(current_items) != len(expected_items):\n            raise RuntimeError("twin linefit changed edge keys")\n        for (key, value), (expected_key, expected_value) in zip(\n            current_items, expected_items, strict=True\n        ):\n            if key is not expected_key:\n                raise RuntimeError("twin linefit changed edge keys")\n            if value is not expected_value:\n                raise RuntimeError("twin linefit changed an edge binding")\n    stats["steal_twin_linefit_coordinate_changed_nodes_observed"] = changed\n    return returned\n\n\ndef _twin_debug_path_identities(debug_path: Path) -> tuple[Path, Path]:\n    debug_resolved = debug_path.resolve(strict=False)\n    debug_parent_resolved = debug_path.parent.resolve(strict=False)\n    return debug_resolved, debug_parent_resolved / debug_path.name\n\n\ndef _twin_paths_samefile(first: Path, second: Path) -> bool:\n    if not first.exists() or not second.exists():\n        return False\n    try:\n        return os.path.samefile(first, second)\n    except OSError:\n        return False\n\n\ndef _raise_twin_debug_alias(debug_path: Path, artifact: Path) -> None:\n    raise ValueError(f"twin debug target {debug_path} aliases run artifact {artifact}")\n\n\ndef _twin_debug_identity_overlaps_protected_path(\n    debug_identities: Sequence[Path], protected_resolved: Path\n) -> bool:\n    return any(\n        identity == protected_resolved\n        or identity.is_relative_to(protected_resolved)\n        or protected_resolved.is_relative_to(identity)\n        for identity in debug_identities\n    )\n\n\ndef _reject_twin_debug_aliases(\n    geff_dir: Path,\n    geffs: Sequence[Path],\n    debug_path: Path,\n    ordinary_artifacts: Sequence[Path],\n) -> None:\n    debug_resolved, debug_entry_resolved = _twin_debug_path_identities(debug_path)\n    debug_identities = (debug_resolved, debug_entry_resolved)\n    bundle_resolved = geff_dir.resolve(strict=True)\n    if not bundle_resolved.is_dir():\n        raise ValueError(f"GEFF input bundle is not a directory: {geff_dir}")\n    if _twin_debug_identity_overlaps_protected_path(debug_identities, bundle_resolved):\n        _raise_twin_debug_alias(debug_path, geff_dir)\n\n    for geff_path in geffs:\n        geff_resolved = geff_path.resolve(strict=True)\n        if geff_resolved.is_dir():\n            if _twin_debug_identity_overlaps_protected_path(debug_identities, geff_resolved):\n                _raise_twin_debug_alias(debug_path, geff_path)\n        elif geff_resolved.is_file():\n            if _twin_debug_identity_overlaps_protected_path(\n                debug_identities, geff_resolved\n            ) or _twin_paths_samefile(debug_path, geff_path):\n                _raise_twin_debug_alias(debug_path, geff_path)\n        else:\n            raise ValueError(f"GEFF input is not a regular file or directory: {geff_path}")\n\n    for artifact in ordinary_artifacts:\n        artifact_resolved = artifact.resolve(strict=False)\n        if _twin_debug_identity_overlaps_protected_path(\n            debug_identities, artifact_resolved\n        ) or _twin_paths_samefile(debug_path, artifact):\n            _raise_twin_debug_alias(debug_path, artifact)\n\n\ndef _unselected_association_priors(\n    priors: dict[tuple[int, int], float] | None,\n    raw_edges: list[dict[str, object]],\n    nodes_by_id: dict[int, dict[str, object]],\n) -> dict[tuple[int, int], float]:\n    """Validate learned association priors and keep only ILP-unselected pairs.\n\n    Returns a new dict of float probabilities for pairs that are *not* present\n    in the ORIGINAL ``raw_edges`` (selected pairs keep whatever probability the\n    edge itself carries; they are never overwritten).  Pairs whose endpoints no\n    longer exist in ``nodes_by_id`` are excluded -- pre-linefit has already\n    dropped ILP-unselected nodes.  Every other pair must reference two existing\n    nodes whose builtin ``int`` times satisfy ``t == source_t + 1``; missing or\n    invalid times raise ``ValueError``, including for selected pairs.\n\n    No input is mutated.\n    """\n    if priors is None:\n        return {}\n    if type(priors) is not dict:\n        raise ValueError("association_priors must be a dict or None")\n\n    selected_pairs = {\n        (int(edge["source_id"]), int(edge["target_id"])) for edge in raw_edges\n    }\n\n    def _time(node_id: int) -> int:\n        t = nodes_by_id.get(node_id, {}).get("t")\n        if type(t) is not int:\n            raise ValueError(\n                f"association_priors endpoint {node_id!r} needs a builtin int t: {t!r}"\n            )\n        return t\n\n    result: dict[tuple[int, int], float] = {}\n    for key, value in priors.items():\n        if type(key) is not tuple or len(key) != 2:\n            raise ValueError(\n                f"association_priors keys must be 2-tuples of builtin ints: {key!r}"\n            )\n        sid, tid = key\n        if type(sid) is not int or type(tid) is not int:\n            raise ValueError(\n                f"association_priors keys must be 2-tuples of builtin ints: {key!r}"\n            )\n        if type(value) is bool or type(value) not in (int, float):\n            raise ValueError(\n                f"association_priors values must be builtin int or float: {value!r}"\n            )\n        if value < 0 or value > 1:\n            raise ValueError(\n                f"association_priors values must be within [0, 1]: {value!r}"\n            )\n        prob = float(value)\n        if not math.isfinite(prob):\n            raise ValueError(\n                f"association_priors values must be finite: {value!r}"\n            )\n        if sid not in nodes_by_id or tid not in nodes_by_id:\n            continue\n        if _time(tid) != _time(sid) + 1:\n            raise ValueError(\n                "association_priors endpoints must be adjacent frames: "\n                f"({sid}, {tid}) -> t={_time(sid)}, t={_time(tid)}"\n            )\n        if (sid, tid) in selected_pairs:\n            continue\n        result[(sid, tid)] = prob\n    return result\n\n\ndef filter_output_graph_pre_linefit(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    raw_edges: list[dict[str, object]],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    *,\n    twin_debug_collector: _TwinDebugCollector | None = None,\n    twin_plan_hook: TwinPlanHook | None = None,\n    association_priors: dict[tuple[int, int], float] | None = None,\n    appearance_frames: dict[int, dict] | None = None,\n    consensus_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    consensus_soft_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    short_track_consensus_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    bidirectional_motion_consistency: bool = False,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\n    """Everything ``filter_output_graph`` does *except* the final linefit-smoothing call.\n\n    This is the checkpoint boundary: final node/edge topology is fixed here\n    (``filter_short_track_components`` already ran); only coordinates can\n    still move.\n    """\n    if appearance_frames is not None:\n        # Opt-in appearance features: exact non-empty frame map, motion relink\n        # enabled, and never combined with E27 association priors.\n        if type(appearance_frames) is not dict or not appearance_frames:\n            raise ValueError("appearance_frames must be a non-empty exact dict")\n        if not cfg.OUTPUT_MOTION_RELINK:\n            raise ValueError("appearance_frames requires OUTPUT_MOTION_RELINK")\n        if association_priors is not None:\n            raise ValueError("appearance_frames cannot be combined with association_priors")\n    if appearance_frames is not None and (\n        consensus_pairs_by_t is not None or consensus_soft_pairs_by_t is not None\n    ):\n        raise ValueError("consensus pairs cannot be combined with appearance_frames")\n    if consensus_pairs_by_t is not None and consensus_soft_pairs_by_t is not None:\n        raise ValueError("hard and soft consensus pairs cannot be combined")\n    unselected_priors = _unselected_association_priors(\n        association_priors, raw_edges, nodes_by_id\n    )\n    _require_steal_twin_r1_dry_run(cfg)\n    if cfg.OUTPUT_STEAL_TWIN_REWIRE and cfg.STEAL_TWIN_DEBUG_JSONL and twin_debug_collector is None:\n        raise RuntimeError("twin debug path requires a run-level collector")\n\n    stats = new_stats()\n    stats["raw_edges"] = len(raw_edges)\n    raw_edges_for_consensus = [dict(edge) for edge in raw_edges]\n\n    # One shared frame cache for the whole pre-linefit stack: E23 all-node\n    # centroid refinement reads each timepoint once, and gap-close /\n    # safe-division / DeepCenter reuse those frames below.\n    repair_frame_cache: dict[int, np.ndarray] = {}\n    if cfg.REFINE_ALL_CENTROIDS:\n        # Runs before any edge distance is computed, matching the notebook:\n        # refined coordinates feed the edge-length filter, motion relink,\n        # gap repair, safe divisions and DeepCenter queries.\n        refine_all_centroids(cfg, nodes_by_id, dataset, repair_frame_cache, stats)\n\n    edges: list[dict[str, object]] = []\n    for edge in raw_edges:\n        source = nodes_by_id.get(int(edge["source_id"]))\n        target = nodes_by_id.get(int(edge["target_id"]))\n        if source is None or target is None:\n            continue\n        if cfg.OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:\n            stats["dropped_nonconsecutive_edges"] += 1\n            continue\n        distance_um = edge_distance_um(source, target)\n        edge["distance_um"] = distance_um\n        if cfg.OUTPUT_EDGE_MAX_UM > 0 and distance_um > cfg.OUTPUT_EDGE_MAX_UM:\n            stats["dropped_long_edges"] += 1\n            continue\n        edges.append(edge)\n\n    if cfg.OUTPUT_MOTION_RELINK:\n        learned_edge_probs: dict[tuple[int, int], float] = {}\n        for edge in edges:\n            prob = edge.get("edge_prob")\n            if prob is None:\n                continue\n            try:\n                prob = float(prob)\n            except (TypeError, ValueError):\n                continue\n            if np.isfinite(prob):\n                key = (int(edge["source_id"]), int(edge["target_id"]))\n                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)\n        learned_edge_probs.update(unselected_priors)\n        hard_consensus = (\n            None\n            if consensus_pairs_by_t is None\n            else prepare_consensus_reservations(\n                consensus_pairs_by_t,\n                nodes_by_id,\n                raw_edges_for_consensus,\n                edges,\n                cfg.MOTION_RELINK_TIGHT_UM,\n            )\n        )\n        soft_consensus = (\n            None\n            if consensus_soft_pairs_by_t is None\n            else prepare_consensus_reservations(\n                consensus_soft_pairs_by_t,\n                nodes_by_id,\n                raw_edges_for_consensus,\n                edges,\n                cfg.MOTION_RELINK_TIGHT_UM,\n            )\n        )\n        motion_edges = motion_relink_edges(\n            cfg,\n            nodes_by_id,\n            stats,\n            learned_edge_probs,\n            appearance_frames=appearance_frames,\n            consensus_pairs_by_t=hard_consensus,\n            consensus_soft_pairs_by_t=soft_consensus,\n            bidirectional_motion_consistency=bidirectional_motion_consistency,\n        )\n        if motion_edges:\n            stats["motion_relink_replaced_raw_edges"] = len(edges)\n            edges = motion_edges\n        else:\n            stats["motion_relink_fallback_raw"] = 1\n\n    if cfg.OUTPUT_SINGLE_PARENT_REPAIR and edges:\n        best_by_target: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            target_id = int(edge["target_id"])\n            prev = best_by_target.get(target_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_target[target_id] = edge\n        kept_ids = {id(edge) for edge in best_by_target.values()}\n        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    if cfg.OUTPUT_SINGLE_CHILD_REPAIR and edges:\n        best_by_source: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            prev = best_by_source.get(source_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_source[source_id] = edge\n        kept_ids = {id(edge) for edge in best_by_source.values()}\n        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray] = {}\n    nodes_by_id, edges = close_single_frame_gaps(\n        cfg,\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n    nodes_by_id, edges = recover_strict_gap2(cfg, nodes_by_id, edges, stats, dataset=dataset)\n    edges = add_safe_divisions_postlink(\n        cfg,\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n\n    twin_candidate_active = False\n    if cfg.OUTPUT_STEAL_TWIN_REWIRE:\n        if twin_plan_hook is None:\n            plan = _run_steal_twin_r1_dry_run(\n                cfg,\n                dataset,\n                nodes_by_id,\n                edges,\n                stats,\n                deepcenter_bundle,\n                repair_frame_cache,\n                deepcenter_heatmap_cache,\n                twin_debug_collector,\n            )\n        else:\n            if dataset is None:\n                raise RuntimeError("twin plan hook requires a dataset")\n            plan = _run_steal_twin_r1_dry_run(\n                cfg,\n                dataset,\n                nodes_by_id,\n                edges,\n                stats,\n                deepcenter_bundle,\n                repair_frame_cache,\n                deepcenter_heatmap_cache,\n                twin_debug_collector,\n                defer_debug_allocation=True,\n            )\n            twin_plan_hook(dataset, plan)\n            if cfg.STEAL_TWIN_DEBUG_JSONL:\n                if twin_debug_collector is None:\n                    raise RuntimeError("twin debug path requires a run-level collector")\n                written, dropped = twin_debug_collector.allocate(plan.debug_records)\n                stats["steal_twin_debug_records_written"] = written\n                stats["steal_twin_debug_records_dropped"] = dropped\n        if plan.validation_reason is None and not cfg.STEAL_TWIN_DRY_RUN:\n            edges, mutation = apply_twin_only_v1_plan(nodes_by_id, edges, plan)\n            if mutation.status == "already_applied":\n                raise RuntimeError("pipeline received an already-applied twin plan")\n            accepted = len(plan.accepted_candidates)\n            if mutation.status not in ("applied", "no_changes"):\n                raise RuntimeError("invalid twin mutation status")\n            stats["steal_twin_edges_removed"] = mutation.edges_removed\n            stats["steal_twin_edges_added"] = mutation.edges_added\n            stats["steal_twin_mutations_applied"] = mutation.edges_removed\n            stats["steal_twin_pure_nodes"] = len(nodes_by_id)\n            stats["steal_twin_pure_edges"] = len(edges)\n            stats["steal_twin_pure_fork_sources"] = _twin_fork_sources(edges)\n            stats["steal_twin_pure_edge_symmetric_difference"] = mutation.edge_symmetric_difference\n            if not (\n                stats["steal_twin_planned_edges_removed"]\n                == stats["steal_twin_planned_edges_added"]\n                == stats["steal_twin_accepted"]\n                == accepted\n                == stats["steal_twin_edges_removed"]\n                == stats["steal_twin_edges_added"]\n                == stats["steal_twin_mutations_applied"]\n                and stats["steal_twin_pure_edge_symmetric_difference"] == 2 * accepted\n            ):\n                raise RuntimeError("twin mutation counter conservation failed")\n            twin_candidate_active = True\n    elif twin_plan_hook is not None:\n        if dataset is None:\n            raise RuntimeError("twin plan hook requires a dataset")\n        twin_plan_hook(dataset, None)\n\n    geometry_edges_before = len(edges)\n    if cfg.OUTPUT_DIVISION_GEOMETRY_FILTER and edges:\n        by_source: dict[int, list[dict[str, object]]] = {}\n        for edge in edges:\n            by_source.setdefault(int(edge["source_id"]), []).append(edge)\n\n        filtered: list[dict[str, object]] = []\n        for source_id, source_edges in by_source.items():\n            if len(source_edges) <= 1:\n                filtered.extend(source_edges)\n                continue\n\n            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)\n            source = nodes_by_id[source_id]\n            top1 = ranked[0]\n            top2 = ranked[1]\n            d1 = float(top1["distance_um"])\n            d2 = float(top2["distance_um"])\n            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])\n            valid_division = (\n                max(d1, d2) <= cfg.DIV_PARENT_MAX_UM\n                and sister <= cfg.DIV_SISTER_MAX_UM\n                and int(nodes_by_id[int(top1["target_id"])]["t"]) == int(source["t"]) + 1\n                and int(nodes_by_id[int(top2["target_id"])]["t"]) == int(source["t"]) + 1\n            )\n            if valid_division:\n                filtered.extend([top1, top2])\n                stats["dropped_division_edges"] += max(0, len(ranked) - 2)\n            elif cfg.DIV_DROP_TO_SINGLE_IF_BAD:\n                filtered.append(top1)\n                stats["dropped_division_edges"] += len(ranked) - 1\n            else:\n                filtered.extend(ranked)\n        edges = filtered\n    if twin_candidate_active:\n        stats["steal_twin_geometry_edges_removed_observed"] = _twin_observed_removal(\n            geometry_edges_before, len(edges), "division geometry"\n        )\n\n    prune_nodes_before = len(nodes_by_id)\n    prune_edges_before = len(edges)\n    if cfg.OUTPUT_PRUNE_ISOLATED:\n        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}\n        if incident:\n            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}\n            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)\n            nodes_by_id = kept_nodes\n            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]\n    if twin_candidate_active:\n        stats["steal_twin_prune_nodes_removed_observed"] = _twin_observed_removal(\n            prune_nodes_before, len(nodes_by_id), "isolated prune"\n        )\n        stats["steal_twin_prune_edges_removed_observed"] = _twin_observed_removal(\n            prune_edges_before, len(edges), "isolated prune"\n        )\n\n    short_nodes_before = len(nodes_by_id)\n    short_edges_before = len(edges)\n    if short_track_consensus_pairs_by_t is None:\n        nodes_by_id, edges = filter_short_track_components(cfg, nodes_by_id, edges, stats)\n    else:\n        nodes_by_id, edges = filter_short_track_components(\n            cfg, nodes_by_id, edges, stats,\n            consensus_proof_pairs_by_t=short_track_consensus_pairs_by_t,\n        )\n    if twin_candidate_active:\n        stats["steal_twin_short_nodes_removed_observed"] = _twin_observed_removal(\n            short_nodes_before, len(nodes_by_id), "short-track filter"\n        )\n        stats["steal_twin_short_edges_removed_observed"] = _twin_observed_removal(\n            short_edges_before, len(edges), "short-track filter"\n        )\n        stats["steal_twin_final_nodes"] = len(nodes_by_id)\n        stats["steal_twin_final_edges"] = len(edges)\n        stats["steal_twin_final_fork_sources"] = _twin_fork_sources(edges)\n\n    if bidirectional_motion_consistency and edges:\n        # E34 safety gate: a malformed downstream merge must never publish an\n        # invalid graph. Keep the strongest edge per target and per source.\n        best_target: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            tid = int(edge["target_id"])\n            if tid not in best_target or edge_sort_key(edge) > edge_sort_key(best_target[tid]):\n                best_target[tid] = edge\n        best_source: dict[int, dict[str, object]] = {}\n        for edge in best_target.values():\n            sid = int(edge["source_id"])\n            if sid not in best_source or edge_sort_key(edge) > edge_sort_key(best_source[sid]):\n                best_source[sid] = edge\n        edges = list(best_source.values())\n\n    return nodes_by_id, edges, stats\n\n\ndef filter_output_graph(\n    cfg: PostprocConfig,\n    nodes_by_id: dict[int, dict[str, object]],\n    raw_edges: list[dict[str, object]],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    *,\n    twin_debug_collector: _TwinDebugCollector | None = None,\n    twin_plan_hook: TwinPlanHook | None = None,\n    association_priors: dict[tuple[int, int], float] | None = None,\n    appearance_frames: dict[int, dict] | None = None,\n    consensus_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    consensus_soft_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    short_track_consensus_pairs_by_t: dict[int, list[tuple[int, int]]] | None = None,\n    bidirectional_motion_consistency: bool = False,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\n    nodes_by_id, edges, stats = filter_output_graph_pre_linefit(\n        cfg,\n        nodes_by_id,\n        raw_edges,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        twin_debug_collector=twin_debug_collector,\n        twin_plan_hook=twin_plan_hook,\n        association_priors=association_priors,\n        consensus_pairs_by_t=consensus_pairs_by_t,\n        consensus_soft_pairs_by_t=consensus_soft_pairs_by_t,\n        short_track_consensus_pairs_by_t=short_track_consensus_pairs_by_t,\n        bidirectional_motion_consistency=bidirectional_motion_consistency,\n        **({} if appearance_frames is None else {"appearance_frames": appearance_frames}),\n    )\n    twin_candidate_active = (\n        stats["steal_twin_pure_nodes"] > 0 or stats["steal_twin_pure_edges"] > 0\n    )\n    nodes_by_id = _linefit_with_twin_r2_guard(\n        cfg,\n        nodes_by_id,\n        edges,\n        stats,\n        enabled=twin_candidate_active,\n    )\n    return nodes_by_id, edges, stats\n\n\ndef _load_geff_as_dicts(geff_path: Path) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    graph = load_geff_graph(geff_path)\n\n    nodes_by_id: dict[int, dict[str, object]] = {}\n    for row in graph.node_attrs().iter_rows(named=True):\n        node_id = int(row["node_id"])\n        if node_id in nodes_by_id:\n            raise ValueError(f"{geff_path}: duplicate node_id {node_id}")\n        nodes_by_id[node_id] = {\n            "node_id": node_id,\n            "t": int(row["t"]),\n            "z": float(row["z"]),\n            "y": float(row["y"]),\n            "x": float(row["x"]),\n        }\n\n    raw_edges: list[dict[str, object]] = []\n    for row in graph.edge_attrs().iter_rows(named=True):\n        edge_prob = row.get("edge_prob") if hasattr(row, "get") else None\n        raw_edges.append({\n            "source_id": int(row["source_id"]),\n            "target_id": int(row["target_id"]),\n            "edge_prob": None if edge_prob is None else float(edge_prob),\n        })\n    return nodes_by_id, raw_edges\n\n\ndef _dataset_stats_row(\n    dataset: str,\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    raw_node_count: int,\n    division_sources: dict[int, int],\n) -> dict[str, object]:\n    node_count = len(nodes_by_id)\n    edge_count = len(edges)\n    return {\n        "dataset": dataset,\n        "raw_nodes": raw_node_count,\n        "nodes": node_count,\n        "raw_edges": stats["raw_edges"],\n        "edges": edge_count,\n        "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),\n        "edge_to_node_ratio": edge_count / max(node_count, 1),\n        "gap_added_nodes_frac": stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),\n        **stats,\n    }\n\n\ndef _finish_run(\n    writer: SubmissionCsvWriter,\n    stats_rows: list[dict[str, object]],\n    total_nodes: int,\n    total_edges: int,\n    cfg: PostprocConfig,\n    run_stats_path: Path,\n    predict_seconds: float,\n) -> pd.DataFrame:\n    assert writer.row_id == total_nodes + total_edges, "Internal row counter mismatch"\n    assert total_nodes > 0, "No node rows produced"\n\n    stats_rows_with_meta = [\n        {**row, "predict_minutes_total": predict_seconds / 60.0, "experiment_tag": cfg.EXPERIMENT_TAG}\n        for row in stats_rows\n    ]\n    return write_run_stats(stats_rows_with_meta, run_stats_path)\n\n\ndef run_postproc_core(\n    geff_paths: Sequence[Path],\n    out_csv: Path,\n    cfg: PostprocConfig,\n    run_stats_path: Path | None = None,\n    predict_seconds: float = 0.0,\n    *,\n    deepcenter_loader: DeepCenterLoader | None = None,\n    dataset_start_hook: DatasetHook | None = None,\n    dataset_finish_hook: DatasetHook | None = None,\n    twin_plan_hook: TwinPlanHook | None = None,\n    raw_stats_hook: RawStatsHook | None = None,\n    write_run_stats_output: bool = True,\n    exclusive_output: bool = False,\n    node_serializer: NodeSerializer | None = None,\n    association_priors_by_dataset: dict[str, dict[tuple[int, int], float]] | None = None,\n    appearance_loader: Callable[[str, dict[int, dict[str, object]]], dict[int, dict]]\n    | None = None,\n    consensus_loader: Callable[[str, dict[int, dict[str, object]]], dict[int, list[tuple[int, int]]]]\n    | None = None,\n    consensus_soft_loader: Callable[[str, dict[int, dict[str, object]]], dict[int, list[tuple[int, int]]]]\n    | None = None,\n    short_track_consensus_loader: Callable[[str, dict[int, dict[str, object]]], dict[int, list[tuple[int, int]]]]\n    | None = None,\n    bidirectional_motion_consistency: bool = False,\n) -> dict[str, object]:\n    """Run the full stack over an explicit GEFF sequence without reordering it.\n\n    This is the sole full-run dataset loop.  The legacy entry point supplies a\n    sorted discovery result; the ST-R3 adapter supplies its frozen literal\n    order and production hooks. An optional serializer changes only the node\n    CSV boundary; the default retains the legacy byte representation.\n\n    ``association_priors_by_dataset`` is opt-in.  ``None`` (default) keeps the\n    legacy behaviour exactly.  When supplied it must be an outer ``dict`` whose\n    keys are exactly the unique GEFF stems; every inner mapping is validated up\n    front with ``_unselected_association_priors`` and snapshotted so that later\n    hook activity cannot alter what a downstream dataset consumes.\n    """\n    geffs = tuple(geff_paths)\n    if not geffs:\n        raise RuntimeError("no GEFF paths supplied")\n    if any(type(path) is not type(Path()) for path in geffs):\n        raise TypeError("GEFF paths must be exact pathlib.Path instances")\n\n    if appearance_loader is not None:\n        # Opt-in per-dataset appearance features; validated up front so no\n        # video is ever half-featured.  No global default config is implied.\n        if not callable(appearance_loader):\n            raise ValueError("appearance_loader must be callable")\n        if not cfg.OUTPUT_MOTION_RELINK:\n            raise ValueError("appearance_loader requires OUTPUT_MOTION_RELINK")\n        if association_priors_by_dataset is not None:\n            raise ValueError("appearance_loader cannot be combined with association_priors_by_dataset")\n        stems = tuple(path.stem for path in geffs)\n        if len(set(stems)) != len(stems):\n            raise ValueError("GEFF dataset stems must be unique when an appearance loader is supplied")\n    for loader_name, consensus_callback in (\n        ("consensus_loader", consensus_loader),\n        ("consensus_soft_loader", consensus_soft_loader),\n    ):\n        if consensus_callback is None:\n            continue\n        if not callable(consensus_callback):\n            raise ValueError(f"{loader_name} must be callable")\n        if appearance_loader is not None or association_priors_by_dataset is not None:\n            raise ValueError(f"{loader_name} cannot be combined with appearance/prior loaders")\n        stems = tuple(path.stem for path in geffs)\n        if len(set(stems)) != len(stems):\n            raise ValueError("GEFF dataset stems must be unique when a consensus loader is supplied")\n    if consensus_loader is not None and consensus_soft_loader is not None:\n        raise ValueError("hard and soft consensus loaders cannot be combined")\n    if short_track_consensus_loader is not None:\n        if not callable(short_track_consensus_loader):\n            raise ValueError("short_track_consensus_loader must be callable")\n        if appearance_loader is not None or consensus_loader is not None or consensus_soft_loader is not None:\n            raise ValueError("short-track consensus loader cannot be combined with other feature loaders")\n\n    prior_map: dict[str, dict[tuple[int, int], float]] | None = None\n    if association_priors_by_dataset is not None:\n        if type(association_priors_by_dataset) is not dict:\n            raise ValueError("association_priors_by_dataset must be an exact dict")\n        for key in association_priors_by_dataset:\n            if type(key) is not str:\n                raise ValueError("association_priors_by_dataset keys must be exact str")\n        stems = tuple(path.stem for path in geffs)\n        if len(set(stems)) != len(stems):\n            raise ValueError("GEFF dataset stems must be unique when priors are supplied")\n        if set(association_priors_by_dataset) != set(stems):\n            raise ValueError(\n                "association_priors_by_dataset keys must exactly match the GEFF dataset stems"\n            )\n        prior_map = {}\n        for stem in stems:\n            inner = association_priors_by_dataset[stem]\n            if inner is None or type(inner) is not dict:\n                raise ValueError(\n                    f"association priors for {stem!r} must be an exact dict"\n                )\n            _unselected_association_priors(inner, [], {})\n            prior_map[stem] = inner.copy()\n\n    _require_steal_twin_r1_dry_run(cfg)\n    effective_run_stats_path = run_stats_path or out_csv.parent / "run_stats.csv"\n    twin_debug_collector: _TwinDebugCollector | None = None\n    twin_debug_path: Path | None = None\n    if cfg.OUTPUT_STEAL_TWIN_REWIRE and cfg.STEAL_TWIN_DEBUG_JSONL:\n        twin_debug_path = Path(cfg.STEAL_TWIN_DEBUG_JSONL)\n        _reject_twin_debug_aliases(\n            geffs[0].parent,\n            geffs,\n            twin_debug_path,\n            (out_csv, effective_run_stats_path),\n        )\n        twin_debug_collector = _TwinDebugCollector(cfg.STEAL_TWIN_DEBUG_MAX_RECORDS)\n\n    effective_deepcenter_loader = deepcenter_loader or load_deepcenter_veto_detector\n    deepcenter_detector = effective_deepcenter_loader(cfg)\n\n    out_csv.parent.mkdir(parents=True, exist_ok=True)\n    stats_rows: list[dict[str, object]] = []\n    total_nodes = 0\n    total_edges = 0\n\n    with out_csv.open("x" if exclusive_output else "w", newline="") as handle:\n        writer = SubmissionCsvWriter(handle, node_serializer=node_serializer)\n\n        for sequence, geff_path in enumerate(geffs):\n            dataset = geff_path.stem\n            if dataset_start_hook is not None:\n                dataset_start_hook(sequence, dataset)\n            nodes_by_id, raw_edges = _load_geff_as_dicts(geff_path)\n\n            raw_node_count = len(nodes_by_id)\n            appearance_kwargs: dict[str, object] = {}\n            if appearance_loader is not None:\n                frames = appearance_loader(\n                    dataset, {node_id: dict(node) for node_id, node in nodes_by_id.items()}\n                )\n                if type(frames) is not dict or not frames:\n                    raise ValueError(f"appearance_loader returned no features for {dataset!r}")\n                appearance_kwargs["appearance_frames"] = frames\n                del frames\n            if consensus_loader is not None:\n                consensus = consensus_loader(\n                    dataset, {node_id: dict(node) for node_id, node in nodes_by_id.items()}\n                )\n                if type(consensus) is not dict:\n                    raise ValueError(f"consensus_loader returned non-dict for {dataset!r}")\n                appearance_kwargs["consensus_pairs_by_t"] = consensus\n                del consensus\n            if consensus_soft_loader is not None:\n                consensus = consensus_soft_loader(\n                    dataset, {node_id: dict(node) for node_id, node in nodes_by_id.items()}\n                )\n                if type(consensus) is not dict:\n                    raise ValueError(\n                        f"consensus_soft_loader returned non-dict for {dataset!r}"\n                    )\n                appearance_kwargs["consensus_soft_pairs_by_t"] = consensus\n                del consensus\n            if short_track_consensus_loader is not None:\n                consensus = short_track_consensus_loader(\n                    dataset, {node_id: dict(node) for node_id, node in nodes_by_id.items()}\n                )\n                if type(consensus) is not dict:\n                    raise ValueError(\n                        f"short_track_consensus_loader returned non-dict for {dataset!r}"\n                    )\n                appearance_kwargs["short_track_consensus_pairs_by_t"] = consensus\n                del consensus\n            nodes_by_id, edges, filter_stats = filter_output_graph(\n                cfg,\n                nodes_by_id,\n                raw_edges,\n                dataset=dataset,\n                deepcenter_bundle=deepcenter_detector,\n                twin_debug_collector=twin_debug_collector,\n                twin_plan_hook=twin_plan_hook,\n                association_priors=None if prior_map is None else prior_map[dataset],\n                **appearance_kwargs,\n                bidirectional_motion_consistency=bidirectional_motion_consistency,\n            )\n            del appearance_kwargs\n            if not nodes_by_id:\n                raise AssertionError(f"{dataset}: post-processing removed every node")\n\n            writer.write_nodes(dataset, nodes_by_id)\n            division_sources = writer.write_edges(dataset, nodes_by_id, edges)\n\n            total_nodes += len(nodes_by_id)\n            total_edges += len(edges)\n            stats_row = _dataset_stats_row(\n                dataset,\n                nodes_by_id,\n                edges,\n                filter_stats,\n                raw_node_count,\n                division_sources,\n            )\n            stats_rows.append(stats_row)\n            if raw_stats_hook is not None:\n                raw_stats_hook(MappingProxyType(stats_row.copy()))\n            handle.flush()\n            if dataset_finish_hook is not None:\n                os.fsync(handle.fileno())\n                dataset_finish_hook(sequence, dataset)\n\n        handle.flush()\n        os.fsync(handle.fileno())\n\n    if write_run_stats_output:\n        stats_frame: pd.DataFrame | None = _finish_run(\n            writer,\n            stats_rows,\n            total_nodes,\n            total_edges,\n            cfg,\n            effective_run_stats_path,\n            predict_seconds,\n        )\n    else:\n        assert writer.row_id == total_nodes + total_edges, "Internal row counter mismatch"\n        assert total_nodes > 0, "No node rows produced"\n        stats_frame = None\n    if twin_debug_collector is not None and twin_debug_path is not None:\n        twin_debug_collector.finalize(twin_debug_path)\n\n    return {\n        "datasets": [p.stem for p in geffs],\n        "total_nodes": total_nodes,\n        "total_edges": total_edges,\n        "total_rows": writer.row_id,\n        "run_stats": stats_frame,\n    }\n\n\ndef run_postproc(\n    geff_dir: Path,\n    out_csv: Path,\n    cfg: PostprocConfig,\n    run_stats_path: Path | None = None,\n    predict_seconds: float = 0.0,\n) -> dict[str, object]:\n    """Legacy sorted-discovery wrapper over :func:`run_postproc_core`."""\n    geffs = sorted(geff_dir.glob("*.geff"))\n    if not geffs:\n        raise RuntimeError(f"no *.geff files found in {geff_dir}")\n    return run_postproc_core(\n        geffs,\n        out_csv,\n        cfg,\n        run_stats_path=run_stats_path,\n        predict_seconds=predict_seconds,\n    )\n\n\n# ---------------------------------------------------------------------------\n# Pre-linefit checkpoint: run the expensive part once, then iterate on\n# BIOHUB_OUTPUT_LINEFIT_* (or anything else that only needs final topology +\n# coordinates) without redoing motion-relink / gap-close / safe-divisions.\n# ---------------------------------------------------------------------------\ndef save_prelinefit_checkpoint(geff_dir: Path, checkpoint_dir: Path, cfg: PostprocConfig) -> dict[str, object]:\n    """Run the stack up to (excluding) linefit smoothing; pickle one file per dataset.\n\n    Each ``<dataset>.pkl`` holds ``{"dataset", "raw_node_count", "nodes_by_id",\n    "edges", "stats"}`` -- the exact ``nodes_by_id``/``edges``/``stats`` that\n    :func:`filter_output_graph_pre_linefit` returned, coordinates as the\n    float64 Python floats they already are (pickle round-trips them exactly).\n    A ``manifest.json`` records the dataset order (matches ``sorted(*.geff)``).\n    """\n    geffs = sorted(geff_dir.glob("*.geff"))\n    if not geffs:\n        raise RuntimeError(f"no *.geff files found in {geff_dir}")\n\n    _require_steal_twin_r1_dry_run(cfg)\n    twin_debug_collector: _TwinDebugCollector | None = None\n    twin_debug_path: Path | None = None\n    if cfg.OUTPUT_STEAL_TWIN_REWIRE and cfg.STEAL_TWIN_DEBUG_JSONL:\n        twin_debug_path = Path(cfg.STEAL_TWIN_DEBUG_JSONL)\n        _reject_twin_debug_aliases(\n            geff_dir,\n            geffs,\n            twin_debug_path,\n            (\n                checkpoint_dir,\n                checkpoint_dir / CHECKPOINT_MANIFEST_NAME,\n                *(checkpoint_dir / f"{geff_path.stem}.pkl" for geff_path in geffs),\n            ),\n        )\n        twin_debug_collector = _TwinDebugCollector(cfg.STEAL_TWIN_DEBUG_MAX_RECORDS)\n\n    deepcenter_detector = load_deepcenter_veto_detector(cfg)\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n\n    datasets: list[str] = []\n    for geff_path in geffs:\n        dataset = geff_path.stem\n        nodes_by_id, raw_edges = _load_geff_as_dicts(geff_path)\n        raw_node_count = len(nodes_by_id)\n        nodes_by_id, edges, stats = filter_output_graph_pre_linefit(\n            cfg,\n            nodes_by_id,\n            raw_edges,\n            dataset=dataset,\n            deepcenter_bundle=deepcenter_detector,\n            twin_debug_collector=twin_debug_collector,\n        )\n        if not nodes_by_id:\n            raise AssertionError(f"{dataset}: post-processing removed every node")\n        payload = {\n            "dataset": dataset,\n            "raw_node_count": raw_node_count,\n            "nodes_by_id": nodes_by_id,\n            "edges": edges,\n            "stats": stats,\n        }\n        with (checkpoint_dir / f"{dataset}.pkl").open("wb") as handle:\n            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)\n        datasets.append(dataset)\n\n    manifest = {"geff_dir": str(geff_dir), "datasets": datasets}\n    (checkpoint_dir / CHECKPOINT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2))\n    if twin_debug_collector is not None and twin_debug_path is not None:\n        twin_debug_collector.finalize(twin_debug_path)\n    return manifest\n\n\ndef run_relinefit(\n    checkpoint_dir: Path,\n    out_csv: Path,\n    cfg: PostprocConfig,\n    run_stats_path: Path | None = None,\n    predict_seconds: float = 0.0,\n) -> dict[str, object]:\n    """Load a :func:`save_prelinefit_checkpoint` checkpoint, apply linefit smoothing, write the CSV.\n\n    ``cfg`` only needs to change ``BIOHUB_OUTPUT_LINEFIT_*`` between calls;\n    every other post-processing pass has already been baked into the\n    checkpoint and is not re-run.\n    """\n    manifest_path = checkpoint_dir / CHECKPOINT_MANIFEST_NAME\n    if not manifest_path.exists():\n        raise FileNotFoundError(f"no {CHECKPOINT_MANIFEST_NAME} in {checkpoint_dir} (run --save-prelinefit first)")\n    manifest = json.loads(manifest_path.read_text())\n    datasets: list[str] = manifest["datasets"]\n    if not datasets:\n        raise RuntimeError(f"{manifest_path}: empty dataset list")\n\n    out_csv.parent.mkdir(parents=True, exist_ok=True)\n    stats_rows: list[dict[str, object]] = []\n    total_nodes = 0\n    total_edges = 0\n\n    with out_csv.open("w", newline="") as handle:\n        writer = SubmissionCsvWriter(handle)\n\n        for dataset in datasets:\n            checkpoint_path = checkpoint_dir / f"{dataset}.pkl"\n            with checkpoint_path.open("rb") as f:\n                payload = pickle.load(f)  # noqa: S301 - our own checkpoint, not untrusted input\n\n            nodes_by_id = payload["nodes_by_id"]\n            edges = payload["edges"]\n            stats = dict(payload["stats"])  # copy: don\'t mutate the on-disk checkpoint\'s stats in memory\n\n            twin_candidate_active = (\n                stats.get("steal_twin_pure_nodes", 0) > 0\n                or stats.get("steal_twin_pure_edges", 0) > 0\n            )\n            nodes_by_id = _linefit_with_twin_r2_guard(\n                cfg,\n                nodes_by_id,\n                edges,\n                stats,\n                enabled=twin_candidate_active,\n            )\n\n            writer.write_nodes(dataset, nodes_by_id)\n            division_sources = writer.write_edges(dataset, nodes_by_id, edges)\n\n            total_nodes += len(nodes_by_id)\n            total_edges += len(edges)\n            stats_rows.append(\n                _dataset_stats_row(dataset, nodes_by_id, edges, stats, payload["raw_node_count"], division_sources)\n            )\n\n    stats_frame = _finish_run(\n        writer, stats_rows, total_nodes, total_edges, cfg, run_stats_path or out_csv.parent / "run_stats.csv", predict_seconds\n    )\n\n    return {\n        "datasets": datasets,\n        "total_nodes": total_nodes,\n        "total_edges": total_edges,\n        "total_rows": writer.row_id,\n        "run_stats": stats_frame,\n    }\n'}
E31_HASHES = {'src/biohub/__init__.py': 'da0ec650db1c30715c64601642ce399457c799a231928a60e93b9b0ee2df5ad5', 'src/biohub/io.py': '9252c12a75f01d1c4254b3180d53a7299f6cf49d366ccf4191aca7a16cf54848', 'src/biohub/appearance_cost.py': '3440f73429bd45520897ed1bfb09248e297d662720f9554a5649ecc1c9ab2946', 'src/biohub/consensus_edges.py': '2f4445794e1f131fb23bcee0fbc438c20609aeb106c8a12803e372e7545274b2', 'src/biohub/primary_consensus_observer.py': '72b25eefc81a08709903e3156a128ad0b2f5565a30fa1a4d8935fab95720227c', 'src/biohub/association_instrumentation.py': '9fa79df7176d1f09ba840247dca8a71da204f62a2dbf877a2b8aa5617333b795', 'src/biohub/output_bounds.py': 'd39549fbaeefc9edbd6273e10c3cf96377b33b1ea6d080befec6bd12071e5df5', 'src/biohub/screen_output_bounds.py': 'b97f31d54bd86452862f85b00fea4f698ddb8b352ad02cbeeb21c8d2aba1d11c', 'src/biohub/e31_shards.py': '329d2ae8925cf2d9af06b1e8b07c052dd7636c4c55e7c959446818d5c2feee8b', 'scripts/e31_submission_runtime.py': '5316ceb7ffa577bf69ddb7f85a223a1924a13362685b6cafdeda9bc4db8354c7', 'src/biohub/public_postproc/__init__.py': '489e2a17e97bd3cb710a919294cd9f81a8f539e4160df087a4e5ac9c6861adc4', 'src/biohub/public_postproc/config.py': '415a586ad4d8a81b96314025fa0cf05ce6c797df7ef8eb75e53f48efe192d882', 'src/biohub/public_postproc/csv_out.py': '3e7342a1a4d5e6eff0338c79ed196e5139fc8a4bcb60bc4d9de11fad6158e264', 'src/biohub/public_postproc/deepcenter.py': '7bb6b43b255220be40dba17aa728391f8053e8d41da5664eb5abb0b374161ed4', 'src/biohub/public_postproc/divisions.py': '4d9b16d1bceb96d457b3b957d2d7853313bb7e592895dc3d7e3ce6c232d2081e', 'src/biohub/public_postproc/frames.py': 'f7b1d239ecbb7afc1f182443b22f49d7364f4db9c160006785d83ab85cee3e67', 'src/biohub/public_postproc/geometry.py': '89ced1a13a2af45d45519aa6f07211a5665fa34e2250bedb034534260a1e4084', 'src/biohub/public_postproc/graph_ops.py': '19605c9dffef7b5be1c89798ddc2d6954fd7c6a29f4d67f36aa3e0318fa59cdc', 'src/biohub/public_postproc/pipeline.py': '4321c573c80e58f87cc26baed38e80c693efedbc53db59ce3343bab36cf8c88f'}
import sys, hashlib, os
payloadroot = WORKING_DIR / 'e31_payload'
payloadroot.mkdir(exist_ok=False)
for path, content in E31_FILES.items():
    dest = payloadroot / path
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, 'x') as fh:
        fh.write(content)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'scripts'))
if any(name == 'biohub' or name.startswith('biohub.') for name in sys.modules):
    raise RuntimeError('biohub imported before hashed payload setup')
sys.path.insert(0, str(payloadroot / 'src'))


In [ ]:
import os
import torch
from pathlib import Path
from biohub.e31_shards import run_workers

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        "e31_dual_gpu_runtime requires >=2 visible CUDA devices; no CPU/single-GPU fallback is provided."
    )

raw = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
if raw:
    tokens = [t.strip() for t in raw.split(",")]
    if len(tokens) < 2:
        raise RuntimeError(f"CUDA_VISIBLE_DEVICES must contain at least 2 tokens, got {raw!r}")
    first2 = tokens[:2]
    if not first2[0] or not first2[1]:
        raise RuntimeError(f"First two CUDA_VISIBLE_DEVICES tokens must be non-empty, got {raw!r}")
    if first2[0] == "-1" or first2[1] == "-1":
        raise RuntimeError(f"First two CUDA_VISIBLE_DEVICES tokens must not be -1, got {raw!r}")
    if first2[0] == first2[1]:
        raise RuntimeError(f"First two CUDA_VISIBLE_DEVICES tokens must be distinct, got {raw!r}")
    cuda_tokens = first2
else:
    cuda_tokens = ["0", "1"]

job = {
    "WORKING_DIR": Path(WORKING_DIR),
    "REPO_DIR": Path(REPO_DIR),
    "TEST_DIR": Path(TEST_DIR),
    "_ps": Path(_ps),
    "_primary_materialized_path": Path(_primary_materialized_path),
    "_deepcenter_materialized_path": Path(_deepcenter_materialized_path),
    "SECONDARY_WEIGHTS_PATH": Path(SECONDARY_WEIGHTS_PATH),
    "E31_HASHES": E31_HASHES,
}

run_workers(job, payloadroot, cuda_tokens)
